## This notebook contains my personal in-depth study notes in Russian on the mathematics and theory behind the models used in this projec.

## Общий алгоритм выполнения задания с использованием моделей линейной регрессии и бустинга:   
### EDA

У нас есть 2 основных файла, с которыми мы будем активно работать: `train.csv` и `test.csv`. Первый - это те данные, на которых мы будем обучать модель. В тренировочном файле есть таблица с признаками и значениями, а также в этой таблице есть те данные, которые нужно научиться предугадывать модели. Второй файл - аналогичен первому, это такая же таблица с признаками и значениями, только без колонки с данными, значения которой нужно предугадать.    

В ML мы работаем, как правило, с табличными даннымы - csv. И библиотека `pandas` является фундаментом для работы с данными в ML. В pandas есть 2 основных типа объектов - `Series` и `DataFrame`. Series - это одномерный массив данных (как столбец в таблице или список Python). У него есть индекс (метки строк) и значения. В ML это обычно целевая переменная или один признак. DataFrame - это двумерная структура. Она состоит из множества объектов Series, объединенных общим индексом. В ML - это датасет, где строки это объекты, а столбцы это признаки. Эти 2 класса являются наследственными от класса `NDFrame`. Именно этот класс является родительским для `DataFrame` и `Series`. И все методы, которые определены в `NDFrame` - являются и методами `DataFrame` и `Series` одновременно.   

Сначала преобразовываем данные с помощью библиотеки `pandas`. Преобразовывать таблицные данные необходимо в структуру `DataFrame`. Делается это с помощью функции библиотеки `read_csv()`, которая принимает в качестве аргумента путь к csv-файлу:  
```
import pandas as pd
train = pd.read_csv(path)
```

Далее просматривается основная информация о файле с помощью `info()` - метода объектов `NDFrame`. Данный метод выводит техническую сводку о датасете или столбце. Он показывает сколько строк и столбцов в датасете, сколько в каждом столбце непустых (non-null) значений. Как правило, он примеряется только для объектов `DataFrame`, так как в контексте `Series` он малоинформативен:   
```
train.info()
```

Далее можно использовать метод `describe()` класса `NDFrame`. Он показывает статистику по значениям: count (количество непустых значений), mean (среднее арифметическое), std (стандартное отклонение), min и max (максимальное и минимальное значение) и квартили (25%, 50% , 75%). Данный метод применяется часто как для объектов `DataFrame`, так и для объектов `Series`. Но если его применить на необработанных данных, где строковые и числовые данные смешаны, то строковые признаки просто будут исключены из отчета. А работа будет вестись только с числовыми. Если же необходимо увидеть статистику и по строковым данным, то необходимо использовать параметр `include` со значением `str` или `string`:  
```
train.describe(include='str')
```
Статистика по строковым значениям будет включать в себя: count (количество непустых значений в каждом столбце), unique (количество уникальных значений), top (самое частое значение, мода), freq (сколько раз встречалось "топ" значение).   
Также можно использовать `include = 'all'`. В этом случае будут отображаться все колонки (и числовые и строковые), но для числовых будут столбцов будут пустыми параметры "unique/top", а для текстовых будут пустыми "mean/std".   
Чтобы посмотреть статистику по какой-то одной колонке, нужно применить `describe()` именно к одной колонке:  
```
train['SalePrice'].describe()
```
Таким образом мы сможем проанализировать наличие выбросов (разрыв между 75% и max).   

Теперь необходимо посчитать количество пропусков в абсолютном и процентном соотношении. Это делается с помощью вот такой конструкции:   
```
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_percent = train.isnull().sum() / len(train) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)
```  
В данном случае создается переменная, которая будет хранить в себе пропуски. Для этого используется метод `isnull()`. Это метод NDFrame, то есть доступен как для DataFrame, так и для Series. Мы его применяем к объекту DataFrame. `isnull()` возвращает объект того же типа, заполненный `True/False` (где True - там пропуск, где False - там нет пропуска). То есть мы на выходе получаем ту же самую таблицу, только заполненную не значениями, а True/False.   

Далее к этому DataFrame применяется метод `sum()`. Это метод класса NDFrame. Он доступен и для DataFrame и для Series, но поведение разное. Основная функция - суммирование. При применении к DataFrame возвращает объект Series с суммами по каждому столбцу (если применяется к Series, то возвращает scalar (просто одно число)). Это можно регулировать с помощью параметра `axis`, который принимает 2 значения: `0` - суммирование вниз, по строкам; `1` - суммирование вправо, по столбцам. В нашем случае мы получаем объект Series, который представляет из себя колонку с количеством пропусков, так как все пропуски (True) заменяются единицами и суммируются друг с другом по столбцам.   

В Python есть такая же функция `sum()`. Но Python `sum()` и Pandas `sum()` устроены по-разному. В синтаксисе Python - `sum(iterable)`, а в синтаксисе Pandas - `series.sum()` или `df.sum()`. Pandas `sum()` работает быстрее и умеет работать с пропусками, чего не умеет Python `sum()`.   

Далее применяется метод `sort_values()`. Этот метод отдельно определен в DataFrame и отдельно в Series, так как логика сортировки разная. Сам метод занимает сортировкой. Для Series часто применяется параметр `ascending`, который может иметь значения `True` (для сортировки по возрастанию) и `False` (для сортировки по убыванию). Если мы говорим о применении для DataFrame, то тут обязательным является параметр `by`, принимающий в качестве знаяения имя столбца или столбцов по значениям которых сортровать строки.   

Прежде, чем использовать сортировку, нужно применить фильтрацию к данным. В первой строке мы получили объект Series, в котором было собрано количество пропусков по всем колонкам. Но работать мы будем только с нулевыми, поэтому нужно отфильтровать колонки и оставить только те, где есть хоть один пропуск:   
```
missing = missing[missing > 0]
```
Разберемся с тем, как работает эта строка. Это называется **булевая индексация**. `missing` - это Series. Индексы = имена колонок, значения - кол-во пропусков. Вычисление идет изнутри наружу. Сначала вычисляется `missing > 0` (операции индексации вычисляются первыми) - возвращает тот же Series, но с булевыми значениями. То есть напротив наименования колонки у нас теперь не кол-во пропусков, а булева величина (True - если есть пропуски, то значит значение было больше нуля, соответственно True, если пропусков нет (то есть напртив наименования колонки 0), то получается False). Заметьте, что длина Series на этом этапе не меняется. Мы в скобках просто создали **булеву маску**.    
Далее идет вычисление `missing[...]`, мы применяем эту булеву маску к Series. Квадратные скобки с булевой серией внутри = фильтр. Остаются только те строки, где условие было True. То есть Series оставляет только те элементы, где маска равна True.   
После этого идет присваивание результата.   
Применение вот таких масок - это синтаксис Pandas. На чистом Python такое не получится сделать со списками. Но в чистом Python применяется другой подход - **генераторы списков** (с нимим мы познакомимся позже). 

В противовес фильтрации рассмотрим трансформацию, с котрой мы встретимся на этапе feature engineering. Возьмем вот такую строку для рассмотрения:  
```
train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)
```
Эта строка создает новый признак на основе имеющегося в датасете. Метод `astype()` рассматривать сейчас не будем, наша цель - разобрать то, как сравнение работает внутри квадратных скобок и снаружи. Как работает внтури мы разобрали, теперь посмотрим как работает снаружи.   
`train['Fireplaces']` - это объект Series (колонка), которая содержит в себе данные о количестве каминов в доме. И в коде выше мы проводим векторизированную операцию - поэлементно сравниваем значения с 0. И после сравнения мы получаем то же самый Series, то вместо числовых значений - True/False. А далее мы преобразовываем их в числа 0/1.   

Почему мы не можем применить булеву маску в этом случае? Например:   
```
train['HasFireplace'] = train[train['FirePlaces'] > 0]
```
Дело в том, что в этом случае мы бы собрали все колонки, где есть хотя бы один камин и присвоили бы это все новому признаку. Но получили бы ошибку, так как из Series с 1460 строками, мы бы получили Series с, например, 800 строками. И попыталсь бы присвоить итог Series с 1460 строками.   
Важно помнить, что при **фильтрации** мы отсеиваем ненужно, Series уменьшается в размерах, а при **трансформаци** мы приравниваем значения к True/False, но ничего не исключаем, Series имеет прежнюю длину.  

Также, дополнительно разберем метод `value_counts()`. Это один из самых полезных методов для работы с категориальными параметрами. Это метод объектов класса Series. Он работает следующим образом: подсчитывает количество уникальных значений в объекте Series и возвращает новую Series, где индекс - уникальные значения из исходной Series, а значения - количество раз, которое каждое значение встретилось. Есть много параметров, но наиболее полезным является `dropna`. При значении False, в Series попадает значение NaN, если же значение True, то NaN откидывается и мы получаем статистику только по реальным значениям.  

### Preprocessing

Теперь начнем заполнять пропуски. Пропуски заполняются с помощью метода `filna()`. Это метод объектов класса NDFrame. В метод передается значение, котором заполняется пропуск:   
```
train['PoolQC'] = train['PoolQC'].fillna('NoPool')
```
С числовыми признаками также, только вместо строки - число.    

Также нужно помнить о том, что есть признаки, в которых пропуски нужно заполнять медианой. Например, в House Prices есть признак "LotFrontage". Он означает ширину участка по фасаду, то есть длину той стороны земленного участка, которая граничит с дорогой или улицей. Имеет числовой тип.    
В учебном датасете имеет множество пропусков. И просто заполнить медианой их нельзя. Для грамотного препроцессинга нужно заполнить пропуски медианой по районам (Neighborhood). То есть для одного района высчитывается одно медианное значение LotFrontage, для другого района - другая медиана.   

В этом поможет метод `groupby()`. Это метод определен в NDFrame. Он разбивает данные на группы по ключу (значению столбца или индекса), выполняет функцию для каждой группы независимо и собирает результат обратно в объект DataFrame или Series. Важно понимать, что сам по себе `groupby()` ничего не вычисляет, он лишь создает объект - "инструкцию", который хранит информацию в группах. Вычисления начинаются только после вызова метода агрегации (.sum(), .mean(), .transform() и т.д.). Это называется **ленивым вычислением**.   

Рассмотрим принцип работы `groupby()` более подробно на примере с признаком LotFrontage:  
```
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
```
В примере выше мы разбиваем весь датасет на группы по значениям признака Neighborhood. У нас есть 25 уникальных значений в колонке Neighborhood. А значит весь датасет DataFrame разбивается на 25 частей и превращается в DataFrameGroupBy. Во вторых скобках (квадратных) мы указываем еще один признак, со значениями которого мы будем работать. И на этом этапе мы получаем уже объект SeriesGroupBy, который содержит 25 групп колонок со значениями LotFrontage. И далее мы применяем метод `transform()`. Это метод объектов класса `GroupBy`, который является родительским для классов `DataFrameGroupBy` и `SeriesGroupBy`.   
Так вот, `transform()` принимает какой-то блок кода, который потом выполняется для каждой из 25 групп. В данном случае мы передаем в `transform()` lambda-функцию, в которой `x` - это группа (каждая из 25). И мы к каждой группе применяем метод `fillna()`, который принимает значение медианы каждой группы. `median()` - это метод объектов NDFrame. Он вычисляет медиану для каждой их 25 групп.   

Таким образом, мы вычисляем медиану на каждой группе и заполняем пропуски в каждой группе этими медианами.   

Если же так получилось, что у нас есть целлый район, где нет вообще никаких данных признака LotFrontage, то мы должны заполнить их глобальной медианой:  
```
if train['LotFrontage'].isnull().any():
    global_median = train['LotFrontage'].median()
    train['LotFrontage'] = train['LotFrontage'].fillna(global_median)
```
Метод `any()` - это метод класса NDFrame. Данный метод проверяет, есть ли хоть один элемент, который оценикается как True.    
И в коде выше, мы преобразовываем все ячейки LotFrontage в True/False и применяем метод `any()`, так как True будет являться пропуском, и если есть хотя бы одно True - оно будет заполнено глобальной медианой.  

Далее удалим строки, которые не нужны. Смотрим на количество пропусков по столбцам, и если видим, что в каком-то столбце есть 1-2 пропуска, то можно просто удалить строку, которая образует этот пропуск. На общую картину это не повлияет.   
Строки удаляются с помощью метода `dropna()`. Это метод объектов класса NDFrame. Данный метод удаляет метки (строки или столбцы), содержащие пропущенные значение (NaN или None). По умолчанию `dropna()` не изменяет объект, а возвращает новый, поэтому необходимо использовать параметр `inplace=True` или присваивание ноаой переменной. Если применяем к DataFrame, то важно использовать аргумент `subset='...'`, в котором указывается название колонки. В этой колонки проверяются пропуски и удаляются те строки, в которых припущены значения в указанной колонке:   
```
train.dropna(subset='Electrical', inplace=True)
```

После того, как мы заполнили все пропуски, необходимо устанить выбросы. Выбросы устраняются путем логарифмирования целевой переменной. Если у нас есть в выборке несколько очень дорогих домов, то это значит, что нужно сжать перечень цен, иначе это будет сбивать модель с толку.   
Логарифмирование - это принцип, при котором вместо обучения модели на абсолютных значениях, мы обучаем на логарифмах значений. Это позволяет сжать большие значения и растянуть маленькие. Будем использовать натуральный логарифм для этих целей. Это делается с помощью функции из NumPy `numpy.log1p()`. Эта функция - натуральный логарифм (логарифм по основанию $e$ (~2.718)), к аргументу которого прибавляется 1. Зачем прибавлять к аргументу единиицу? Это защита от нулевых значений. Даже есть значение вдруг будет нулевым, то мы прибавим к нему единицу и значение станет положительным.   

Каким образом логарифмирование помогает? Вот пример в контексте данного соревнования:  

| Цена ($) | `ln(Цена)` | Прирост логарифма | Комментарий |
|:---------|:-----------|:------------------|:------------|
| 10 000   | ~9.21      | —                 | Базовый уровень |
| 100 000  | ~11.51     | +2.30             | ×10 к цене → +2.3 к логу |
| 500 000  | ~13.12     | +1.61             | ×5 к цене → уже меньший прирост |
| 700 000  | ~13.46     | +0.34             | ×1.4 к цене → совсем маленький прирост |   

Как видно из таблицы, в абсолютных числах разрыв между 10 000 и 700 000 просто огромный. Но если взять натуральный логарфм от этих чисел, то разрыв между ними получится минимальным, чуть больше 4.   

Дело в том, что логарифм измеряет не абсолютное изменение, а относительное. Вот взять, к примеру, логарифм по основанию 10. Этот логарифм отвечает на вопрос: "В какую степень возвести основание 10, чтобы получить это число". Именно поэтому мы получаем такие числа. И это помгает более качественно обучить модель. А потом эти логарифмы можно вернуть в абсолютные значения.   

Теперь то, как это реализуется на практике:  
```
train['SalePrice_log'] = np.log1p(train['SalePrice'])
```

### Feature Engineering  
Теперь переходим к следующему этапу - создание новых признаков. В контексте данного соревнования можно создать следующие признаки:   
- Общая жилая площадь. В этом признаке мы просто складываем площадь подвала и площади этажей дома. Сложение колонок - векторизированно. Все как в линейной алгебре, сложение поэлементно. Только нужно учитывать то, что это сложение идет не по порядковому номеру строки, а по индексам. И если в какой-то колонке отсутствует индекс, то это вызовет ошибку при сложении и мы получим NaN на конкретном индексе, а не сумму числа и нуля. Поэтому важно перед feature engineering заполнить все пропуски, иначе модель просто упадет при встрече с ними. Вот так выглядит это сложение: `train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']`.   
- Возраст дома. Для добавления данного признака мы вычтем из года продажи год постройки. Такая же векторная операция, как и предыдущая: `train['HouseAge'] = train['YrSold'] - train['YearBuilt']`.  
- Время с последнего ремонта. Вычитаем из года продажи год последнего ремонта: `train['RemodAge'] = train['YrSold'] - train['YearRemodAdd']`.   
- Взвешенное общее количество ванных комнат, приведенное к эквиваленту "полных ванн". В этом признаке мы складываем полноценную ванну над землей (ванна/душ + туалет + раковина), гостевой туалет над землей (только туалет + раковина), полноценная ванная в подвале и гостевой туалет в подвале. У полноценных ван коэффициент 1, а у неполноценных - 0.5. Эти коэффициенты - индустриальный стандарт в риелторской аналитике. В коде это выглядит вот так: `train['TotalBath'] = train['FullBath'] + 0.5*train['HalfBath'] + train['BsmtFullBath'] + 0.5*train['BsmtHalfBath']`.  
- Наличие гаража. Чтобы узнать это, мы используем площади гаража в домах и сравниваем их с нулем. Получаем True или False, а дальше переводим данные из булевой величины в 1/0. Это делается для того, чтобы отделить факт наличия от размера/количества. Это важно, так как дом с гаражом 15 м² и дом без гаража 0 м² сильно арзличаются по цене. Но для модели 0 и 15 лежат на одной числовой прямой. Модель может решить, что гараж 1 м² почти ничего не стоит, хотя сам факт наличия гаража повышает ликвидность. `train['HasGarage'] = (train['GarageArea'] > 0).astype(int)`.  
- Наличие подвала. То же самое, что и с гаражом.  
- Наличие камина. То же самое, что и с гаражом.   

Отдельно разберем метод `astype()`. Это метод объектов класса NDFrame. Возвращает объект того же типа, к которому он применялся. На вход принимает Python-тип (в данном случае int) и превращает данные из Series или DataFrame в этот тип.   

### Feature Transformation
Теперь перейдем к кодированию признаков. Поскольку модель не может работать с текстом, а только с числами. Нужно преобразовать все полученные данные в числа. Поэтому разделим все признаки на 3 типа: Numeric (числовые), Ordinal (порядковые) и Categorical (категориальные).   

**Numeric features** не нужно преобразовывать, так как они и так будут понятны модели. **Ordinal features** - это порядковые признаки. То есть если мы говорим о качестве камина, то он имеет несколько значений: Ex (Excellent, лучшее качество), Gd (Good, хорошоее качество), TA (Average, среднее качество) и так далее. Они должны быть расположены строго от худшего к лучшему. Это необходимо для правильного перевода в число. Все эти возможные значения будут представлены в массиве, и числовое значение будет присвоено в зависимости от индекса. То есть самое худшее будет иметь значение 0 (так как будет находиться на начальной позиции в массиве). **Categorical features** - это обычные строковые параметры. Например, параметр "фундамент" имеет возможные значения: "кирпич и черепица", "шлакоблок", "бетонный монолит" и другие. Тут не возможно сказать, что что-то лучше, а что-то хуже.     
Более подробно о том, как преобразуются в числа эти значения, будет дальше.   

Итак, для начала следует создать словарь для порядковых признаков. Ключами словаря будет наименование признака, а значениями будет массивы, где в правильной последовательности указаны все возможные значения этих признаков. Это необязательный пункт, но он нужен для удобства, так как из этого словаря мы будем формировать массивы для числовых параметров. И если потребуется что-то изменить, то проще будет внести изменение в словарь, из которого изменения перейдут в массивы, чем исправлять сначала один массив, а потом другой.    

In [ ]:
ordinal_mapping = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['NoBasement', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['NoGarage', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['NoPool', 'Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'PavedDrive': ['N', 'P', 'Y'],
    'LotShape': ['Reg', 'IR1', 'IR2', 'IR3'],
    'Fence': ['NoFence', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
}

После этого можно перейти к формированию массивов с различными группами признаков.    

Сначала сформируем массив числовых признаков. Это делается с помощью метода `select_dtypes()`. Это метод объектов класса NDFrame. Он возвращает объект того же типа, к которому применялся. Суть метода заключается в выборе подмножества колонок. Имеет параметр `include`, который должен быть приравнен к какому-то типу. Например, если мы хотим получить колонки только числового типа, то аргумент должен выглядеть так: `include=['int64', 'float64']`. Если хотим только строковые данные, то так: `include=['object', 'str']`. Причем `object` уже является устаревшим, и его скоро уберут (вместо него нужно использовать str/string). При применении к DataFrame возвращает также DataFrame, но только с колонками того типа, который указан в include. Также у данного метода есть нюанс: в pandas/numpy bool считается подтипом целых чисел, это нужно учитывать. Поэтому при передаче значения "number" нужно быть осторожнее, если не нужны булевы значения, то лучше писать конкретные типы, такие как int64 и float64. И немного поговорим про то, как этот метод работает в Series. Поскольку данный метод вычленяет колонки необходимого типа, а Series в pandas - это однородный контейнер и, соответственно, все элементы в Series имеют один dtype, то логика на Series вырождается в простую проверку. Если dtype в Series совпадает с include - метод возвращает ту же самую серию целиком, если не совпадает, то возвращает пустую серию.   

После `select_dtypes()` у нас на руках DataFrame, но только с теми колонками, которые нам подходят по типу. А нам нужно только наименования этих колонок. Чтобы получить наименования, используется `columns`. Это атрибут-свойство из класса DataFrame. Он возвращает объект `pandas.Index` - это специальная упорядоченная структура данных pandas. Она достаточно сильно отличается от списка в Python и даже от списоков ndarray из NumPy. `pandas.Index` неизменяем, имеет метаданные и является строго однородным.    
Из-за его особенностей, он привязан к pandas-контексту, и при передаче его sklearn, json или API это может вызвать FutureWarning или скрытые баги. Также возникают проблемы с строгой типизацией и сериализацией.   

Поэтому мы будем использовать метод `to_list()`. Это метод из pandas, но оригинальное название пришло из NumPy и исторически `tolist()` был методом NumPy объектов `ndarray`. Данный метод принадлежит исключительно одномерным структурам: `pandas.Index`, `pandad.Series` и `numpy.ndarray`. Данный метод возвращает обычный Python-список.   

После этого у нас появляется массив - `numeric_features`, где собраны все числовые признаки (наименования). Но есть нюанс. В оригинальном train у нас есть такие числовые колонки, как Id, SalePrice и SalePrice_log. Они нам тут не нужны, так как мы формируем массивы, которые в дальнейшем будут использоваться для обучения структруы, которая будет преобразовывать DataFrame в понятные для модели матрицы. И на этих матрицах модель будет обучаться. А для обучения модели нужно убрать лишние столбцы (так как Id не имеет информационной ценности, а цены должны быть скрыты от модели).    

Для этого применим **List Comprehension** с фильтрацией. Это идиоматический паттерн Python для создания нового списка на основе существующего с отбором элементов по условию. Имеет следующую структуру: `[элемент for элемент in исходный_список if условие]`. На реальном примере это выглядит вот так: `numeric_features = [col for col in numeric_features if col not in ['Id', 'SalePrice', 'SalePrice_log']]`.    

То есть в блоке `for col in numeric_features` мы перебираем исходный массив со всеми числовыми элементами. В блоке `if col not in ['Id', 'SalePrice', 'SalePrice_log']` ставим условие, что перебираемый элемент не должен быть в указанном массиве (в массиве указываем ненужные колонки). А `col`, который в самом начале - это то, что заносится в новый массив. Также не забываем, что весь этот алгоритм должен находиться в квадратных скобках.    

Теперь создадим массив из порядковых колонок. Для этого воспользуемся созданным ранее словарем и методом `keys()`. Это метод из встроенного класса `dict` в Python. То есть данный метод применяется к словрю для того, чтобы получить перечень ключей этого словаря. Метод возвращает объект типа `dict_keys`. Этот объект не является списком ключей, скорее "окном" в ключи словаря. Если добавить/удалить ключ из словаря, то объект dict_keys обновится мгновенно. Чтобы получить именно массив ключей, необходимо воспользоваться функцией `list()`.   

Функция `list()` определена в модуле `builtins`. Она принимает любой итерабельный объект (который можно перебрать в цикле) и создает новый изменяемый массив в памяти, последовательно извлекая элементы из итератора и записывая их в новый список. Возвращает объект типа `list`.    
Нельзя путать функцию `list()` с методом `to_list()`. Первое работает с любым итерируемым объектом и определено в Python, второе - является методом numpy/pandas и работает только на ndarray, Series и Index.   

В коде это выглядет так: `ordinal_features = list(ordinal_mapping.keys())`.   

Далее проделываем то же самое, но уже с категориальными признаками. В этом случае мы воспользуемся уже известными методами `select_dtypes()`, только отбирать нужно будет уже не числовые колонки, а строковые. В этом случае аргументу `include` нужно передавать значение `object`, но оно уже устарело, поэтому теперь нужно передать `string` ли `str`. Либо, чтобы наверняка, можно передать `['object', 'str']`.   

На основании получившегося массива создадим новый, так как строковые признаки могут быть как порядковыми, так и категориальными. А значит в текщем массиве у нас мешанина из строковых категориальных и порядковых признаков. Для решения пробемы воспользуемся **List Comprehension**. Условием будет следующим: `if col not in ordinal_features`. То есть в новый массив должны попасть все признаки, кроме тех, что есть уже в массиве ordinal_features. В коде это будет выглядеть так:  
```
categorical_features = train.select_dtypes(include=['str']).columns.to_list()
categorical_features = [col for col in categorical_features if col not in ordinal_features]
```

Также создадим еще один массив - `categories_list`. Это будет массив, в котором каждый элемент - это массив со значениями порядковых признаков, выстроенных в правильном порядке. Он потребуется нам позже. Создавать этот массив будет так же, как и предыдущие, с помощью **List Comprehension**: `categories_list = [ordinal_mapping[col] for col in ordinal_features]`.    

Теперь разберем причину, по которой нам нужен был словарь более подробно. Далее мы будем создавать энкодеры. Их будет всего 2 - энкодер для категориальных признаков и энкодер для порядковых признаков. 

Энкодер реализует замену строковых признаков на числа, чтобы математическая модель могла их "понять". Всего существует 2 вида энкодеров: OrdinalEncoder (порядковый) и OneHotEncoder (категориальный).  

**OrdinalEncoder** переводит порядковые признаки в числа. Ранее мы уже создавали словарь, в котором ключи = порядковые признаки, а значения = массив из возможных значений. Мы тогда расставляли их в порядке от худшего к лучшему. Это нужно, чтобы энкодер мог присвоить корректное число значению. Самый худший признак идет первым в массиве возможных значений. И поскольку он стоит первым, ему причисляется индекс 0. Следующее значение, которое чуть лучше, имеет уже индекс 1. И так далее. Таким образом, все возможные значения порядковых признаков переводятся в числа исходя из их позиции в массиве.   

**OneHotEncoder** переводит категориальные признаки в числа. Значения категориального признака не может быть хуже или лучше остальных. К примеру, если мы говорим о цвете забора (этого параметра нет в данных, просто пример), то он может иметь значение "желтый", "зеленый" или "красный". Мы не можем сказать, что какой-то цвет лучше, а какой-то хуже. OneHotEncoder кодирует такие признаки следующим образом: вместо одной колонки категориального признака формируется несколько колонок, количество которых зависит от количества возможных значений признака. Эти колонки бинарные и могут преобретать всего 2 значени - 0 или 1. Если забор имеет зеленый цвет, то в колонке зеленого цвета ставится 1, а остальные колонки, соответственно, преобретают значение 0. И так со всеми категориальными признаками.   

Очень важно не перепутать тип признака, иначе модель будет думать, что, например, зеленый цвет забора лучше красного. Или же, например, что лучшее качество камина ничем не отличается от среднего.    

Сами энкодеры ничего не преобразовывают. Это лишь "формы", которые содержат правила преобразования для каждой категории признаков. Приступим к реализации в коде. Для начала импортируем классы `OrdinalEncoder` и `OneHotEncoder` из пакета `sklearn`. Они находятся в подпакете/модуле `preprocessing`.    
А потом создадим 2 энкодера:       

```
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

ordinal_encoder = OrdinalEncoder(
    categories=categories_list,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

categorical_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)
```
В OrdinalEncoder мы передаем список из массивов, в которых все значения выстроены в правильном порядке (от худшего к лучшему). Также регулируем поведение энокодера в случае, если он на тестовых данных встретит незнакомое значение, которого не было в тренировочных данных: если встретит - пусть присовит этому значению -1.    
В OneHotEncoder также устанавливаем паттерн поведения при встрече с незнакомыми занчениями: энкодер просто будет игнорировать данное значение, и во всех бинарныз колонках признака проставит нули. А итоговой матрицой будет в формате numpy.ndarray.   

После реализации всех энкодеров нужно собрать их все вместе. Так как каждый энкодер отвечает за конкретные признаки, а реальный датасет состоит из множества разных признаков, нам потребуется инструмент, который сможет "связать" эти энкодеры вместе, чтобы их можно было применить к реальному датасету.   
Для этого необходимо создать объект `preprocessor` класса `ColumnTransformer`. Данный класс находится в модуле `sklearn.compose`. Импортируем его, а далее реализуем preprocessor:   
```
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers = [
        ('num', 'passthrough', numeric_features),
        ('ord', ordinal_encoder, ordinal_features),
        ('cat', categorical_encoder, categorical_features)
    ],
    remainder = 'drop',
    sparse_threshold = 0.3,
    verbose_feature_names_out = True
)
```
В данном случае были использованы все параметры, которые только есть у класса ColumnTransformer. Всего их 4. Первый параметр `transformers`. Это единственный обязательный параметр класса. Он всегда должен присутствовать в инициализации объекта. Принимает массив кортежей, каждый кортеж состоит из произвольного наименования трансформера, энкодера и списка признаков. В числовом кортеже вместо энкодера используется строка 'passthrough', это означает, что числовые колонки нужно передать в выходную матрицу как есть, без каких либо изменений.    
Далее идет параметр `remainder`. В данном случае он имеет значение 'drop', что означает "исключить все колонки из X_train (передаваемого датасета), которых нет в numeric_features + ordinal_features + categorical_features".   
Параметр `sparse_threshold` имеет базовое значение в 0.3, его можно не трогать, а оставить по умолчанию. Это значит, что если в выходной матрице будет ненулевых значений меньше 30% - выходная матрица будет иметь формат scipy.sparse (разряженная матрица).   
Параметр `verbose_feature_names_out` имеет значение True, что означает, что к наименованиям колонок будет приписано "name__", где "name__" - это произвольное название трансформера. То есть колонка, например, "FireplaceQu" будет иметь в выходной преобразованной матрице название "ord__FireplaceQu".   

Также необходимо понимать, что порядок колонок в выходной матрице строго соответствуют порядку кортежей в transformers, затем идут колонки из remainder (если remainder='passthrough', а не 'drop', как в нашем случае). То есть в нашей выходной матрице будут иди колонки в следующем порядке: числовые -> порядковые -> категориальные. В конце "хвост" (но не в нашем случае, так как remainder='drop').   

Теперь, когда мы закончили с проектированием preprocessor-а, нужно разделить наш исходный датасет. У preprocessor-а, как и у любого другого трансформера (ColumnTransformer просто один из множества трансформеров в sklearn) есть 3 ключевых метода: `fit()`, `transform()` и `fit_transform()`. И как мы знаем, preprocessor нужен для преобразования исходного датасета в числовую матрицу. Так вот этим занимается метод `transform()`, но до этого метода нужно вызвать метод `fit()`, так как сначала мы обучаем preprocessor, а только потом он преобразовывает датасет в матрицу. И в понимании разницы между `fit()` и `transform()` кроется ключ к пониманию того, почему обучать preprocessor нужно на части данных, а не на всех сразу.   

Сначала разберемся, что такое метод `fit()`. Когда мы передаем датасет в метод `fit()`, preprocessor сканирует датасет, находит в нем уникальные категории, сортирует их, считает частоты и самое главное - строит словарь-правило и запоминает его внутри себя. Казалось бы, зачем все это нужно, если мы и так самостоятельно отсортировали необходимые колонки на числовые, порядковые и категориальные признаки, передали всю необходимую информацию в энкодеры и упаковали это все в preprocessor. Но на самом деле, preprocessor самостоятельно ничего не учит. При вызове `fit()` у preprocessor-а, он берет наш датасет (X_train), находит в нем колонки из ordinal_features и categorical_features. Проверяет, что они существуют (если нет, то кидает ValueError). Далее он вызывает метод `fit()` у каждого энкодера по отдельности: `ordinal_encoder.fit(X_train[ordinal_features])` и `categorical_encoder.fit(X_train[cat_features])` (когда передаем в квадратные скобки в DataFrame список, pandas интерпретирует это как "верни новый DataFrame, содержащий только те колонки, которые есть в переданном массиве"). Каждый энкодер пробегает по переданным колонкам и создает в памяти быструю хеш-таблицу. Например, ordinal_encoder выглядит так после `fit()`:
```
self.mapping_ = {
    'ExterQual': {'Po':0, 'Fa':1, 'TA':2, 'Gd':3, 'Ex':4},
    'BsmtQual': {'NoBasement':0, 'Po':1, ...}
}
```
А categorical_encoder выглядит как упорядоченный список numpy.ndarray.   
`fit()` запоминает, какие категории фактически встретились. Если в categories_list было ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], а в данных только ['TA', 'Gd'], энкодер все равно создаст маппинг для всех 6 значений, но будет знать, что остальные пока не встречались.   

Метод `transform()` же просто применяет все эти правила и преобразовывает датасет. Берет готовый словарь-правило (созданные энкодерами хеш-таблицу) и просто заменяет текст на числа.    

Теперь перейдем к ответу на вопрос: "Зачем разбивать датасет на части для обучения preprocessor-а?". Обычно разбивают на 2 части с соотношением 80/20. 80% - это часть, на которой будем тренировать preprocessor, а 20% - это валидационная выборка. Это делает для того, чтобы не случалось Data Leakage (утечка данных). Дело в том, что во время вызова метода `fit()` preprocessor сканирует датасет, находит уникальные категории, сортирует их и строит словарь-правило. Мы более подробно разобрали выше, этим всем занимаются по сути энкодеры. И сложность процедуры заключается в том, что правила замены формируются из данных. И если мы дадим preprocessor-у "посмотреть" на 100% данных, то случиться эта самая утечка данных, ведь он будет выстраивать данные преобразования с учетом будущего. Рассмотрим пример для лучшего понимания для OneHotEncoder с categories = 'auto':   

Допустим, в train (80%) район Neighborhood имеет 20 уникальных значений. В val (20%) случайно попал 21-й район: 'NewDistrict'. Neighborhood - это категориальный признак. В нашем случае, у OneHotEncoder параметр handle_unknown равен 'ignore', соответстввенно, все новые значения, которые энкодер увидит в тестовом датасете - будут проигнорированы, а имеющиеся колонки получат по нулям. Когда мы передаем в preprocessor данные, мы передаем лишь энкодер и массив наименований категориальных колонок. То есть энкодер (категориальный) знает, какие конкретно колонки он должен трансформировать, но всех возможных значений он не знает. И во время обучения он строит ограниченое количество колонок, только под те значения, которые встречает. И если мы обучим его на 80%, то на валидационной выборке он встретит 21-ое значение, и будет вынужден проигнорировать его, а во все 20 колонок проставить нули. Если бы мы обучали энкодер на 100% данных, то энкодер заранее знал бы, сколько всего колонко нужно построить, и построил бы 21 колонку, в однй из которых стояла бы единица. Из-за вот таких различий модель может давать завышенный результат на валидационной выборке, если preprocessor обучался на 100% данных.    

Таким образом, нам необходимо тренировать preprocessor на 80% тренировочных данных, чтобы сразу видеть, какой результат способна выдавать модель. А не удивляться, почему на тренировочных данных хороший результат, а на тестовых - нет.   

Теперь, когда мы разобрались, зачем делить датасет перед обучением preprocessor-а, приступим к реализации:  

```
from sklearn.model_selection import train_test_split

X = train.drop(['Id', 'SalePrice', 'SalePrice_log'], axis=1)
Y = train['SalePrice_log']

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
```

Как видно из кода выше, разделение осуществляется с помощью функции `train_test_split()`, которая находится в подмодуле model_selection библиотеки sklearn. Импортируем эту функцию и начинаем формировать X и Y. X - это наш датасет, из которого мы удаляем ненужные колонки: 'Id', 'SalePrice', 'SalePrice_log'. Id удаляем, так как в этой колонке нет никакой ценности, она не влияет на цену дома и ее наличие может запутать модель. SalePrice и SalePrice_log - это целевые переменные, данные, которых не будет в тестовом датасете. Соответственно, нет смысла учить preprocessor на колонках, которых не будет.   

Удаление лишних колонок реализуется с помощью метода `drop()`. Это метод объектов класса NDFrame. Ранее мы встречали похожий метод - `dropna()`. Но `dropna()` удалял строки/колонки, где есть пропуски, а `drop()` удаляет конкретные строки или колонки по имени/индексу. Ключевыми аргументами являются `axis`, который принимает значение либо 0 (по умолчанию, удаляет строки по индексу), либо 1 (удаляет колонки по имени), а также нужно передать список удаляемых колонок.   

Далее мы создаем переменную Y, которая будет в себе содержать Series с логарифмированными ценами домов. Это те результаты, которые мы хотим получить от модели.   

Функция `train_test_split()` принимает на вход 4 аргумента: X и Y - датасет и целевую переменную; test_size - этот аргумент означает процент валидационной выборки, обычно имеет значение 0.2 (то есть 20% всего датасета станет валидационной, на остальных 80% будем учить preprocessor). Но test_size можно заменить на train_size - данный аргумент будет означать процент обучающей выборки, остальное пойдет в валидационную. Обычно имеет значение 0.8 (то есть 80% на обучение, оставшиеся 20% на вылидацию). И последний аргумент - `random_state`. Он отвечает за генерацию seed - зерна для генератора псевдослучайных чисел. В коде он равен 42. Если мы не будем менять значение этого параметра, то при каждом новом запуске мы будем получать один и тот же split.    
Возвращает функция 4 переменных: пару X - тренировочную и валидационную, и пару Y - так же тренировочную и валидационную. Все типы сохраняются, DataFrame остается DataFrame, Series остается Series. Позиции переменных важны, сначала располагается X_train и X_val, а потом такие же Y (названия произвольные, можно поменять на любые другие).   

Теперь, когда у нас есть тренировочная и тестовая выборка, мы можем перейти к обучению preprocessor-а. Обучение важно проводить исключительно на тренировочной (80%) выборке, о причинах говорилось ранее:  
```
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
```
После обучения, у нас есть preprocessor, который умеет преобразовывать датасет в числовую матрицу. И после этого преобразуем в числовую матрицу оба датасета: тренировочный и валидационный.   

### Modeling  
Modeling является заключительным этапом в построении модели. Именно на этом этапе происходит подбор гиперпараметров и проектирование самой модели. К этому моменту у нас должно быть 4 числовых матрицы: X_train_processed, X_val_processed, Y_train и Y_val.   

Итак, начнем реализацию самой базовой модели линейной регрессии. В классическом ML есть группа моделей - линейные модели. В эту группу входят модели линейной регрессии (Ridge, Lasso, ElasticNet), а также модель логистической регрессии. Все они используют **линейную комбинацию признаков** и могут быть описаны одной главной формулой: 
$$ \hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b $$
где:   
$\hat{y}$ - итоговое число, которое выдает линейная комбинация. В линейной регрессии это уже готовый ответ, в логистической это промежуточное значение, которое потом превратится в вероятность.   
$x_n$ - это признаки, которые мы переводили в числовой тип с помощью preprocessor-а.   
$w_n$ - это веса, которые умножаются на признаки, то есть степерь влияния каждого признака на результат. Они показывают то, насколько важен тот или иной параметр.   
$b$ - сдвиг, это свободный член или "базовове значение". Он позволяет модели быть гибкой и не проходить обязательно через нулевую точку на графике.   

Ранее было перечислены все модели, для которых эта формула является основой. Чем же тогда они различаются? Логистическую регрессию разбирать тут не будем, заострим внимание на 4 главные модели линейной регрессии.   
| Модель | Итоговая функция потерь | Особенность |
|--------|----------------|-------------|
| **Обычная линейная регрессия** | MSE (без регуляризации) | Простейшая модель, но подверженная переобучению при большом количестве признаков |
| **Ridge** | MSE + α·Σwᵢ² | L2-регуляризация: штраф за возведение весов в квадрат. Все веса уменьшаются, но не обнуляются |
| **Lasso** | MSE + α·Σ\|wᵢ\| | L1-регуляризация: штраф за модули веса. Она может сбрасывать неважные характеристики (автоматический выбор) |
| **ElasticNet** | MSE + α₁·Σ\|wᵢ\| + α₂·Σwᵢ² | Комбинация L1 и L2 регуляризаций |   

Как видно из таблицы, все модели имеют функцию потерь. Разберемся, что это такое.    
**Функция потерь (Loss Function)** - это математический способ определить, насколько сильно ошибается модель в предсказании. Как видно из таблицы, разница между моделями заключается в итоговой функции потерь.    **Итоговая функция потерь (Objective Function)**- это функция потерь (Loss Function) + регуляризация.   

Разберем разницу между Loss Function и Objective Function более подробно, а также добавим в терминологию Cost Function.   

**Loss Function** (Функция потерь) - это функция, измеряющая непоредственную ошибку модели на конкретном примере. Она фокусируется только на том, как сильно модель ошиблась на данных. Нужна для вычисления градиентов и обновления весов:
$$L_i = (y_i - \hat{y}_i)^2$$
где:   
$L_i$ - это функция потерь только на одной строке датасета    
$y_i$ - настоящее/ожидаемое значение    
$\hat{y}_i$ - предсказанное моделью значение   

Если рассматривать функцию потерь в контексте Kaggle соревнования House Prices, то у нас есть датасет, состоящий из 1460 домов, на которых нужно обучить модель. У нас есть параметры домов и целевая переменная (реальная стоимость домов из датасета). Реальная стоимость - это $y_i$, именно такое число мы ожидаем от модели. Предсказанная стоимость - это $\hat{y}_i$, то, что модель по итогу обучения предсказывает. Чтобы понимать, насколько ошибается модель, существует формула, по которой можно высчитать ошибку - это и есть Loss Function. Считается очень просто, как видно из формулы: ожидаемая цена - предсказанная цена. И разница возводится в квадрат. На каждой строке из 1460 строк высчитывается значение этой функции. Возведение в квадрат обусловлено тем, что без квадрата функция ошибки может быть отрицательной. Возведение в квадрат решает эту проблему. Также квадрат обеспечивает гладкую дифференцируемость (нет излома в 0, как у модуля) и математически соответствует предположению, что ошибки распределены нормально.   

**Cost Function** (Функция стоимости) - это среднее значение Loss по всему трировочному датасету. Это и есть то самое MSE из таблицы. MSE (Mean Squared Error) - среднеквадратичная ошибка:
$$\mathrm{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
где:    
$y_i$ - настоящее/ожидаемое значение      
$\hat{y}_i$ - предсказанное моделью значение     
${n}$ - число строк в датасете    

Это среднее арифметическое всех функций потерь. Мы складываем вместе функции потерь на каждом объекте и делим на количество объектов в датасете. Именно для этого этапа нам и нужно возводить в квадрат разницу ождиемого и предсказанного значения, чтобы функция потерь была только положительной. В противном случае, на каких-то объектах она может быть положительной, а на других - отрицательной. И эти объекты будут компенсировать друг друга. И в коненчном итоге, мы не получим объективной оценки.   

**Objective Function** (Целевая функция) - это глобальная функция, которую алгоритм стремиться оптимизировать на всем процессе обучения. Она может включать не только ошибку на данных, но и регуляризацию, ограничения, бизнес-метрики. Вот, к примеру, так выглядит целевая функция модели Ridge:
$$J(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2 + \alpha \sum_{j=1}^{m} w_j^2$$
где:    
$\frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$ - MSE    
$\alpha \sum_{j=1}^{m} w_j^2$ - L2-регуляризация    


Теперь, когда мы разобрались с тем, как выглядит формула линейных моделей, а также как оценивается качество модели, пришло время разобраться с тем, как модель линейной регрессии учится.    

Модели линейной регрессии (LinearRegression, Ridge, Lasso, ElasticNet) могут использовать разные методы для обучения. Рассмотрим каждый метод обучения для каждой изучаемой модели:   
- LinearRegression (базавая модель линейной регрессии без регуляризаций) - данная модель обучается только с помощью *аналитического решения* (scipy.linalg.lstsq через SVD/QR). Аналитическое решение используется так как MSE дифференцируема, есть закрытая формула. Быстрый и точный метод для средних данных.   
- Ridge (базовая модель линейной регрессии с L2 регуляризацией) - данная модель использует *аналитическое решение* для плотных данных или *SAGA* (стохастический GD) для больших данных.  
- Lasso (базовая модель линейной регрессии с L1 регуляризацией) - данная модель использует *Coordinate Descent*.  
- ElasticNet (базовая модель линейной регрессии с L1+L2 регуляризацией) - данная модель использует *Coordinate Descent*.  

#### Градиентный спуск (начало)
Но обо всем по порядку. Сначала разберем градиентный спуск. В данном случае у нас есть 3 важные формулы, на которых все завязано.   
**Первая формула** - это формула модели линейной регрессии:
$$ \hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n $$
Мы уже ее видели ранее, но в немного другой форме. В данной форме $b$ (сдвиг) просто заменили на $w_0$. Именно по этой формуле вычисляется предсказание на каком-то **одном** объекте/строчке в датасете. В случае с датасетом из соревнования House Prices на Kaggle: На первом объекте (доме) берутся все признаки (их 81) и перемножаются каждый на свой вес (весов, соответственно, тоже 81). Далее все произведения складываются и добавляется сдвиг. Сдвиг необходим, чтобы регрессия не проходила через 0 (начало координат), так как такое редко бывает. К примеру, в данном соревновании, даже если у дома нет ничего, просто пустой участок, то пустая земля тоже что-то стоит. Поэтому если все признаки равны нулю, объект из датасета все равно не может стоить 0. Именно для этого и нужен сдвиг.    

**Второй ключевой формулой** является формула целевой функции (Objective function):
$$J(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2 + \alpha \sum_{j=1}^{m} w_j^2$$
Она также была недавно разобрана. Именно по этой формуле определяется качество модели. Чем меньше значение целевой функции, тем более качественный результат она дает.    
Функция состоит из суммы MSE и L2-регуляризации (разбираем на примере Ridge).   
MSE (Mean Squared Error) - первое слагаемое; среднеквадратичная ошибка, она показывает, насколько далеко предсказания модели от реальности по всему датасету.   
L2-регуляризация - она нужна, чтобы предотвращать переобучение. Переобучение - это процесс, при котором модель начинает принимать шум за закономерность. Например, у нас есть дом, который стоит аномально дорого из-за того, что в нем жил известный человек. Но в датасете об этом информации нет. В таком случае модель без регуляризации может принять эту аномалию за правило и задрать вес для комбинации "Забор красного цвета + Район А + Крыша из черепицы + Постоен в 1994 году". Такая уникальная комбинация может встречаться только у того самого одного дома. И для этого дома модель подберет веса так, чтобы ошиба была 0. А для всех остальных домов в датасете эта комбинация не сработает (множитель $x$ будет равен 0) и на их цену эти гигантские веса не повлияют. И как итог, MSE на всем датасете станет меньше, потому что на одном доме ошибка исчезла, а на других не появилась. При этом модель не выучила закон рынка, она просто "зазубрила" аномалию.    

##### Переобучение (overfitting)
Но возникает вопрос, если модель подгоняет веса под комбинацию 4 признаков там, чтобы сумма была огромной, а ошибка на ообъекте была 0, то каким образом другие дома не пострадают от аномально больших весов на этих признаках? Ведь модель регулирует веса не отдельно для каждого дома, а только для каждого признака. То есть на весь датасет может быть только 81 уникальный вес. Модель подбирает вес $w_1$, и он умножается на каждое $x_1$ в датасете, коих в нашем случае 1460. И получается, что задаранные веса на этих 4-х признаках все равно подкинут проблема для остальных домов, в которых попадется комбинация 2 или 3 признаков из четырех.    

Объясняется это легко, секрет в бинарных признаках. Бинарные признаки принимают значения только 0 или 1. И если год 1994 является числовым и он, умножаясь на вес, даст огромную цифру, то условный забор зеленого цвета вообще никак не повредит другим домам. Ведь аномальный вес на красном заборе того самого дома будет умножаться на 1, а на других обычных домах, где забор другого цвета, аномального веса вообще не будет. Ведь после преобразования датасета с помощью препроцессора, признак "Цвет забора" расщепляется на несколько бинарных колонок. И аномальный вес будет причисляться только новой колонке "Красный забор". Для колонок "Зеленый забор", "Желтый забор", "Черный забор" модель будет подбирать совсем другие веса.   

Также, чтобы ошибка на соседе не взорвалась, модель ищет **еще более уникальный признак**, который есть **только** у аномального дома. Например, в данных может быть какой-то редкий технический параметр, который равен 1 только для этого объекта и 0 для всех остальных 1459 домов. В таком случае overfitting никак не повлияет на другие объекты, зато миллион "из воздуха" добавит очень дорогому дому. В этом случае ошибка на одном доме исчезла, а на остальных даже не шелохнулась, MSE упал, но модель ничего полезного не выучила, а просто нашла "костыль", чтобы объяснить одну точку. 

#### Градиентный спуск (продолжение)


Без регуляризации модель готова дать дать огромный вес какому-то признаку, который есть только у одного объекта, чтобы быстро подогнать ошибку поближе к нулю. Регуляризация L2 не дает этого сделать, так как даже если MSE упадет чуть-чуть, L2-штраф ($w^2$) станет таким огромным, что общая функция J "улетит в стратосферу". В итоге модель вынуждена искать общие признаки (которые есть у многих ддомов), потому что тогда один умеренный вес помогает уменьшить ошибку сразу у 500 домов. А это выгоднее с точки зрения математики. Рассмотрим более подробно формулу L2-регуляризации:
$$L2 = \alpha \sum_{j=1}^{m} w_j^2$$
Как видно из формулы, это сумма квадартов весов, умноженная на $\alpha$. Обратите внимание, что сумма весов идет с $j_1$, то есть в формуле регуляризации не участвует сдвиг ($w_0$). Если с весами все понятно, их всего 81 (в данном датасете) и они распространяются на все объекты, то что такое $\alpha$ мы еше не говорили.    
$\alpha$ - это обычный коэффициент, который регулирует штраф. Если он равен 0, то модель игнорирует регуляризацию (так как умножаем сумму квадартов на 0), а если, например, 1000, то модель будет готова пожертвовать точностью (MSE), лишь бы сделать веса крошечными.   

Теперь перейдем к последней основной формуле. **Третья формула** - формула градиентного спуска:
$$w_j^{\text{new}} = w_j^{\text{old}} - \eta \frac{\partial J}{\partial w_j}$$
Именно эта формула отвечает за обновления весов в модели. Новый вес равен разнице старого веса и производной $J$ по $w$, умноженной на $\eta$ (скорость обучения, насколько сильно меняем вес после вычисления производной).   

По такому же принципу обновляется и сдвиг:   
$$ b^{\text{new}} = b^{\text{old}} - \eta \frac{\partial J}{\partial b} $$


#### Алгоритм обучения Ridge модели  
Теперь, когда мы разобрали все 3 формулы, можно разобрать то, как обучается модель с использованием этих трех формул.   

У нас есть датасет, состоящий из 1460 объектов и 81 признака + сдвиг. Для каждого объекта вычисляется $\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$. Иксы инициализируются числами блогодаря энкодерам, а начальные веса инициализируются либо нулями, либо случайными малыми числами. Вычисялем ошибку на каждом объекте, а потом считаем среднюю ошибку. Получаем MSE. Добавляем к ней регуляризацию L2 и высчитываем первое значение $J(w)$. Это значение будет считаться baseline для модели, от этого значения буем отталкиваться.   

Запускаем оптимизацию $J(w)$ через градиентный спуск. Значения старых весов у нас уже есть, осталось посчитать $\eta \frac{\partial J}{\partial w_j}$ и $\eta \frac{\partial J}{\partial b}$.    
Чтобы найти производную $J(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2 + \alpha \sum_{j=1}^{m} w_j^2$ необходимо найти сначала производную первого слагаемого (MSE), а потом производную второго слагаемого (L2), так как **производная суммы равна сумме производных**.    

Производная $MSE = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$ вычисляет по правилу сложной функции: **"Чтобы найти производную сложной функции, нужно найти производную внешней функции, не меняя того, что внутри и умножить ее на производную внтренней функции"**.   
Поскольку мы дифференцируем MSE по $w$, нам необходимо разложить формулу, чтобы в ней появилось $w$. Иначе мы не сможем дифференцировать MSE, так как мы анализируем изменение MSE при изменении $w$. Представим $\hat{y}_i$ в виде $x_i w_j$. Да, $\hat{y}_i$ - это сумма произведений, а не произведение какого-то одного веса на признак. Но когда мы берем произведение по $w_j$, то мы смотрим, как меняется функция при изменении какого-то одного $w_j$. И если мы берем производную $J$ по $w_1$, то остальные $w$ не меняются. То есть мы меняем $w_1$, это приводит изменению $J$, а $w_2, w_3, w_4, \dots$ становятся константами и, соответственно, призводная константы равна нулю. Таким образом формулу MSE можно переписать так:
$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - w_j x_{ij})^2$$
Но вычислять мы будем производную только $(y_i - w_j x_{ij})^2$, так как мы используем свойство линейной производной $(f(x) + g(x))' = f'(x) + g'(x)$ (мы уже выше писали это правило, только словами). Поскольку нам нужно продифференцировать сумму одинаоквых функци, мы дифференцируем общий вид слогаемого, а затем суммируем результаты обратно. $\frac{1}{n}$ - это константа, которая по правилам дифференцирования просто выносится за знак производной. То есть в MSE мы не "опускаем" сумму навсегда. Мы берем производную одного слогаемого и возвращаем сумму, потому что $w_j$ влияет на ошибку в каждой строке датасета.    

Теперь мы получили сложную функцию, производную которой можно вычислить. Внешняя функция - это квадрат. Производная в этом случае будет вычисляться по формуле $y = u^2 => y' = 2 * u$. Соответственно внешняя функция будет выглядеть так: $2(y_i - w_j x_{ij})$. Внутренняя функция - вычисляется по правилу $(kx)' = k$, а $y_i$ является константой, так как это ожидаемое значение и оно не меняется (равно 0). Соответственно внутренняя функция будет выглядеть так: $-x_{ij}$. Итоговый вид производной:
$$((y_i - w_j x_{ij})^2)' = 2(y_i - w_j x_{ij})(-x_{ij})$$
Вынесем минус к двойке:
$$-2 (y_i - w_j x_{ij}) x_{ij}$$
Итоговая производная MSE будет выглядеть так:
$$MSE' = \frac{-2}{n}\sum_{i=1}^{n}(y_i - w_j x_{ij}) x_{ij}$$
Тут мы просто перенесли -2 вперед, умножив на степень.   

Теперь разберем, как найти производную L2-регуляризации. Итак, нужно дифференцировать $L2 = \alpha \sum_{j=1}^{m} w_j^2$. Тут, в отличии от MSE, сумму мы не будем "опускать". Почему? Потому что в MSE мы считали производную $J$ по $w_i$, и этих $w_i$ было 1460, и они все влияли на значение MSE. Тут же совсем другой подход. L2 мы можем представить вот в таком виде:
$$L2 = \alpha (w_1^2 + w_2^2 + w_3^2 + \dots + w_j^2)$$
Как видно из этого представления, в дифференцировании L2 будет только одно $w_j$, ведь все остальные члены будут константами и превратятся в нули. И таким образом, нам нужно вычисляить производную вот этого выражения:
$$L2 = \alpha w_j^2$$
В этом случае нужно применить правило $(u)^2 = 2*u$. Получаем $2 w_j$. Далее умножаем это на $\alpha$. Может возникнуть вопрос, почему тут не применяется правило $(kx)' = k$? Дело в том, что $\alpha$ - это константа, но не как "одинокое число", как например в $(w^2 + b)$. Тут $b$ действительно превращалось бы в ноль. $\alpha$ является константой, как "коэффициент". В $(w^2 * b)$ - $b$ множитель. По правилам дифференцирования мы выносим его за знак производной. Поэтому полный вид производной L2 юудет иметь вид:
$$L2' = 2 \alpha w_j$$

Что касается производной $\frac{\partial J}{\partial b}$, то тут принцип аналогичен нахождению производной по $w_j$.  

Теперь переходим к вычислению суммы производных:
$$\frac{\partial J}{\partial w_j} = \frac{\partial MSE}{\partial w_j} + \frac{\partial L2}{\partial w_j}$$
А полная формула градиентного спуска будет выглядеть так:
$$w_j^{\text{new}} = w_j^{\text{old}} - \eta (\frac{\partial MSE}{\partial w_j} + \frac{\partial L2}{\partial w_j})$$
Или же:
$$w_j^{\text{new}} = w_j^{\text{old}} - \eta (\frac{-2}{n}\sum_{i=1}^{n}(y_i - w_j x_{ij}) x_{ij} + 2 \alpha w_j^{\text{old}})$$
Но более корректно было бы заменить $(y_i - w_j x_{ij})$ на $(y_i - \hat{y_i})$:
$$w_j^{\text{new}} = w_j^{\text{old}} - \eta (\frac{-2}{n}\sum_{i=1}^{n}(y_i - \hat{y_i}) x_{ij} + 2 \alpha w_j^{\text{old}})$$

Таким образом, мы вычисляем новый вес для каждого из 81 $w$, так как $j$ принимает значение от 1 до 81 включительно. Также мы вычисляем новый сдвиг по такой же формуле, только считаем производную $J$ по $b$. Выичление всех новых весов происходит одновременно. После этого высчитывается новое значение целевой функции $J(w)$. Этот процесс называется **эпохой**. Но сколько таких эпох должно быть, чтобы модель стала достаточно обученной? Это происходит всегда по разному, чтобы сказать, что модель обучена, $J()$ должна оказаться на самом дне параболы, где касательная становится горизонтальной, что означает, что производная равна нулю. А если производная равна нулю, то шаг $\eta * 0$ тоже становится равен нулю. Но это в теории, а если на практике, то мы останавливаемся, когда ошибка $J(w)$ перестает заметно уменьшаться (например, разница между шагами стала меньше 0.00001). Это назвается **сходимостью**.   

В предыдущем абзаце упоминалось, что для того, чтобы модель обучилась, нужно, чтобы $J(w)$ оказалась на дне параболы. При чем тут парабола? Дело в том, что график $J(w)$ - это график параболы. Такая форма является следствием нашей формулы. MSE - это квадрат разности $(y_i - \hat{y_i})^2$. В математике любая функция, где главная переменная стоит в квадрате ($w^2$) - это парабола. Квадрат делает функцию "выпуклой". И у выпуклой функции всегда есть только одно дно (глобальный минимум).   

Начертим (или представим) график параболы. Ось Y - $J(w)$. Ось X - $w$. Поскольку первые веса инициализируются случайными малыми числами, то начальный вес может оказаться где угодно на этом графике параболы. Ращделим график пополам, на 2 зеркальные части. В левой части функция убывает, соответственно, на этом участке производная < 0 (отрицательная). А на правой части функция растет, соответсвенно, на этом участке производная > 0 (положительная). И вот мы обновляем какой-то вес с помощью градиентного спука, пускай будет $w_1$. И, предположим, начальное значение этого веса где-нибудь на левой части графика. В этом случае, чтобы найти новый вес, мы из начального вычтем  $\eta \frac{\partial J}{\partial w_j}$. Но поскольку в левой части производная отрицательная, мы умножаем ее на $-\eta$, что дает $+$. И новый вес становится больше, предыдущего. Соответственно, мы продвигаемся "вправо" по оси $x$, ближе ко дну. Если следующий вес вдруг оказался слишком большим и "улетел" в правую часть графика, то на этой части функция растет, производная положительная. Значит умножаем ее на $-\eta$ и вычитаем из предыдущего веса. Новый вес становится меньше предыдущего, а значит мы смещаемся "влево" по оси $x$, снова ближе ко "дну". И так до тех пор, пока мы не дойдем до этого самого дна, означающего минимум функции $J(w)$.   

Вот таким образом работает градиентный спуск. Модель сама не думает, куда ей идти, все подчинается исключительно законам математики. 

#### Стохастический градиентный спуск (SGD - Stochastic Gradient Descent)
Помимо обычного градиентного спуска (Batch GD), сущестует еще стохастический градиентный спуск (SGD). Оба метода используют для минимизации функции потерь, но объём данных, используемых для вычисления градиента на каждом шаге, отличается (градиентом называют список/массив из всех $\frac{\partial J}{\partial w_j}$). Разберем плюсы и минусы SGD, а также то, чем конкретно он отличается от обычного GD.    

Используя GD, мы вычисляем градиент на всем датасете. Плюсом такого подхода является то, что градиент вычисляется точно, так как происходит усреднение по всем данным, а направление спуска стабильное. К минусам относят скорость всего процесса, а также требовательность к количеству памяти.   

В случае же с SGD, на каждой итерации градиент вычисляется по одному случайному объекту (после перемешивания датасета) и обновляет веса. На следующей итерации будет взят другой случайный объект, будет вычислен градиент и также обновлены веса. Так будет до тех пор, пока не пройдем по всему датасету. Всего за одну эпоху будет 1460 обновлений весов. Перемешивание датасета происходит в начале каждой эпохи. Плюсами такого подхода является скорость, SGD, как правило, требуется меньшее количество эпоъ, чем Batch GD для сходимости. Такой метод очень быстро проводит обучение модели на больших данных. Также такой метод не требует загрузки всех данных в память одновременно, можно обучать модель на потоке данных. К минусам относят "шумность" градиента, веса могут скакать, а не плавно сходиться. Также нет точного критерия остановки (ошибка может колебаться).    

Формула целевой функции при SGD:
$$J(\mathbf{w}) = \left(y_i - \hat{y}_i\right)^2 + \alpha \sum_{j=1}^{m} w_j^2$$
Как видно из формулы, мы просто убрали среднее арифметическое из MSE, тем самым превратив его в $L_i$. И поскольку у нас изменилась формула целевой функции (на самом деле она все еще использует MSE, просто в момент каждого конкретного шага SGD "притворяется", что датасет сузился до одного объекта), соответственно, мы несколько иначе вычисляем веса в градиентном спуске, ведь нам не нужно брать производную от суммы $MSE + L2$. Мы теперь берем производную от суммы $L_i + L2$:
$$w_j^{\text{new}} = w_j^{\text{old}} - \eta (-2 (y_i - \hat{y_i}) x_{ij} + 2 \alpha w_j^{\text{old}})$$

Теперь разберемся, как это все работает на практике. У нас есть датасет, целевая функция, которую нужно минимизировать изменилась, формула спуска соответственно тоже. Процесс обновления весов аналогичен Batch GD, мы также обновляем все 82 веса (81 вес + b) одновременно. Разница заключается лишь в том, что мы не считаем градиенты исходя средней арифметической ошибки по всем 1460 объектам. Мы перемешиваем датасет (на каждой эпохе) и начинаем вычилять градиенты для каждого объекта. Начинаем, например, с 42-го дома (но первого по перемешенному списку). Считаем градиенты (все 82) только на основании одного этого дома. И после этого обновляем все веса. Далее берем следующий дом, например, 812-й (но второй по порядку в перемешенном датасете), и считаем градиенты на всех весах и снова обновляем их. И таким образом модель проходится по всем 1460 домам (по всем объектам, что есть в датасете). И это и есть **одна эпоха** в SGD. Значение $J(w)$ обычно считается после одной эпохи, то есть после того, как модель пройдется по всем 1460 домам. Так как вычисление $J(w)$ после каждого обновления весов значительно понижает производительность. В процессе обучения SGD модель "видит" только локальную ошибку $L_i$ и не знает, какая ошибка на всем датасете, но ей это и не нужно для того, чтобы сделать шаг. Именно поэтому SGD быстрее Batch GD. В Batch GD модель смотрит на все дома, обновляет веса, считает $J(w)$ и это является одной эпохой. То есть 1 шаг за одну эпоху. При SGD модель делает 1460 шагов за одну эпоху, так как проходится по всем домам, но при этом градиент вычисляется быстрее из-за того, что модель не обращает в этот момент внимания на остальные 1459 домов.  

#### Улучшенный стохастический усредненный градиент (SAGA - Stochastic Average Gradient Augmented)
Данный метод обучения является улучшенной версией SGD. Главная проблема SGD заключается в том, что когда мы приблежаеся ко "дну" параболы, шаги одного случайного дома могут увести нас в сторону. SAGA решает это через память.   

Перед началом обучения SAGA выделяет место для хранения градиентов по каждому объекту. Градиент - это вектор, состоящий из $\frac{\partial J}{\partial w_j}$. В нашем случае - это список из 81 производной по весу и еще одной производной по сдвигу: $[\frac{\partial J}{\partial w_1}, \frac{\partial J}{\partial w_2}, \frac{\partial J}{\partial w_3}, \dots, \frac{\partial J}{\partial w_{81}}, \frac{\partial J}{\partial b}]$. И когда мы говорим про рассчет градиента для n-ого дома, это означает найти этот вектор.   

В Batch GD нам не нужна была такая таблица, так как мы считали средний вектор градиентов, обновляли вес и забывали про этот вектор. В SAGA нам потребуется матрица размером 1460 x 82, где для каждого $i$-го дома будет лежать вектор его последнего вычисленного градиента $g_i$. Изначально эта таблица заполняется нулями или градиентами первого прохода. Также мы храним **средний градиент** по всем объектам:   
$$AvgGrad = \frac{1}{n}\sum_{i=1}^{n}g_i$$
где $g_i$ - это градиент $i$-го объекта.    

Представим таблицу размером 1460 x 82. По оси Y располагаются объекты (1460), по оси X располагаются веса (82). Каждая ячейка заполнена частной производной. Таким образом, каждая строка является градиентом, коих у нас будет суммарно 1460. Средний градиент не является частью таблицы, это отдельный один-единственный вектор из 82 частных производных. В этом векторе располагаются усредненные частные производные после того, как мы суммировали 1460 градиентов и разделили на количество элементов. Вот это и есть средний градиент (AvgGrad).   

В SAGA принято делать один "холостой" проход (первую эпоху) по датасету. Мы берем наши текущие веса (случайные малые числа) и проходи по всем 1460 домам. Считаем градиенты и записываем их в таблицу, ничего не обновляя. В конце этого прохода считаем первый AvgGrad.   
После этого начинается вторая эпоха, где мы можем увидеть алгоритм SAGA полностью.   

Со второй эпохи перемешиваем дома (как и в SGD, перемешиваем на каждой эпохе), берем первый дом (i=1) и считаем новый градиент $J_i'$ (вектор из 82 чисел). Тут, как и в SGD, $J(w) = L_i + L2$. Далее достаем из таблицы старый градиент $g_i$ этого дома (i=1) и тут же обновляем веса по формуле:
$$w_j^{\text{new}} = w_j^{\text{old}} - \eta (J_{ij}' - g_{ij} + AvgGrad_j)$$
Или же:
$$w_j^{\text{new}} = w_j^{\text{old}} - \eta (\frac{\partial L_i}{\partial w_j} + \frac{\partial L2}{\partial w_j} - g_{ij} + AvgGrad_j)$$

И в этот же момент мы обновляем AvgGrad и заменяем $g_i$ в таблице на свежий $J_i'$. То есть мы вычисялем полный градиент (все 82 веса, они вычисляются одновременно), а потом в этот же момент меняем в таблице градиентов старый градиент ($g$) объекта $i$ на новый ($J$), только что вычисленный градиент на этом же объекте $i$. AvgGrad тоже меняется в этот же момент.   

AvgGrad обновляется по формуле:
$$AvgGrad_{new} = AvgGrad_{old} + \frac{1}{n}(J_i' - g_i)$$
Обратите внимание, что ($J_i' - g_i$) - это разница между новым и старым градиентом какого-то объекта. А $n$ - это количество объектов в датасете (в нашем случае 1460).   

Теперь еще раз. Когда мы начинаме учить модель через SAGA, мы берем объект, вычисляем все новые веса одновременно (а чтобы вычислить все веса, у каждого веса нужно вычислить частную производную $\frac{\partial L_i}{\partial w_j} + \frac{\partial L2}{\partial w_j}$, что и является элементом градиента). В этот же момент мы меняем имеющийся в таблице градиент этого объекта на новый, только что вычисленный градиент. И в этот же момент мы вычисляем новый средний градиент путем сложения старого вектора с произведением $1/n$ и разницы между новым градиентом этого объекта и старым градиентом этого же объекта.    

Далее переходим ко второму дому и проделываем все то же самое. Когда мы пройдемся по всем 1460 объектам - это будет одна **эпоха**, и как и в SGD, в SAGA за одну эпоху мы делаем 1460 шагов. Таким образом, к началу каждой новой эпохи у нас будет абсолютно новая таблица градиентов, которая будет полностью отличаться от той, что была в начале предыдущей эпохи. И проделываем весь этот процесс до состояния сходимости.

##### Еще информация про графики и про то, почему SAGA работает
Ранее мы говорили о том, почему работает обычный GD. Если коротко, то целевая функция $J(w) = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2 + \alpha \sum_{j=1}^{m} w_j^2$ является квадратичной, так как имеет вид $y = x^2 + b$. Квадратичные функции имеют график параболы. Наша функция $J(w)$ соответственно тоже имеет график параболы. По оси Y - значение функции $J(w)$, по оси X - значение $w$. И, соответсвенно, парабола имеет "дно", к которому нам нужно прийти. И если наш вес оказывается на левой половине этой параболы, то частная производная < 0 (так как функция уменьшается), а значит по формуле градиентного спуска мы умножаем отрицательную частную производную на $-\eta$ и прибавляем к старому весу какое-то число, получая новый вес, который больше старого. Соответственно, наша точка сдвигается вправо по оси X, и мы приближаемся ко дну параболы. Если же изначальная точка оказывается на правой части параболы, то происходи все то же самое. Единственное отличие в том, что на правой части функция растет, соответственно, производная > 0, а занчит мы умножаем положительную частную производную на $-\eta$, и отнимаем это от старого веса. А значит новый вес меньше старого, соответственно, движемся влево по оси X.   

Это была упрощенная схема того, почему работает градиентный спуск. На самом же деле все несколько сложнее. Ведь чтобы такой график полностью отражал смысл GD, все веса должны быть одинаковыми, либо линейная модель должна состоять только из одного $x$, и соответственно, одного $w$. На деле же признаков, как и весов, много, и они все разные. Как тогда представить график с одной осью Y и 82-я осями X? Можно представить трехмерную чашу, где по центру проходит ось Y, а перпендикулярно ей 82 оси X. Но тогда возникает вопрос, если у всех признаков разные веса, где же на таком графике тот самый минимум, к которому мы должны прийти? Представим обычную комнату (3D-пространство). У любой точки в этой комнате есть 3 координаты: длина (X), ширина (Y) и высота (Z). Если мы положим мяч в угол такой комнаты, то его координаты могут быть, например, X=0, Y=0, Z=0, а если положим его в центр комнаты, координаты будут, к примеру, X=3, Y=4, Z=0. И тут можно заметить, что в обеих случаях, точка дна была одна и та же, разнились только координаты длины и ширины комнаты.   

В нашем случае у 82-мерной чаше "дно" - это тоже всего одна точка дна. Но чтобы описать положение этой точки, нам нужно 82 координаты. Для $w_1$ координата может быть 0.8, для $w_2$ координата может быть 0.2, для $w_3$ может быть -0.5. Все эти числа разные, потому что каждый признак влияет на цену по-своему. Но только при таком конкретном наборе этих разных числе общая ошибка J (ось Y) станет минимальной. Все эти 82 оси **перпендикулярны** друг другу, в аналитической геометрии перпендикулярность означает **независимость**. Каждый вес - это отдельное измерение, отдельное направление движения, а дно - это место, где пересекаются "идеальные" значения каждого из этих направлений. И представить это на графике бессмысленно, как только мы переходим за пределы 3D, мы перестаем "видеть" и начинаем вычислять. Поэтому проще представить не график, а к примеру, эквалайзер. Это будет проще для понимания. 

Также есть еще один важный момент, который нужно осветить. Это касается того, как обновляются графики параболы при каждом способе обучения. За то, как выглядит парабола и как она располагается в пространстве отвечают $y_i$ и $X_i$. Если мы возьмем Batch GD, то во время всего процесса обучения расположение параболы не меняется, так как мы берем усредненную ошибку и, соответственно, $y_i$ не меняется, и $X_i$ используется полностью. И эта одна парабола никак не меняет свое расположение относительно осей X и Y.    
Если же мы говорим про SGD, то тут ситуация уже другая, ведь мы не используем усредненную ошибку. Вместо этого мы на каждом шаге используем новый объект, соответственно, $y_i$ и $X_i$ на каждом шаге новые. А значит и параболу мы чертим на каждом шаге новую. И каждая новая парабола будет менять свое расположение в пространстве, "прыгая" по осям X и Y. По оси X парабола будет "прыгать" из-за того, что у каждого объекта есть свой идеальный вес по каждому признаку. Например, у нас есть $x_1$ - цена за квадратный метр. У особняка $w_1$ будет идеальным будет 0.9, а у сарая на окраине города 0.2. Модель ищет оптимальный вариант, который устроил бы все объеты. Скачки по оси Y обусловен разной разностью квадартов у каждого объекта. У одного дома может быть ошибка 10000 и дно параболы будет прижато к оси X, а у другого ошибка будет 1000000 и дно будет далеко от оси X.   
SAGA является гибридом. Мы так же как и в SGD берем новый дом на каждом шаге, но при этом вычитам старый градиент и прибавляем средний градиент по всему датасету. И получается, что парабола пытается прыгнуть, так как $y_i$ и $X_i$ изменился, но SAGA "привязывает" параболу к усредненной параболе. Сдвиги по осям происходят, но очень короткие и направленны гораздно точнее в сторону общего среднего дна. По оси X, благодаря таблице градиентов, дно меняется гораздо меньше, чем в SGD, а по оси Y прыжки остаются, так как ошибка конкретного дома никуда не девается. Но поскольку шаг становится точнее, мы не успеваем прыгнуть высоко вверх по ошибке из-за случайного шага.   

Теперь поговорим о том, почему же SAGA работает. SAGA работает на принципе **сокращения дисперсии**. Если мы посмотрим на формулу шага, то увидим, что AvgGrad - это "якорь", который удерживает параболу, чтобы она не "убежала" за одним странным домом:
$$w^{\text{new}} = w^{\text{old}} - \eta (J_i' - g_i + AvgGrad)$$
А $(J_i' - g_i)$ - это "дифференцированная поправка". Если мы обновили веса и увидели, что для объекта №10 градиент изменился, мы добавим только это изменение к обзему среднему.   
То есть в SAGA мы строим параболу, которая имеет такую же кртизну, как и текущий дом ($J_i'$), но ее дно постоянно корректируется вектором AvgGrad.    

Для того, чтобы было проще понять принцип работы SAGA - визуализируем его на параболе. Предположим, что у для некого признака $w_1$ идеальным дном для всего датасета (1460 домов) будет значение 0.5. А сейчас мы стоим на точке $w_1 = 0.55$. То есть мы почти у цели, модели осталось сделать небольшой шаг влево. И тут при обучении модель натыкается на дом-лачугу, для которой дном на $w_1$ будет являться значение 0.1. Для этой лачуги мы стоим слишком далеко справа, ее производная ($g_{ij}^{\text{new}}$) - это огромный положительный забор. Обычный SGD в этом случае бы умножил производную на $-\eta$ и катапультировал бы вес из точки 0.55 куда-нибудь в 0.2. Мы бы перелетели общее дно. В случае с SAGA все иначе, так как работает память (таблица градиентов). Мы смотрим в эту таблицу и видим старую производную этого же дома ($g_{ij}^{\text{old}}$), которая равна, к примеру, 0.8 и тот положительный забор производной был еще выше.   
Мы берем берем текущую производную и вычитаем из нее производную прошлой эпохи. Получаем отрицательную производную (0.55 - 0.8). Таким образом мы получаем **разностную поправку**. Если текущий объект "тянет" вес в одну сторону, мы вычитаем его предыдущее значение из памяти. И в данном случае эта отрицательная разница выступает как торомоз, не давая весу совершать инерционный прыжок в сторону дна параболы этого конкретного объекта. К полученной разности прибавляется усредненный градиент. И поскольку AvgGrad отражает "компромиссное" направление всей выборки, он доминирует над локальным шумом.   

#### Аналитическое решение (Normal Equation)
Итак, поскольку Ridge использует **аналитическое решение** для *плотных* данных и **SAGA** (стохастический GD) для *больших* данных, нужно разобраться, чем эти данные отличаются.   
**Плотные данные (Dense Data)** - это табличные данные, где почти в каждой ячейке стоит какое-то число, отличное от нуля. В противовес плотным данным существуют разряженные данные.  
**Разряженные данные (Sparse Data)** - это тбличные даннеы, где почти все ячейки заняты нулями.   

Если данных много (Big Data), то используется SAGA, вне зависимости от того, какие это данные, плотные или разряженные. Аналитический метод просто не сможет "переварить" слишком большой объекм данных. Аналитическое решение требует операции **инверсии матрицы** (возведение в степень -1). В математике это "тяжелая операция". Поэтому используется SAGA/SGD, так как этим методам безразницы сколько данных в таблице, для работы они берут по одному объекту за шаг.    
А вот если данные мало и они плотные, то себя хорошо проявит аналитическое решение, ведь если данные плотные, компьютер быстро и эффективно перемножит жти числа блоками.   
Если же данных мало и они разряженные, то в этом случае выбор чаще всего падает на итеративные методы, такие как SAGA или другие градиентные спуски. Разряженные матрицы хранятся в памяти особым образом. Итеративные алгоритмы умеют работать с такими форматами напряму, не "раздувая" матрицу до плотной. Также на разряженных данных градиент вычисляется очень быстро, поэтому SAGA долетит до дна параболы почти мгновенно.    

Но в любом случае, в библиотеке sklearn у Ridge есть параметр `solver='auto'`, так что он сам выберет лучший метод для обучения. Но все же нужно знать, чем отличаются матрицы и почему для одних данных используется спуск, а для других аналитическое решение.     

Суть аналитического метода заключается в том, что нам больше не нужно "спускаться" по склону параболы. Вместо итераций используется алгебра, чтобы сразу найти точку, где производная функции потерь равна нулю. Если в градиентном спуске мы ищем минимум эксперементально, то здесь мы решаем уравнение.    

**Формула аналитического решения** для Ridge-регрессии (с учетом L2-регуляризации):
$$w = (X^{\text{T}} X + \alpha I)^{-1} X^{\text{T}} y$$
Для обычной же модели (без регуляризации) используется такая формула:
$$w = (X^{\text{T}} X)^{-1} X^{\text{T}} y$$

В данной формуле:   
$w$ - это вектор, столбец из 82 весов и одного сдвига. В аналитическом методе нет шагов, как в GD/SGD/SAGA, мы находим все 83 веса одновременно, одним действием.   
$X^{\text{T}} X$ - матрица Грама. $X$ - это исходная матрица размером 1460 x 81 (1460 x 226 после препроцессинга), каждая строка - отдельный объект, каждый столбец - числовое значение признака; $X^{\text{T}}$ - это транспонированная матрица. При транспонировании матрицы мы берем первую строку и превращаем ее в первый столбец, потом берем вторую строку и превращаем ее во второй столбец. Таким образом матрица, к примеру, 2 x 4 становится 4 x 2. То есть строки становятся столбцами, а столбцы строками.    
$\alpha I$ - это альфа, умноженная на единичную матрицу. Единичная матрица - это квадратная матрица, у которой по диагонали стоят 1, а все остальное 0. В мире матриц это является аналогом единицы. Мы умножаем альфу на единичную матрицу, потому что не можем просто число прибавить к матрице, нам нужно сначала это число преобразовать в матрицу. Когда мы прибавляем $\alpha I$ к матрице Грама, мы прибавляем $\alpha$ только к диагонали матрицы. Это "подпирает" веса и гарантирует, что матрицу можно будет инвертировать, даже если признаки почти одинаковые.   
$X^{\text{T}} y$ - исходная матрица, "положенная на бок" и умноженная на вектор правильных ответов. Это своего рода суммарное влияение признаков на результат до того, как мы учли их взаимосвзь.   

Теперь разберем алгоритм более подробно на примере соревнования House Prices. Для удобства возьмем матрицу 1460 x 82, несмотря на то, что после препроцессинга колонок становится 226.   
Итак, сначала мы умножаем транспорированную матрицу ($X^{\text{T}}$ = 82 x 1460) на исходную матрицу ($X$ = 1460 x 82). **Умножать матрицы можно только в том случае, если количество столбцов первой матрицы совпадается с количеством строк второй матрицы**. Это важнейшее правило и умножение возможно только если оно строго соблюдается, то есть умножение возможно только если (m x **n**) * (**n** x k), ответом будет матрица (m x k). Само умножение проводится путем перемножения векторов: умножаем $i$-ю строку первой матрицы на $j$-й столбец второй матрицы, а перед вычислением сначала нужно определить размер результата: сколько строки и столбцов будет у итоговой матрицы. Например, если мы умножаем матрицу 2x2 на другую матрицу 2x2, то мы сначала вычисляем размер: 2 строки и 2 колонки. Далее мы должны умножить строку первой матрицы на колонку второй матрицы. Первый элемент строки умножается на первый элемент колонки, второй элемент строки умножается на второй элемент колонки, а потом результаты произведений складывются и у нас получается первый элемент новой матрицы. Не забываем, что расположение нового числа итоговой матрицы зависит от координат строк и столбцов исходных матриц. Если мы умножаем первую строку на первую колонку, то итоговое число будет имееть координаты (1, 1) в итоговой матрице. Если мы хотим получить число на позиции (2, 2 - правый нижний угол), то нужно перемножать вторую строку на вторую колонку. А если, к примеру, мы хотим получить значение с координатами (2, 1 - левый нижний угол), то нужно перемножать вторую строку на первую колонку:
$$
\begin{pmatrix} 
\color{red}a & \color{red}b \\ 
c & d 
\end{pmatrix} 
\times 
\begin{pmatrix} 
\color{blue}e & f \\ 
\color{blue}g & h 
\end{pmatrix} 
= 
\begin{pmatrix} 
(\color{red}a \cdot \color{blue}e + \color{red}b \cdot \color{blue}g) & (a \cdot f + b \cdot h) \\ 
(c \cdot e + d \cdot g) & (c \cdot f + d \cdot h) 
\end{pmatrix}
$$
Для примера возьмем простое перемножение матриц и рассмотрим, как получилось несколько каких-нибудь значений:
$$
\begin{bmatrix}
1 & 0 & 2 & -1 \\
3 & 1 & 0 & 2 \\
-2 & 4 & 1 & 0
\end{bmatrix}
\times
\begin{bmatrix}
2 & 1 & 0 \\
-1 & 3 & 1 \\
0 & 2 & 4 \\
1 & 0 & -2
\end{bmatrix}
=
\begin{bmatrix}
1 & 5 & 10 \\
7 & 6 & -3 \\
-8 & 12 & 8
\end{bmatrix}
$$
На примере выше видим умножение матрицы (3 x 4) на (4 x 3). Значит итоговая матрица будет размером (3 x 3). Рассмотрима, как найти значение на позиции (2, 2) - пересечение второй строки и второго столбца. Значение на этой позиции - это результат перемножения двух векторов: второй строки первой матрицы и второго столбца второй матрицы. Перемножим их: $3 * 1 + 1 * 3 + 0 * 2 + 2 * 0 = 3 + 3 + 0 + 0 = 6$.    
Теперь попробуем вычислить значение на позиции (1, 3) - пересечение первой строки первой матрицы и третьего столбца второй матрицы: $1 * 0 + 0 * 1 + 2 * 4 + (-1) * (-2) = 0 + 0 + 8 + 2 = 10$  

Ровно по такому же принципу перемножаются матрицы $X^{\text{T}}$ и $X$, образую матрицу Грама, размером (82 x 82).    
Зачем же мы это делаем вообще? Дело в том, что каждый дом - это буквально отдельное уравнение. В нашем случае у нас 1460 уравнений и всего 82 неизвестных. В математике такую систему называют **переопределенной**. Мы не можем найти одно идеальное решение, которое удовлетворит каждый дом на 100%, потому что данные шумные и противоречивые. И когда мы умножаем матрицы $X^{\text{T}}$ и $X$, мы буквально "сплющиваем" информацию о 1460 домах. И каждая ячейка в итоговой матрице (82 x 82) - это скалярное произведение двух столбцов признаков, потому что когда мы перемножаем эти 2 матрицы, строки из признаков на колонки из признаков. И вся матрица Грама - это матрица, где колонки и строки - это признаки. Таким образом мы получаем матрицу, в которой по диагонали (с верхнего левого угла и до правого ниженего) располагаются суммы квадартов каждого значения (сначала сумма квадартов $x_1$, потом $x_2$ и так далее до $x_{82}$). Каждый элемент этой диагонали называется **нормой** вектора. Она показывает общий масштаб этого признака. Если у признака огромный масштаб (значение в ячейки слишком большое), то модели нужен крошечный вес, чтобы не переборщить с итоговой оценкой. Все остальные числа матрицы Грама, которые располгаются за пределами этой диагонали, отображают **взаимную направленность**. Когда мы умножаем, к примеру, вектор площади $x_1$ на вектор количества комнат $x_2$, то получаем значение, которое показывает, насколько эти признаки согласованы. Если число большое и положительное, значит признаки $x_1$ и $x_2$ дублируют друг друга (чем больше комнат, тем больше площадь дома). А если число около нуля, значит признаки независимы, они несут уникальную информацию. Это нужно, чтобы не давать большой вес признакам, которые дублируют информацию, иначе будем учитывать одну и ту же информацию дважды.    

Далее мы прибавляем к этой матрице Грама значение $\alpha$. Поскольку $\alpha$ - это одно единственное число, то мы не можем прибавить его к матрице. Сначала нужно превратить его матрицу. Для этого умножим $\alpha$ на единичную матрицу $I$. Единичная матрица - это аналог обычной единицы в арифметике, если мы умножим какое-то число на 1, то результатом будет это же самое число. С единичной матрицой также, $\alpha$ останется $\alpha$, просто превратится в матрицу. Сама по себе единичная матрица является квадратной матрицей (число колонок совпаадет с числом строк), чья главная диагональ (от левого верхнего угла до правого нижнего) состоит из единиц, а все остальные элементы равны нулю.   
Перед умножением нужно определить, какого размера будет единичная матрица. Как мы помним, складывать матриы можно только если они одинаковго размера. Первая матрица (матрица Грама) имеет размер (82 x 82), значит единичная матрица должна иметь ровно такой же размер (82 x 82). Теперь можно переходить к умножению на $\alpha$. Умножение матрицы на число называется **скалярным умножением**. При скалярном умножении мы умножаем скаляр на каждый элемент матрицы. Соответственно, результатом будет матрица, у которой главная диагональ состоит из значений $\alpha$, а все, что за пределами этой диагонали - нули.  

Теперь мы можем прибавить к матрице Грама нашу матрицу $\alpha$. Вспоминаем, что **складывать матрицы можно только если они абсолютно одинакового размера**. У нас обе матрицы имеют размер (82 x 82), значит можем приступать к сложению. Сложение матриц более простая операция, чем перемножение. Достаточно просто сложить 2 числа на одинаковых позициях:
$$
\begin{pmatrix}
a_{11} & a_{12} \\
a_{21} & a_{22}
\end{pmatrix}
+
\begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix}
=
\begin{pmatrix}
a_{11} + b_{11} & a_{12} + b_{12} \\
a_{21} + b_{21} & a_{22} + b_{22}
\end{pmatrix}
$$
Ровно таким же образом мы складываем матрицу Грама и матрицу $\alpha$.   

Для чего мы прибавляем к матрице Грама матричную $\alpha$? На это есть несколько причин.   
Во-первых, аналитический метод решения требует возведения матрицы в степень $-1$. Это операция **инверсии**, ее суть в том, чтобы найти такую матрицу, которая при умножении с исходной, дала бы единичную матрицу. Причем порядок множетелей тут не важен, так как действует правило $A * A^{-1} = A^{-1} * A = I$. И если 2 признака идеально дублируют друг друга, матрица Грама становится "вырожденной" (определитель равен 0). Определитель(детерминант) - это число, которое характеризует матрицу. Его можно представить как "объем" или "площадь", которую образуют векторы признаков. Например, если у нас есть 2 признака: "площадь в метрах" и "площадь в сантиметрах", то с точки зрения математики, один вектор лежит на другом, и "площадь", которую образуют эти 2 признака равна нулю. Матрица, построенная на таких признаках, считается **вырожденной**. И попытка инвертировать такую матрицу, все равно что пытаться поделить на ноль. Так как для того, чтобы вычислить обратную матрицу, все элементы исходной матрицы должны делиться на ее определитель, а если определитель равен нулю, то и поделить не получится. Поэтому мы добавялем к исходной матрице Грама матричную $\alpha$, чтобы чуть-чуть изменить числа на диагонали, чтобы определитель стал хотя бы крошечным, но не нулевым. Таким образом мы делаем матрицу Грама **полноранговой**. Ранг матрицы - это количество "полезных", уникальных столбцов, которые не являются копиями друг друга. И в нашем случае, если матрица Грама полноранговая, то ее ранг равен 82. Это значит, что ни один признак нельзя идеально предсказать через другие. В такой матрице нет лишнего "мусора", и она всегла инвертируется. Каким образом увеличение числа на диагонали "разлепляет" признаки? Дело в том, что мы меняем только значения $x_1 * x_1, x_2 * x_2, x_3 * x_3$ и т.д. А внедиагональные элементы остаются прежними. Даже если признаки $x_1$ и $x_2$ были идентичны, после добавления $\alpha$ их "мощности" на диагонали чут-чуть изменятся относительно их взаимной связи. И эти небольшие изменения нарушают идеальную пропрорцию между строками. Векторы в многомерном пространстве меняются и перестают ровно лежать друг на друге.   
Во-вторых, смысл прибавления у матричной $\alpha$ такой же, как и в других методах нахождения весов. Чем больше число на диагонали матрицы Грама, тем меньший вес модель хочет присовить этому признаку. И когда мы прибавляем к диагональным значениям значения $\alpha$, мы дополнительно завышаем мощность каждого признака. Это заставит модель еще силнее уменьшить вес для признаков.   

Далее необходимо провести операцию инверсии. Итоговую матрицу после суммирования нужно возвести в степень -1. Как уже ранее было сказано, оперция инверсии - это поиск матрицы, которая при умножении на исходную матрицу давала бы единичную матрицу. Важно помнить, что **инвертировать можно только квадратную матрицу**. И итоговая матрица как раз является квадратной (82 x 82).    
Для небольших матриц используется **метод присоединенной матрицы**. Но для больших матриц такой метод не подойдет. Для вычисления инвертированной матрицы будем использовать классический инженерный стандарт - **LU-разложение** (или по другому "метод Гаусса"). Вместо того, чтобы инвертировать матрицу "в лоб", мы представляем ее ($A$) как произведение двух треугольных матриц:
$$A = L * U$$
где    
$L$ (Lower) - нижняя треугольная матрица (числа только внизу и на диагонали)    
$U$ (Upper) - верхняя треугольная матрица (числа только вверху и на диагонали)   

Далее необходимо инвертировать каждый из множителей ($L^{-1}$ и $U^{-1}$). После этого мы сможем инвертировать матрицу $A$, она будет равна произведению инвертированных $L$ и $U$, но только теперь нужно умножать $U^{-1}$ на $L^{-1}$, а не наоборот:
$$A^{-1} = U^{-1} * L^{-1}$$

Рассмотрим, как это реализуется на примере матрицы размером (3 x 3):
$$
A = \begin{pmatrix}
2 & 1 & 1 \\
4 & 3 & 3 \\
8 & 7 & 9
\end{pmatrix}
$$
**Шаг №0. Подготовка**. Для начала нам нужно создать заготовки для $L$ и $U$. На данном этапе $U$ - это копия $A$, а $L$ - это единичная матрица:
$$
L = \begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}, \quad
U = \begin{pmatrix}
2 & 1 & 1 \\
4 & 3 & 3 \\
8 & 7 & 9
\end{pmatrix}
$$
**Шаг №1. Работаем с первым столбцом**. Нам нужно обнулить числа под двойкой в первом столбце матрицы $U$ (это числа 4 и 8). Если бы матрица была больше, мы бы обнуляли не 2 числа, а столько, сколько располагается под ведущим элементом ($A_{11}$ - это в данном случае 2).   
Для обнуления необходимо вычислить множители $m$. Множитель $m$ - это ответ на вопрос "на сколько нужно умножить число на диагонали, чтобы получить число под ним?". Таким образом, мы делим каждое число под ведущим элементом на ведущий элемент:   
$m = 4/2 = 2$    
После этого в матрице $U$ из второй строки вычитаем первую строку, умноженную на множитель $m$:  
- $4 - (2 * 2) = 0$
- $3 - (1 * 2) = 1$
- $3 - (1 * 2) = 1$    

После этого записываем в $L$ множитель 2 в позицию (2, 1). Почему мы записываем $m$ в $L$, да еще и на позицию (2, 1)? Дело в том, что матрица $L$ - это "память" о том, как мы меняли исходную матрицу $A$, чтобы превратить ее в $U$. А также мы меняем строку, из которой вычитали на ту, которую получили. То есть вторую строку в матрице $U$ мы заменяем на 0, 1, 1. 

Далее считаем множитель $m$ следующего элемента:    
$m = 8/2 = 4$    
Вычитаем из третьей строки первую строку, умноженную на 4:
- $8 - (2 * 4) = 0$
- $7 - (1 * 4) = 3$
- $9 - (1 * 4) = 5$ 

После этого вписываем в $L$ множитель 4 в позицию (3, 1). И заменяем 3-ю строку в матрице $U$ на получившуюся по итогу вычислений: 0, 3, 5.    

Промежуточный итог выглядит так:
$$
L = \begin{pmatrix}
1 & 0 & 0 \\
2 & 1 & 0 \\
4 & 0 & 1
\end{pmatrix}, \quad
U = \begin{pmatrix}
2 & 1 & 1 \\
0 & 1 & 1 \\
0 & 3 & 5
\end{pmatrix}
$$

**Шаг №2. Работаем со вторым столбцом**. Теперь нам нужно обнулить число под диагональю во втором столбце матрицы $U$. Работаем с числом 3 (именно оно одное-единственное располагается под диагональю во втором столбце). Вычисляем множитель $m$:   
$m = 3/1 = 3$    
И начинаем вычитать из третьей строки вторую, умноженную на множитель 3:
- $0 - (0 * 3) = 0$
- $3 - (1 * 3) = 0$
- $5 - (1 * 3) = 2$

В матрице $L$ ставим множитель на позицию (3, 2). А в матрице $U$ заменяем третью строку на 0, 0, 2:
$$
L = \begin{pmatrix}
1 & 0 & 0 \\
2 & 1 & 0 \\
4 & 3 & 1
\end{pmatrix}, \quad
U = \begin{pmatrix}
2 & 1 & 1 \\
0 & 1 & 1 \\
0 & 0 & 2
\end{pmatrix}
$$

Теперь у нас "на руках" две треугольные матрицы. И если мы их перемножим, то получим исходную матрицу $A$. То есть мы разложили матрицу $A$ на 2 множителя.   

Теперь нам нужно инвертировать каждую из них. Для треугольных матриц используется **метод обратной подстановки**.    
Инверсия для матрицы $L$ - это поиск такой матрицы $L^{-1}$, что $L * L^{-1} = I$. Мы ищем столбцы этой новой матрицы по одному. Сначала инвертируем элементы на самой диагонали. На главной диагонлаи матрицы $L$ стоят только единицы, а нули находятся справа от диагонали. Слева же расположены множители. Найдем первый столбец матрицы $L^{-1}$:
$$
\begin{pmatrix}
x_{11} \\
x_{21} \\
x_{31}
\end{pmatrix}
$$
Для того, чтобы найти значения первого столбца, мы решаем простое уравнение, в котором перемножаем треугольную матрицу $L$ на первый столбец матрицы $L^{-1}$(который мы ищем) и получаем первый столбец единичной матрицы:
$$
\begin{pmatrix}
1 & 0 & 0 \\
2 & 1 & 0 \\
4 & 3 & 1
\end{pmatrix} \cdot \begin{pmatrix} 
x_{11} \\ 
x_{21} \\ 
x_{31} 
\end{pmatrix} = \begin{pmatrix} 
1 \\ 
0 \\ 
0 
\end{pmatrix}
$$
Мы уже знаем, как перемножаются матрицы, поэтому приступим к вычислению:    
$1 * x_{11} + 0 * x_{21} + 0 * x_{31} = 1 => x_{11} = 1$    
$2 * x_{11} + 1 * x_{21} + 0 * x_{31} = 0 => 2 * x_{11} + x_{21} = 0 => 2 + x_{21} = 0 => x_{21} = -2$    
$4 * x_{11} + 3 * x_{21} + 1 * x_{31} = 0 => 4 - 6 + x_{31} = 0 => x_{31} = 2$     
Таким образом мы получили первый столбец матрицы $L^{-1}$. Перейдем к нахождению второго столбца. В случае со вторым столбцом, нужно приравнять произведение уже ко второму столбцу единичной матрицы:
$$
\begin{pmatrix}
1 & 0 & 0 \\
2 & 1 & 0 \\
4 & 3 & 1
\end{pmatrix} \cdot \begin{pmatrix} 
x_{12} \\ 
x_{22} \\ 
x_{32} 
\end{pmatrix} = \begin{pmatrix} 
0 \\ 
1 \\ 
0 
\end{pmatrix}
$$
Приступим к вычислению:    
$1 * x_{12} + 0 * x_{22} + 0 * x_{32} = 0 => x_{12} = 0$    
$2 * x_{12} + 1 * x{22} + 0 * x_{32} = 1 => 2 * x_{12} + x_{22} = 1 => x_{22} = 1$    
$4 * x_{12} + 3 * x_{22} + 1 * x_{32} = 0 => 3 + x_{32} = 0 => x_{32} = -3$    
Второй столбец матрицы $L^{-1}$ получен, переходи к вычислению третьего столбца. Вычисляем по формуле:
$$
\begin{pmatrix}
1 & 0 & 0 \\
2 & 1 & 0 \\
4 & 3 & 1
\end{pmatrix} \cdot \begin{pmatrix} 
x_{13} \\ 
x_{23} \\ 
x_{33} 
\end{pmatrix} = \begin{pmatrix} 
0 \\ 
0 \\ 
1 
\end{pmatrix}
$$
Переходим к заключительному вычислению:   
$1 * x_{13} + 0 * x_{23} + 0 * x_{33} = 0 => x_{13} = 0$    
$2 * x_{13} + 1 * x_{23} + 0 * x_{33} = 0 => x_{23} = 0$    
$4 * x_{13} + 3 * x_{23} + 1 * x_{33} = 1 => x_{33} = 1$    

Теперь мы можем собрать все получившиеся столбцы воедино и посмотреть на итоговую матрицу $L^{-1}$:
$$
L^{-1} = \begin{pmatrix}
1 & 0 & 0 \\
-2 & 1 & 0 \\
2 & -3 & 1
\end{pmatrix}
$$

Теперь перейдем к созданию обратной матрицы $U$. Принци по большей части тот же самый, единственная разница заключается в том, что нам нужно будет вычислять снизу вверх, так как именно внизу матрицы будет 1 неизвестная, а сверху сразу 3.    
Начнем вычисление первого столбца обратной матрицы:
$$
\begin{pmatrix}
2 & 1 & 1 \\
0 & 1 & 1 \\
0 & 0 & 2
\end{pmatrix} \cdot
\begin{pmatrix}
x_{11} \\
x_{21} \\
x_{31}
\end{pmatrix}
=
\begin{pmatrix}
1 \\
0 \\
0
\end{pmatrix}
$$
Как видим, вычисление нужно начать с последней строки, так как именно при умножении последней стороки на вектор, мы получим уравнение с одним неизвестным:    
$0 * x_{11} + 0 * x_{21} + 2 * x_{31} = 0 => x_{31} = 0$    
$0 * x_{11} + 1 * x_{21} + 1 * x_{31} = 0 => x_{21} = 0$    
$2 * x_{11} + 1 * x_{21} + 1 * x_{31} = 1 => 2 * x_{11} = 1 => x_{11} = 1/2$    
Получили значения первого столбца обратной матрицы $U$: $(1/2, 0, 0)^{\text{T}}$   

Теперь вычислим второй столбец $U^{-1}$:
$$
\begin{pmatrix}
2 & 1 & 1 \\
0 & 1 & 1 \\
0 & 0 & 2
\end{pmatrix} \cdot \begin{pmatrix} 
x_{12} \\ 
x_{22} \\ 
x_{32} 
\end{pmatrix} = \begin{pmatrix} 
0 \\ 
1 \\ 
0 
\end{pmatrix}
$$
Перейдем к вычслению:   
$0 * x_{12} + 0 * x_{22} + 2 * x_{32} = 0 => x_{32} = 0$    
$0 * x_{12} + 1 * x_{22} + 1 * x_{32} = 1 => x_{22} = 1$    
$2 * x_{12} + 1 * x_{22} + 1 * x_{32} = 0 => 2 * x_{12} + 1 = 0 => x_{12} = -1/2$   
Нашли второй столбец обратной матрицы $U$: $(-1/2, 1, 0)^{\text{T}}$     

И по этой формуле ищем третий столбец:
$$
\begin{pmatrix}
2 & 1 & 1 \\
0 & 1 & 1 \\
0 & 0 & 2
\end{pmatrix} \cdot \begin{pmatrix} 
x_{13} \\ 
x_{23} \\ 
x_{33} 
\end{pmatrix} = \begin{pmatrix} 
0 \\ 
0 \\ 
1 
\end{pmatrix}
$$
Вычисляем:    
$0 * x_{13} + 0 * x_{23} + 2 * x_{33} = 1 => 2 * x_{33} = 1 => x_{33} = 1/2$    
$0 * x_{13} + 1 * x_{23} + 1 * x_{33} = 0 => x_{23} + 1/2 = 0 => x_{23} = -1/2$    
$2 * x_{13} + 1 * x_{23} + 1 * x_{33} = 0 => 2 * x_{13} -1/2 + 1/2 = 0 => x_{13} = 0$   
И последний столбец обратной матрицы $U$: $(0, -1/2, 1/2)^{\text{T}}$   

Итоговый вид матрицы $U^{-1}$:
$$
U^{-1} = \begin{pmatrix}
\frac{1}{2} & -\frac{1}{2} & 0 \\
0 & 1 & -\frac{1}{2} \\
0 & 0 & \frac{1}{2}
\end{pmatrix}
$$

Теперь мы можем выполнить последнее действие для того, чтобы получить матрицу $A^{-1}$. Для этого нужно умножить матрицу $U^{-1}$ на $L^{-1}$:
$$
U^{-1} \cdot L^{-1} = 
\begin{pmatrix}
\frac{1}{2} & -\frac{1}{2} & 0 \\
0 & 1 & -\frac{1}{2} \\
0 & 0 & \frac{1}{2}
\end{pmatrix}
\begin{pmatrix}
1 & 0 & 0 \\
-2 & 1 & 0 \\
2 & -3 & 1
\end{pmatrix}
=
\begin{pmatrix}
\frac{3}{2} & -\frac{1}{2} & 0 \\
-3 & \frac{5}{2} & -\frac{1}{2} \\
1 & -\frac{3}{2} & \frac{1}{2}
\end{pmatrix}
$$
Таким образом мы получили матрицу $A^{-1}$:
$$
A^{-1} = 
\begin{pmatrix}
\frac{3}{2} & -\frac{1}{2} & 0 \\
-3 & \frac{5}{2} & -\frac{1}{2} \\
1 & -\frac{3}{2} & \frac{1}{2}
\end{pmatrix}
$$

Ровно по такому же принципу и вычисляется инвертированная матрица Грама с $\alpha$. Но зачем нам инвертировать матрицу Грама с $\alpha$? Дело в том, что когда мы берем производную от функции потерь Ridge-регрессии (MSE + регуляризация) и приравниваем ее к нулю (потому что когда производная равна нулю, это значит, что мы дошли до дна параболы), у нас получается базовое линейное уравнение. В матричном виде оно записывается так:
$$(X^{\text{T}} X + \alpha I) w = X^{\text{T}} y$$
Тут $w$ - это то неизвестное, которое нам нужно найти. Это уравнение вида $A x = b$, где:    
$A = (X^{\text{T}} X + \alpha I)$ - матрица Грама с матричной альфой     
$b = X^{\text{T}} y$ - вектор связи признаков с ответами    

Чтобы вытащить $w$ из-под матрицы, мы не можем поделить на матрицу, так как в матричной алгебре нет такой операции. Вместо этого мы умножаем обе части уравнения на обратную матрицу $(X^{\text{T}} X + \alpha I)^{-1}$:
$$\underbrace{(X^T X + \alpha I)^{-1} \cdot (X^T X + \alpha I)}_{I} \cdot w = (X^T X + \alpha I)^{-1} \cdot X^T y$$
И таким образом мы вычленяем $w$.    

Остается только вычислить $X^T y$. Что это вообще такое? Если $X^T X$ - это матрица, которая отображает взаимосвязь между признаками, то $X^T y$ - это матрица, которая отображает взаимосвязь каждого признака с целевой переменной. При умножении $X^T y$ мы получаем вектор-столбец (82 x 1). Каждое число в этом векторе - это скалярное произведение столбца признака на столбец цен. Оно показывает "суммарный вклад" или корреляцию конкретного признака с ценой. Если, к примеру, признак "Площадь" сильно влияет на цену, то в векторе $X^T y$ на соответствующей позиции будет очень большое число. Если же признак, к примеру, "Цвет забора" никак не связан с ценой, там будет число близкое к нулю.   
Перемножение матриц мы уже разбирали ранее, поэтому с этим проблем быть не должно. Вычисления по данной формуле дадут нам вектор оптимальных весов. 

##### Кросс-валидация
Разберем еще одну важнейшую тему - *кросс-валидация*. Кросс-валидация необходима для подбора гиперпараметра $\alpha$. Что из себя представляет кросс-валидация? Это метод, который позволяет найти оптимальное $\alpha$ методом разбивания датасета на **K-folds**.    
К примеру, у нас есть тренировочный датасет в 1000 строк. При кросс-валидации мы разбиваем весь этот датасет на несколько частей, к примеру, на пять. Таким образом у нас получается 5 датасетов - 5-folds по 200 строк каждый. Первый фолд будет являться валидационным. Остальные фолды (2, 3, 4 и 5) предназначены для обучения. Модель обучается на этих 800 строках и проверяется на качество на первом фолде. Далее следует еще одна такая же итерация, только уже берется в качестве валидационного датасета - второй фолд. Модель снова учится на 800 строках (1, 3, 4 и 5 фолды). И проверяется на валидационной выборке, на втором фолде. И таких итераций должно быть столько, сколько всего фолдов. Абсолютно каждый фолд должен побывать в роли валидационного. На каждой такой итерации мы получаем метрику качества. А потом из этих метрик выводим среднее значение, которое показывает то, насколько хорош $\alpha$-параметр.   
Таким образом мы тестируем несколько значений $\alpha$ и выбираем наиболее подходящий - тот, который дает наилучшую метрику качества.   

Перейдем к реализации кросс-валидации. Ранее мы закончили на разбиении датасета. Мы разделили `X` и `y` на тренировочные и валидационные выборки. Обучили препроцессор на X_train и преобразовали датасеты в матрицы с помощью препроцессора, получив `X_train_processed` и `X_val_processed`.   

Есть 3 варианта реализации кросс-валидации:
- Перебор $\alpha$ вручную в цикле
- Автоматизация через GridSearchCV
- Использование вструенных инструментов (RidgeCV, LassoCV, ElasticNetCV)

Разберем каждый из вариантов.

##### Перебор $\alpha$ вручную в цикле
При переборе вручную в цикле, у нас есть 2 варианта реализации: без использования Pipeline и с использованием Pipeline.   
Вариант без использованиея Pipeline нежелателен, так как в нем легко можно допустить утечку информации, и он занимает слишком много строчек в проекте.   
Как можно допустить утечку данных при таком подходе? Представим, что наш `X_train_processed` состоял из 1000 строк. Препроцессор посчитал медианы и средние значения по всей 1000 строк. Когда `cross_val_score()` (это функция, с помощью которой мы реализуем кросс-валидацию вручную, познакомимся с ней позже) взял 200 строк (1 фолд) в качестве валидационной выборки, эти 200 строк внутри себя уже содержат изменения, основанные на медианах и средних значениях всего `X_train`. Модель на тренировочных фолдах будет оцениваться по данным, которые неявно "знают" распределение валидационного фолда. Чтобы кросс-валидация была честной, препроцессор должен вычислять медианы и средние только по тем 4 фолдам (800 строк), которые выбраны для обучения внутри текущей итерации цикла, и ничего не знать про оставшиеся 200 строк.    
В реализации такой подход достаточно утомителен, поэтому лучше использовать Pipeline, который защищиает от утечки данных. Его и рассмотрим.   

Когда мы реализуем кросс-валидацию с использованием Pipeline, нам не нужно обучать препроцессор. Достаточно только создать его и разделить датасет на тренировочный и валидационный. Строки `preprocessor.fit()` и `preprocessor.transform()` не нужны.   

Теперь можем создать массив из значений гиперпараметров и пустой массив, который будет содержать в себе результаты каждой итерации:
```
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

alphas = [0.1, 0.5, 1, 5, 10, 20, 50, 100]
cv_scores = []
```
Мы подобрали ряд возможных значений гиперпараметра. Теперь можно запускать цикл `for`, в котором на каждой итерации будет обучаться модель с одним конкретным гиперпараметром:
```
for alpha in alphas:
    pipe = Pipeline(steps=[
        ('prep', preprocessor),
        ('ridge', Ridge(alpha=alpha))
    ])
```
Как видно из кода выше, мы на каждой итерации будем создавать пайплайн, который состоит из препроцессора и модели с гиперпараметром, равным гиперпараметру текущей итерации. Данный пайплайн соединяет препроцессор и модель.   

Далее, в этом же цикле, ниже пайплайна прописываем:
```
scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='neg_mean_squared_error')

```
`cross_val_score()` - это и есть та самая функция, которая реализует кросс-валидацию. Разберем, какие аргументы она принимает. "pipe" - это наш estimator, в данном случае мы используем не модель, а именно пайплайн. Это значит, что следующему параметру мы должны передать сырой датасет, ведь обработка препроцессором уже заложена в пайплайне. "X_train" - это и есть наш сырой датасет. Заметьте, хоть мы и создавали "X_train_preprocessed", в данном случае мы им не воспользовались. Мы передаем именно сырой датасет. Следом идет "y_train", наш вектор целевой переменной. Далее передаем в параметр "cv" значение "5". Это говорит о том, что мы будем делить датасет на 5 частей. И последний аргумент "scoring", которому мы передаем значение "neg_mean_squared_error" (отрицательная MSE). Это наша метрика. И если с первыми четырьмя параметрами все понятно, то на "scoring" мы заострим внимание, ведь в переменную scores попадают результаты вычисления этой метрики для каждого фолда.   

Тут нужно небольшое отступление. В scikit-learn заложено математическое правило разработки: **чем выше значение метрики, которую возвращает встроенная функция оценки, тем лучше модель**. Это назвается "функция максимизации". Такие ошибки, как MSE или RMSE являются метриками ошибки. Для них правло работает ровно наоборот: **чем меньше ошибка, тем лучше модель**. Ошибка 0.0 - это идеал.   
Чтобы не ломать внутреннюю логику библиотеки (где "больше = лучше"), разработчики scikit-learn решили взять стандартную ошибку MSE и умножить ее на -1. Поэтому метрика называется `neg_mean_squared_error` (Negative MSE). Если модель сработала идеально, ее реальная MSE = 0.0. Функция вернет 0.0. Если модель сработала хорошо, то ее реальная MSE = 0.02. Функция вернет -0.02. Если же модель сработала плохо, то ее MSE = 0.15. Функция вернет -0.15.   
И мы видим, что число -0.02 больше -0.15, а 0.0 больше -0.02. А значит фундаментальное правило библиотеки работает. Чем больше, тем лучше.   

Итак, таким образом получается, что на каждой итерации мы получаем по 5 значений Negative MSE. Почему 5? Потому что параметр cv=5, значит делим датасет на 5 частей. И соответственно, у нас за одну итерацию - 5 оценок. Переменная `scores` является массивом NumPy из пяти отрицательных чисел.   

А далее мы переводим эту метрику в RMSLE на каждой итерации:
```
rmsle = np.sqrt(-scores.mean())
```
`-scores.mean()` - это конструкция, которая превращает массив отрицательных чисел в положительное среднее число. Действия выполняются изнутри наружу. То есть в первую очередь вычисляется `mean()`, а потом уже умножается результат на минус. Метод `mean()` определен в numpy.ndarray и возвращает среднее арифметическое по массиву numpy. После этого мы "умножаем" на минус, чтобы избавиться от минуса (минус на минус = плюс).    
После этого мы получаем из массива Negative MSE одно обычное MSE на каждой итерации.   

Далее мы вычисляем квадрат получившегося числа с использованием функции `np.sqrt()`. И таким образом мы получаем RMSE. Но, как вы заметили, переменная называется `rmsle`, а не `rmse`. Но на самом деле, поскольку мы работали с логарифмированной целевой переменной, мы получили именно RMSLE. Фактически и физически, функция `cross_val_score()` считала отрицательный MSLE, просто она об этом "не знала", потому что для нее это были обычные числа "y".   

Но параметр `scoring` может принимать значение `neg_mean_squared_log_error`. Почему же мы не указали данное значение, раз работаем с логарифимрованной ценой? Дело в том, что тогда бы функция попыталась бы еще раз взять логарифм от уже логарифмированного SalePrice_log. Обычно, когда применяется функция `cross_val_score()`, то мы можем либо передать обычную цену и посчитать `neg_mean_squared_log_error`, либо мы можем сами перевести таргет в логарифм, а модель попросить посчитать обычный `neg_mean_squared_error`, на стыке этих двух действий MSE автоматически превращается в MSLE.    

Далее мы просто на каждой итерации добавляем получившееся значение RMSLE в наш пустой массив `cv_scores` с помощью метода массивов `append()`:
```
cv_scores.append(rmsle)
```

После того, как мы закончили все итерации в цикле и получили массив из 8 чисел (результатов RMSLE), мы должны выбрать наилучший. Применим для этого функцию NumPy `argmin()`, которая принимает массив и возвращает индекс самого мальнького элемента в массиве. Не забываем, что в массиве RMSLE, а значит нам нужен наименьший результат, так как чем он ниже, тем меньше модель ошибается:
```
best_alpha = alphas[np.argmin(cv_scores)]
```
И поскольку количество элементов в alphas и cv_scores одинаковое, мы вычленяем индекс наименьшего RMSLE и используем его в массиве alphas в качестве индекса.   

Полный код выглядит вот так:
```
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

alphas = [0.1, 0.5, 1, 5, 10, 20, 50, 100]
cv_scores = []

for alpha in alphas:
    pipe = Pipeline(steps=[
        ('prep', preprocessor),
        ('ridge', Ridge(alpha=alpha))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    rmsle = np.sqrt(-scores.mean())
    cv_scores.append(rmsle)

best_alpha = alphas[np.argmin(cv_scores)]
```

##### Подбор $\alpha$ с помощью GridSearchCV
Теперь перейдем ко второму способу определения оптимального значения $\alpha$ - GridSearchCV. Тут также, как и в первом случае, можно обойтись без Pipeline. Но без него пришлось бы писать вложенные циклы, чтобы избежать утечки данных. Поэтому будем использовать Pipeline.   

Начнем реализацию:
```
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

pipe = Pipeline(steps=[
    ('prep', preprocessor),
    ('ridge', Ridge())
])
```
Обратите внимание на то, что мы не пишем в `Ridge()` аргумент `alpha`, а оставляем скобки пустыми.    

Далее задаем сетку параметров:
```
param_grid = {'ridge__alpha': [0.1, 0.5, 1, 5, 10, 50, 100]}
```
Это один из обязательных параметров класса GridSearchCV. Аргументом должен быть словарь, в котором ключ - это наименование шага, в котором определяем модель + __ + наименование параметра. А значением всегда является массив проверяемых значений.   

Далее инициализируем объект класса GridSearchCV и запускаем поиск по сетке:
```
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid.fit(X_train, y_train)
```
`pipe` - это наш пайплайн. `param_grid` - сетка паарметров, которую будем перебирать. `cv=5` - делим датасет на 5 фолдов. `scoring='neg_mean_squared_error'` - в качестве метрики качества используем отрицательную MSE. `n_jobs=-1` - задействем все ядра компьтера. Потом используем метод `fit()`, чтобы обучить grid.    

Также важно заметить то, что grid имеет еще один важный параметр `refit`, который по умолчанию имеет значение True. Это значит, что когда grid перебрал все значения и нашел лучшее, он переобучит модель на всех переданных параметрах и сипользованием оптимального alpha. И объект grid после выполнения `fit()` уже является готовой обученной моделью. Мы даже после этого можем у нее вызвать метод `predict()`.   

Теперь разберемся, как вытащить из GridSearchCV RMSLE. Для этого есть атрибут `cv_results_`, который появляется у объектов GridSearchCV сразу же после вызова `fit()`. Он содержит словарь, в котором GridSearchCV сохраняет всю историю вычислений по каждому фолду и по каждой комбинации параметров. Для того, чтобы превратить его в таблицу, можно использовать класс DataFrame:
```
results = pd.DataFrame(grid.cv_results_)
```
Потом из этой таблицы вытаскиваем колонку средней MSE, убираем минус, берем корень и создаем колонку RMSLE:
```
results['mean_rmsle'] = np.sqrt(-results['mean_test_score'])
```
А потом выводим результат:
```
print(results[['param_ridge__alpha', 'mean_rmlse']])
```
На экране появится чистый лог, где для каждой альфы будет написан ее RMLSE.    

#### Model Deployment
Итак, когда исслледование закончено и мы определились с тем, как будет выглядеть наша модель, мы можем перейти к развертыванию. Развертывание модели имеет несколько принципиальных отличий от того, что мы делали в Jupyter Notebook. В исследовательской части наша работа была четко разделена на данные пукнты:
1) EDA - первичный анализ данных. Мы смотрели на размер табличных данных и количество пропусков. 
2) Preprocessing - подготовка табличных данных. Тут мы устраняли пропуски и логарифмировали целевую переменную.  
3) Feature Engineering - создание новых признаков. На этом этапе мысоздавали новые колонки в наших табличных данных. 
4) Feature Transformation - преобразование данных. Тут мы делили таблицу на разные подгруппы (числовые, порядковые и категориальные признаки), создавали препроцессор, делили данные на тренировачные и валидационные и обучали препроцессор. В общем, преобразовывали табличные данные в матрицу. 
5) Modeling - построение модели. В данном пункте мы проектировали модель, подбирали лучший гиперпараметр и регуляризацию.   

При разворачивании модели дела обстоят иначе. Мы создаем в корне проекта папку "service", в которой необходимо создать 3 файла: `main.py`, `shemas.py`, `train.py`. В main.py настраивается API-сервер. Файл отвечает за прием входящих запросов, передачу данных в модель и отправку ответа пользователю. Файл shemas.py - это контракт данных. В нем описывается труктура входящих и исходящих данных. Он гарантирует, что сервис не "упадет", если пользователь прешлет текст вместо числа. В train.py содержится логика обучения. Файл содержит код для загрузки данных, предобработки, обучения модели и сохранения готового файла (весов) в формате ".pkl", ".onnx" или ".h5".    

##### train.py
Начнем разбор файла `train.py`. В данном файле:
- Настраиваем импорты (подключаем инструменты)
- Загружаем данные (через pandas.read_csv(), а также логарифмируем цены) 
- Разделение на списки колонок (необходимо, чтобы Pipeline знал, какой инструмент к какой колонке применить)
- Создание Pipeline (фундамент деплоя)
- Тренировка модели (вызов метода fit())

Для начала необходимо сформировать виртуальное окружение. То есть нам необходимо создать среду для проекта, в котором разработка велась бы с использованием библиотек конкретных версий. К примеру, мы можем в терминале вызвать команду `pip list` и увидеть все библиотеки, которые установлены в глобальной среде.    
Процесс реализации описан для Windows. Перейдем к реализации виртуального окружения - `venv`. Для формирования виртуального окружения под проект, перейдем в терминале в папку проекта и пропишем команду `python -m venv venv`. После этого в корне проекта появится папка `venv` (обычно ее добавляют в .gitignore). После этого необходимо активировать эту папку. Активация производится командой `venv\Scripts\activate`. После этого в терминале слева от пути высветится пометка `(venv)`, которая будет зеленого цвета. Команду `venv\Scripts\activate` нужно будет вводить каждый раз, когда мы захотим поработать с проектом, просто пеерйти в корень в терминале будет недостаточно. Мы не входим в виртуальное окружение автоматически, только по команде.    

Теперь, когда виртуальное окружение готово, можно приступить к установке в это окружение бибилиотек. При активном `(venv)`, необходимо прописать команду `python -m pip install library`, где вместо "library" пишется наименование бибилиотеки. После этого бибилиотека будет установлена в виртуальное окружение. Но это не совсем удобно. Гораздо удобнее создать в корне проекта обычный текстовый файл `requirements.txt`, в котором перечислить все необходимые в работе инструменты. Содержимое файла должно выглядеть примерно так:
```
fastapi==0.109.0
joblib==1.3.2
numpy==1.26.4
pandas==2.2.0
```
После того, как мы создадим такой файл, необходимо ввести команду `python -m pip install -r requirements.txt`. Флаг `-r` означает, что после него мы введем путь к файлу с перечнем бибилиотек.   

Что касается Python, то его устанавливать в venv отдельно не нужно, так как глобальный Python скопировал себя внутрь папки проекта. Он лежит в venv/Scripts и называется `python.exe`. Чтобы проверить глобальную версию, нужно использовать команду `python --version`, а чтобы проверить локальную версию, нужно в venv прописать `python -V`.    

Также важно отметить то, что мы в будущем будем контейнеризировать наше приложение. И когда мы начинаем работать уже с развертыванием модели, нам обязательно нужно создать venv и файл requirements.txt. В файле requirements.txt мы прописываем все необходимые библиотеки для работы. Потом устанавливаем их в venv. После этого в venv вызываем перечень всех установленных библиотек с помощью команды `pip list`. В перечне мы увидим гораздо больше бибилиотек, чем мы устанавливали. Дело в том, что бибилиотеки часто "подтягивают" за собой и другие библиотеки, которые как раз и отобразятся в этом перечне. Поэтому мы копируем этот перечнь и вставляем в requirements.txt. Данный файл с полным перечнем бибилиотек мы потом будем использовать для контейнеризации приложения.   

Теперь, когда мы установили все необходимые инструментв в venv, перейдем к реализации файла train.py. Для начала настраиваем импорты:
```
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib
```

Далее загружаем данные и логарифмируем целевую переменную:
```
train = pd.read_csv('data/train.csv')
train['SalePrice_log'] = np.log1p(train['SalePrice'])
```

Далее мы создаем списки признаков разного формата. Для начала формируем словарь `ordinal_mapping`, в котором мы указывали перечень всех возможных значений в правильном порядке для каждого порядкового признака. Обратите внимание на то, что мы переноситм словарь из Jupyter notebook целиком. Этот словарь нам поможет сформировать массивы паризнаков.   

Далее нам необходимо сформировать массивы признаков. Если в Jupyter Notebooks у нас было всего 4 массива: по одному на каждый вид признаков и еще один массив из массивов для порядковых признаков. То в данном случае таких массивов у нас будет много. Причина, по которой их будет много заключается в следующем: в ноутбуках мы вручную заполняли пропуски, при деплоее мы будем использовать для этого объекты класса `Pipeline`. И в каждом таком объекте Pipeline мы будем использовать `SimpleImputer`, который будет заполнять пропуски, а следом мы будем использовать еще и энкодер, в котором будут определены правила преобразования значений признаков. И поскольку мы не можем заполнить все пропуски каким-то одним значением (у каждого признака свой "заполнитель"), то мы будем использовать множество пайплайнов.   

Рассмотрим несколько таких пайплайнов:
```
num_pipe = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

numeric_zero_pipe = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])
```
Как мы видим, оба пайплайна занимаются обработкой числовых признаков. Если в Jupyter мы вручную обрабатывали все колонки и потом в препроцессор вписывали один массив со всеми числовыми пирзнаками, то в деплоее мы уже не обрабатываем ничего вручную, этим занимается пайплайн.    
Первый пайплайн работает с числовыми признаками, в которых пропуски можно заменить медианой, а второй папйплайн работает с числовыми признаками, пропуски в которых можно заменить нулями. То есть мы работаем с двумя разными массивами. И соответственно, мы доджны сразу разделить эти признаки:
```
numeric_features = train.select_dtypes(include=['int64', 'float64']).columns.to_list()
numeric_features = [col for col in numeric_features if col not in ['Id', 'SalePrice', 'SalePrice_log', 'GarageYrBlt', 'MasVnrArea']]
numeric_features_zero = ['GarageYrBlt', 'MasVnrArea']
```
Как видно из примера, мы создаем 2 массива. В первом массиве располагаются все числовые признаки, но только те, в которых пропуски можно заменить медианой. Мы исключили из этого массива 3 ненужные колонки, а также еще 2 колонки признаков, в котрых пропуски должны заменяться нулями.   

И под эти 2 массива мы создали 2 пайплайна. Сам синтаксис пайплайна прост, у `Pipeline` есть параметр `steps`, который принимает массив из кортежей. Каждый кортеж - это один этап обработки. Первый этап - заполнение пропусков. Обратите внимание, что в самом пайплайне мы не пишем, к какому конкртно массиву он относится, это будет происходить в препроцессоре (мы даже не пишем к каким признакам это относится, числовым, порядковым или категориальным). В пайплайне мы только устанавливаем правила обработки. В первом кортеже мы указываем нименование процесса, оно может быть произвольным. В нашем случае это `imputer`, а на второй позиции кортежа стоит класс `SimpleImputer()`. Для инициализации используется 2 основных параметра: `strategy` и `fill_value`. Первый параметр отвечает за метод заполнения пропусков, может принимать значения: `mean` (среднее арифметическое, только для чисел), `median` (медиана, только для чисел), `most_frequent` (самое частое значение, мода, подходит для строк и чисел) и  `constant` (заданная константа, применяется в связке с параметром fill_value). Второй параметр задает конкретное значение для замены, используется только если strategy='constant'. В первом пайплайне мы заполняем все пропуски медианой, во втором мы заполняем все пропуски значением 0.    

Второй кортеж используется под энкодеры. Но поскольку в данном случае мы формируем пайплайны для числовых признаков, которые не преобразуются в числа еще раз, то мы используем класс `StandardScaler()`. В чем его смысл? В данных как правило есть числовые признаки с абсолютно разными единицами измерения. К примеру, в нашем случае это LotArea (площадь участка), которая может измеряться тысячами, и признак FullBath (количество ванн), которое измеряется в количестве нескольких штук (1, 2 или 3). Если передать их в модель без обработки, алгоритм может "подумать", что площадь участка важнее просто потому, что числа там больше. `StandrdScaler` исправляет это неравенство. Он преобразует каждый признак так, чтобы его среднее значение ($\mu$) стало 0, а стандартное отклонение ($\sigma$) равно 1. Для каждого значения $x$ он выполняет формулу:
$$z = \frac{x - \mu}{\sigma}$$
Как работает данная формула: представим, что у нас есть один признак - количество комнат в трех домах (2, 4, 6). Среднее значение ($\mu$) считается как среднее арифметическое, складываем все значения и делим на количество. Получаем 4, это и есть наше $\mu$. После этого StandardScaler() отнимает от каждго значения полученное среднее:
- 2 - 4 = -2
- 4 - 4 = 0
- 6 - 4 = 2

Теперь наши данные "вращаются" возле нуля. Те, что были меньше среднего, стали отрицательными, а те, что больше - положительными. Если сложить эти новые числа (-2 + 0 + 2), то получим ноль. Таким образом среднее и становится нулем.    

Среднее отклонение ($\sigma$) - это мера того, насколько сильно "разбросаны" числа относительно среднего. Если бы знаечниями были 4, 4, 4, то разброса нет, $\sigma = 0$. Но у нас значения 2, 4, 6, а значит разброс есть. Процесс нахождения среднего отклонения сложнее, чем поиск среднего значения. Чтобы его найти, нужно использовать формулу **смещенного стандартного отклонения**. Алгоритм выглядит так:
1) Ищем среднее значение ($\mu$). Оно у нас равно 4. 
2) Производим расчет квадатров отклонений. Для этого мы находим разность каждого числа и среднего значения и возводим результаты в квадрат. Например, для числа 2 будет следующией алгоритм: $(2 - 4)^2 = 4$. И так для каждого числа. Как итог, получаем значения 4, 0, 4.   
3) Находим среднюю дисперсию. Складываем все квадраты отклонений, которые получили в предыдущем пункте и делим на общее число значений. $(4 + 0 + 4)/3 \approx 2.6667$. 
4) Извлекаем квадратный корень. Стандартное отклонение - это квадартный корень из дисперсии. Извлекаем корень из 2.6667 и получаем ~1.63299. Это и есть наша $\sigma$.

После этого мы можем посчитать нашу $z$. К примеру, $z$ для числа 2 будет считаться так:
$$z = \frac{2 - 4}{1.63299} \approx -1.22$$
А для числа 4 формула будет выглядеть так:
$$z = \frac{4 - 4}{1.63299} = 0$$
Для числа 6 то же самое:
$$z = \frac{6 - 4}{1.63299} \approx 1.22$$

После этой операции и огромная площадь, и маленькое количество комнат будут находиться в сопоставимом диапазоне (обычно от -3 до 3).   
Это нужно, чтобы модель получала справедливые штрафы. Регуляризация штрафует модель за большие коэффициенты. Если признаки не масштабированы, веса для маленьких чисел будут искусственно завышены, и модель "накажет" их несправедливо сильно.   
Это был пример на крохотных числах, но на реальных данных все работает ровно так же.     

Рассмотрим теперь пайплайны с категориальными признаками:
```
misc_feature_pipe = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='constant', fill_value='NoMiscFeature')),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

alley_pipe = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='constant', fill_value='NoAlley')),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])
```
Данные папйплайны будут использоваться для всего одного признака. Первый пайплайн будет отвечать за признак "MiscFeature", а второй пайплайн за "Alley". Несмотря на то, что оба признака категориальны, мы не можем совместить их в пайплайне, потому что пропуски в обоих колонках будут заполняться разными значеними. Первый пайплайн заполняет пропуски значением "NoMiscFeature", а второй - "NoAlley".    
Что касается второго кортежа, то в нем используется уже категориальный энкодер. Поскольку признаки не числовые, а категориальные, нам не нужно ничего масштабировать. Вместо этого нужно перевести значения в числовой тип. Как видно из кода, синтаксис энкодеров ничем не отличается от того, что мы писали в Jupyter Notebook.   

Теперь рассмотрим один пайплайн, созданный для порядковых признаков:
```
bsmt_str_cols = ['BsmtExposure', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1']
bsmt_pipe = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='constant', fill_value='NoBasement')),
    ('ord', OrdinalEncoder(categories=[ordinal_mapping[col] for col in bsmt_str_cols],
                           handle_unknown='use_encoded_value',
                           unknown_value=-1))
])
```
Как мы видим, тут мы также формируем пайплайн под определенный набор признаков. Мы формируем массив из необходимых порядковых признаков, в которых пропуски можно заменить значением "NoBasement". Поскольку порядковому энкодеру нужно передавать массив массивов с правильным расположением всех значений, мы формируем в аргументе `categories` необходимый перечень таких массивов. Делается это с помощью словаря порядковых признаков и генератора списков.    
Как несложно догадаться, если у нас есть особенные группы порядковых признаков, которые нужно заполнять определенными значениями, то значит у нас есть и остальные порядковые признаки, которые не попадают в какие-то "уникальные" группы. И когда мы строим пайплайн под такие признаки, мы должны следить, чтобы в массиве порядковых признаков были только обычные порядковые признаки, а "уникальные" были исключены из них. Вот пример, как это делается:
```
ordinal_features = list(ordinal_mapping.keys())
ordinal_features_exceptions = ['PoolQC', 'Fence', 'FireplaceQu', 'GarageType', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'BsmtCond']
ordinal_features_new = [col for col in ordinal_features if col not in ordinal_features_exceptions]
```
Как видим, мы сначала формируем массив из всех порядковых признаков с помощью ранее заготовленного словаря. Потом мы формируем массив из исключений среди порядковых признакв. Для них мы создавали отдельные пайплайны. А потом формируем окончательный список порядковых признаков путем исключения из общего массива исключительных признаков. Это все необходимо для правильного проектирования препроцессора.    

После того, как мы спроектировали все пайплайны, можно переходить к проектированию препроцессора. Он выглядит так:
```
preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_pipe, numeric_features),
        ('num_zero', numeric_zero_pipe, numeric_features_zero),
        ('pool', pool_pipe, ['PoolQC']),
        ('misc_feat', misc_feature_pipe, ['MiscFeature']),
        ('alley', alley_pipe, ['Alley']),
        ('fence', fence_pipe, ['Fence']),
        ('mas_vnr_type', mas_vnr_type_pipe, ['MasVnrType']),
        ('fireplace', fireplace_pipe, ['FireplaceQu']),
        ('garage', garage_pipe, garage_str_cols),
        ('garage_type', garage_type_pipe, ['GarageType']),
        ('bsmt', bsmt_pipe, bsmt_str_cols),
        ('cat', categorical_pipe, categorical_features),
        ('ord_simple', ordinal_pipe, ordinal_features_new)
    ]
)
```
Как видно из кода выше, сам принцип проектирования ничем не отличается от того, что мы делали в Jupyter Notebook. Мы так же используем `ColumnTransformer`, только теперь в параметре `transformers` мы используем кортежи, в которых вместо энкодеров используются пайплайны.   

Далее мы снова используем папйплайн. Но уже для того, чтобы объекдинить тренировку препроцессора и модели:
```
full_model_pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=0.000543))
])
```
Тут так же как и в предыдузих пайплайнах, в параметр `steps` мы передаем кортежи. Первый кортеж для препроцессора. Второй кортеж для модели. Обратите внимание на то, что мы не подбираем уже лучший гиперпараметр. Мы его определяем в момент исследования в Jupyter Notebook.    

Далее мы создаем полноценный датасет и целевую переменную для обучения: 
```
X = train.drop(columns=['Id', 'SalePrice', 'SalePrice_log'])
y = train['SalePrice_log']
```
Удаляем ненужные колонки у будущей матрицы и выносим целевую переменную в отдельную переменную.  

После этого мы делим наш датасет на тренировачный и валидационный:
```
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42
)
```

И начинаем тренировать модель на тренировочных данных:
```
full_model_pipeline.fit(X_train, y_train)
```
Обратите внимание на то, что данная строка заменяет метод `fit_transform()` у препроцессора и заменяет метод `fit()` у модели. То есть эти два метода объединяются, так как в пайплайне у нас был и препроцессор и модель.    

Далее тестируем модель на валидационной выборке и получаем результат работоспособности модели.  
```
y_pred = full_model_pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
```

Если все в порядке и модель выдает корректный результат, то обучаем препроцессор на полном датасете, и превращаем полный датасет в матрицу. И модель тренируем тоже на полном датасете: 
```
full_model_pipeline.fit(X, y)
joblib.dump(full_model_pipeline, 'model.joblib')
```
А также сериализуем модель в полноценный файл, который будет расположен у нас в корне проекта.   

##### schemas.py
Теперь можно переходить к реализации модуля `schemas.py`. Данный модуль гарантирует, что на вход в модель попадут только те данные, которые она понимает, в нежном формате и количестве.   
Реализация файла начинается и импортов:
```
import joblib
import numpy as np
from pydantic import create_model, BaseModel
from typing import Optional
```
Далее переходим к реализации модуля. Сначала нужно загрузить саму модель. Это делается с помощью десереализации. Используем метод `load()` модуля joblib и присваиваем результат вызова переменной `model`. Далее создаем вторую переменную, которая будет хранить массив из оригинальных колонок. Для этого к модели (объект класса Pipeline) применяем атрибут `feature_names_in_` (есть у всех объектов, классы которых наследуются от BaseEstimator), который возвращает наименования всех колонок исходного DataFrame в формате ndarray. После этого применяем метод `tolist()`, который переводит ndarray в обычный Python list. Все это заворачиваем в try/except. Если не получилось десереализовать модель или ее нет в корне проекта, то просто возвращаем сообщение об этом и пустой массив: 
```
try:
    model = joblib.load('model.joblib')
    feature_names = model.feature_names_in_.tolist()
except Exception as e:
    print(f'Failed to load model: {e}')
    feature_names = []
```
Обратим внимание на синтаксис исключения. `except` - это ключевое слово, которое работает в связке с `try`. После этого слова идет тип исключения (ошибки), которое мы хотим обработать. Обычно используются ошибки `ValueError`, `TypeError` и другие. Они наследуются от класса `Exception`. Когда мы указываем в типе ошибки `Exception`, мы отлавливаем любые стандартные проблемы, которые могут возникнуть. Конструкция `as e` является созданием переменной. Мы берем текст ошибки, которую поймал Python и приравниваем его в переменную `e`, чтобы можно было удобно оперировать им. В нашем случае мы просто выводим текст ошибки через `print()`.  

После того, как мы получили список строк (названий колонок), необходимо превратить их в Python-класс, который умеет проверять входящие данные.    
Создаем генератор словаря (dict comprehension). Мы уже ранее использовали метод list comprehension, тогда мы создавали массив на основе другого массива. В случае с генератором словаря примерно так же. Только сейчас мы создаем словарь на основе списка.   
Синтаксис dict comprehension выглядит так: `{key: value for item in collection}`. В нашем случае это будет выглядет так:
```
input_fields = {name: (Optional[object], None) for name in feature_names}
```
Разберемся более подробно в том, что происходит в коде. `name` - это ключ, который берется из массива `feature_names`, например, "LotFrontage". `(Optional[object], None)` - это значение, в данном случае кортеж. Аннотация `Optional[object]` происходит из модуля `typing`. Она отображает, что значение в этом поле может отсутствовать. К примеру, `Optional[int]` означает `int | None`, то есть цело число или ничего. `object` - "общий тип", он включает в себя и числа, и строки, и списки, и много чего другого. Другими словами `Optional[]` - это просто обозначение, которая говорит, что переменная может иметь тип, указанный в `[]` или быть `None`. И поскольку мы указываем в скобках `object`, мы говорим, что переменная `name` может иметь любой тип, либо `None`.      
Второе значение в этом кортеже - `None`. Оно означает, что поле необязательное. Указывая вторым значением `None` мы разрешаем пользователю отправлять не все 79 признаков.    

То есть другими словами, переменная из списка `feature_name`, к примеру, может быть любым объектом (хоть строкой, хоть числом, хоть списком) или быть None. Но в принципе, пользователю это значение вообще не обязательно даже предъявлять.    
Может возникнуть вопрос, если пользователь предъявит в поле "LotFrontage" строку (хотя модель для этого признака ожидает число), то значит наша схема пропустит эту строку. Но как только строка попадет в модель - препроцессор выдаст ошибку, потому что ожидает увидеть число. Почему же мы задаем такое значение, как `object`? Дело в том, что описывать все 79 признаков - слишком долгая работа и нам сейчас такие заморочки не нужны, поэтому оставим такие "широкие" ворота в этот раз. Но вообще, в серьезных полноценных проектах нужно будет описывать допустимое значение для каждого признака строго.    

И заключительная часть генератора словаря - `for name in feature_names`. В данной строке идет просто перебор элементов в массиве `feature_names`. Видим, что каждый элемент в цикле имеет наименование `name`, как и в строке `name: (Optional[object], None)`. То есть каждый перебираемый элемент - это ключ в итоговом словаре.   
И в конечном итоге мы получили словарь типа `имя колонки: (тип, дефолт)`.   

Далее создаем переменную "HouseData". Эта переменная - полноценный класс Pydantic. У нее будут доступны все методы Pydantic, она теперь умеет проверять данные и главное - мы можем использовать ее как форму для входящих данных. Как мы создадим такую переменную? С помощью функции-конструктора `create_model()` из Pydantic. Суть данной функции в том, чтобы динамически создавать классы. Как правило, мы пишем классы вручную, перечисляя все методы и атрибуты. Функция `create_model()` позволяет создать класс автоматически, на основе нашего словаря. Рассмотрим синтаксис:
```
HouseData = create_model('HouseData', **input_fields)
```  
В функцию `create_model` передается 2 аргумента: "HouseData" (наименование класса) и "** input_fields" (это тот самый словарь, который мы только что сделали, две звездочки означают, что мы "раскрыли" словарь, функция видит это не как один объект-словарь, а как 79 отдельных именованных аргумента). И вместо того, чтобы писать `LotArea = (Optional[object], None)` (и еще 78 похожих полей), это напишет за нас функция `create_model`. То есть при передаче в функцию аргумента `** dict`, мы благодаря этим двум звездочкам можем передать каждую пару из словаря как отдельный аргумент. И в новом объекте "HouseData" каждое новое поле - это пара из словаря, в которой двоеточие заменили на знак равенства.    
Таким образом у нас теперь есть класс "HouseData", у которого 79 атрибутов (полей), и каждое поле - это признак и то, какой тип данных он принимает.    

Мы создали класс, который является "формой", отсеивающей не подходящие данные. Теперь сформируем класс, который будет "формой", в соответствии с котрой мы будем отправлять данные в ответ. Назовем класс "PredictionResponse()". Сделаем так, чтобы этот класс наследовался от класса "BaseModel", чтобы он перенял все методы от родителя. Всего в нашем классе будет 3 поля: предсказанная цена, валюта предсказанной цены, сообщение об успехе:
```
class PredictionResponse(BaseModel):
    predicted_price: float
    currency: str = "USD"
    status: str = "success"
```
Поскольку класс PredictionResponse наследуется от BaseModel, он наследует и все его методы. А главный метод BaseModel - это валидация. То есть мы определяем 3 ключа, которые клиент увидит в своем JSON-ответе. `predicted_price: float` - мы гарантируем, что в этом поле всегда будет число. Если модель выдаст что-то странное, Pydantic это проверит. `currency: str = "USD"` - подставляем к целе валюту, чтобы клиент не получал цифру в вакууме. `status: str = "success"` - просто правило хорошего тона для API. Клиент, получив ответ, сначала посмотрит на статус. Если там "success", значит все прошло штатно.   

После этого, мы превратим данные в JSON. Но это уже будет в следующем файле, отвечающим за API - main.py. 

##### main.py
Файл `main.py` запускает сервер и принимает запросы. Реализацию как всегда начинаем с импортов:
```
import pandas as pd
import numpy as np
import joblib
from fastapi import FastAPI
from service.schemas import HouseData, PredictionResponse
```
После этого инициализируем приложение. Создадим объект класса `FastAPI` и назовем его `app`. Именно этот объект является диспетчером всего веб-приложения. Он выполняет роль маршрутизатора, который распределяет входящий трафик. Также этот объект - это точка входа для веб-сервера. А также много других важных функций. В общем, это сердце всего веб-приложения. Создадим его:
```
app = FastAPI(title='House Price Prediction Service')
```
Загружаем обученную модель:
```
model = joblib.load('model.joblib')
```
Далее создаем эндпоинт для предсказаний. Эндпоинт - это конкретный URL-адрес, по которому веб-приложение принимает запросы от клиентов и возвращает ответы. Эндпоинты в FastAPI состоят из 3 основных частей: декоратор пути (указывает HTTP-метод (GET, POST, PUT, DELETE) и сам URL-путь), функция операции пути (Python-функция, которая выполняется при вызове этого URL) и параметры и валидации (входные данные, которые автоматически проверяются (включая модели Pydantic)).   
Создадим эндпоинт:
```
@app.post("/predict", response_model=PredictionResponse)
```
Тут мы видим декоратор пути - http-метод post, а также url-путь "/predict". Также мы видим тут параметр `response_model`, который отвечает за то, как будет упакован результат работы функции. В нашем случае мы приравняли данный параметр к "PredictionResponse". Это тот самый класс, который отвечает за формат выходящих данных. Кстати, параметры и валидация бывает двух типов: **валидация тела запроса** (Input) и **валидация ответа** (Output). `response_model=PredictionResponse` - это как раз валидация ответа. А с валидацией тела запроса познакомимся чуть позже.   

Теперь создадим функцию операцию пути - последний из 3 обязательных частей эндпоинта:
```
def predict(data: Any):
    df = pd.DataFrame([data.model_dump()])
    log_prediction = model.predict(df)[0]
    real_price = np.expm1(log_prediction)
    return PredictionResponse(predicted_price=float(real_price))
```
Именно эта функция будет вызываться, если клиент пришлет запрос методом POST на адрес "/predict". Разберемся с тем, что тут написано. Мы создаем функцию, которая принимает ранее созданный класс HouseData. Этот класс отображает, данные какого формата мы ждем от клиента. `data: HouseData` - это, к слову, и есть та самая валидация тела запроса. Тут FastAPI делает валидацию того, что пришло от клиента. То есть `data` - это объект, данные, которые прислал клиент. Когда клиент присылает JSON (к примеру, {"LotArea": 8000, "Street": "Pave"}), Pydantic берет свой "чертеж" HouseData и проверяет данные. После этого создает экземпляр. Поэтому уже внутри функции `predict`, объект `data` - это реальный объект, содержащий данные, а не `(Optional[object], None)`.    

В теле функции мы превращаем объект HouseData в объект DataFrame. Чтобы это реализовать, мы сначала превращаем объект HouseData в обычный словарь с помощью метода `model_dump()` (который есть у любого объекта BaseModel). А далее мы оборачиваем этот словарь в список: `[data.model_dump()]`. Почему словарь нужно обернуть в список? Дело в том, что если этого не сделать, pandas выдаст ошибку ValueError, потому что pandas увидит ключи и значения, но не пойеме, мы пытаемся сделать одну строку с кучей колонок или же создать одну колонку, где значениями будут эти данные. Для pandas "скалярные" (одиночные) значения в словаре - это неоднозначность. И когда мы оборачиваем словарь в список, для pandas это четкий сигнал, что каждый словарь в списке - это отдельная строка, ключи словаря это названия колонок, а значения - это данные в ячейках.   

Зачем мы вообще превращаем объект `data` класса HouseData (наследуется от BaseModel) в объект DataFrame? Да потому что модель из sklearn не умеет работать с объектами Pydantic и с чистыми словарями Python, поэтому нам необходимо перевести объект пользователя в понятный для модели вид - в объект DataFrame. И в конечном итоге мы получаем таблицу (объект DataFrame с одной строкой и 79 колонками).   

После этого вызываем у модели метод `predict()` для предсказания цен. Причем обратите внимание на то, что мы передаем в метод наш новый объект DataFrame. Но после передачи, мы также указываем в квадратных скобках индекс 0. Мы специально указываем индекс, так как метод `predict()` возвращает результат в виде массива ndarray. И когда мы вызываем данный метод, он возвращает, к примеру, `array([250340.5])`. И индекс 0 мы используем именно для того, чтобы достать из этого массива само число и отдать его клиенту. Без этого нуля мы бы попытались отправить пользваотелю целый массив, что вызвало бы ошибку при создании JSON. Данный метод вернет нам логарифмированную цену, так как мы учили предсказывать модель именно лагорифм цены.    

Переводим логарифм обратно в абсолютную величину с помощью функции `np.expm1()`.    

И последняя строка - это то, что мы возвращаем из функции. И из функции мы возвращаем объект класса PredictionResponse, в чей конструктор мы передаем значение цены, переведенное во float. В float необходимо заворачивать значение по причине того, что NumPy часто выдает типы вроде `numpy.float64`, которые стандартный JSON-преобразователь иногда не переваривает. Поэтому перевести значение в обычный Python float - это самый безопасный вариант.    

На этом все. Теперь можно протестировать наш API. Для запуска FastAPI приложений используется сервер Uvicorn. Открываем терминал и переходим в папку проекта. Заходим в venv проекта. И вводим команду `uvicorn service.main:app`.    
В терминале будет строка: `Uvicorn running on http://127.0.0.1:8000`, что означает, что сервер Uvicorn успешно запущен и работает локально на этом компьютере. Но если мы перейдем по этому адресу, то нас встретит надпись `{"detail":"Not Found"}`. Так происходит, потому что в main.py мы указали только один путь - `/predict`. Однако переходить мы должны по адресу `http://127.0.0.1:8000/docs`. Когда мы запускаем сервер, FastAPI берет все функции (такие как `predict()`), читает все модели данных (такие как `HouseData`) и генерирует веб-страницу Swagger UI по этому адресу (`/docs`). Это сделано специально для разработчиков для удобства. `/docs` - это автоматическая песочница для тестов, которую FastAPI предоставляет бесплатно.   

Но что будет, если пройти по адресу `http://127.0.0.1:8000/predict`. Ведь путь `/predict` мы указывали явно. В этом случае нас встретит ошибка `{"detail":"Method Not Allowed"}`. Так происходит, потому что когда мы вбиваем какой-то адрес в браузер, он по умолчанию отправляет GET-запрос (так как хочет получить страницу для отображения). Наша же функция предсказания помечена декоратором `@app.post("/predict")`. Это значит, что по этому адресу может попасть только тот, кто "приносит" данные. FastAPI закономерно отклоняет запрос.    

Итак, когда мы переходим по адресу `/docs`, то нас встречает перечень всех адресов. В нашем случае мы должны увидеть одну строку, в которой будет указан метод POST и наименование `/predict`. Нажимаем на эту строку и после этого мы должны увидеть кнопку "Try it out". Нажимаем на нее. После этого ниже должна появится кнопка "Execute", при нажатии на которую, мы можем получить результат на дефолтный JSON. Сам JSON мы можем перестроит под себя. Например, указать в JSON какое-то количество признаков и их значения. После этого, в окошке "Server response" мы увидим результат работы модели в формате класса PredictionResponse, который мы определяли в schemas.py. 

#### Containerization
Суть контейнеризации заключается в том, что мы создаем изолированную среду, внутри которой будет стоять нужная версия Python и все бибилиотеки из файла `requirements.txt`. Это гарантирует, что API запустится на любом сервере.   

Для начала необходимо создать `Dockerfile` в корне проекта. Обратитие внимание на то, что мы создаем файл без какого либо расширения, но обязательно с большой буквы.    
Также сразу нужно прояснить небольшую деталь. Мы используем именно Dockerfile, так как контейнеризируем свое собственное приложение. Dockerfile нужен для сборки нового Docker-образа из исходного кода. Dockerfile пошагово описывает ОС, установку зависимостей и запуск программы. И в конечном итоге мы получаем новый уникальный образ нашего приложения. Также для работы часто используется файл `docker-compose.yml`. Docker Compose нужен для запуска и настройки многоконтейнерных приложений. Он нужен, когда, нам нужно запустить систему из нескольких готовых сервисов. Например, наше приложение, базу данных PostgreSQL и кеш Redis. В таком файле описывается, как контейнеры будут общаться друг с другом, какие порты открыть и где хранить данные. Как результат, данный файл отвечает за одновременный запуск всей IT-инфраструктуры.   

Начнем реализацию Dockerfile. Для начала нам потребуется базовый образ Python. Docker не может запустить Python-код в какой-то пустоте. Ему нужна операционная система. Поэтому установим ее:
```
FROM python:3.12-slim
```
`FROM` - обязательная первая команда в Dockerfile. Она указывает Docker, какой образ взять за основу. `python` является именем официального образа в репозитории Docker Hub. В него уже встроены сам Python, менеджер пакетов pip и базовые утилиты. Далее ставим двоеточие и указываем версию Python. Поскольку в venv у нас установлена версия 3.12.10, то и в контейнер мы тоже должны устанавливать Python такой же версии. Нам не обязательно указывать патч-версию (прямо 3.12.10 (3-мажорная, 12-минорная, 10-патч-версия)), тега 3.12 более чем достаточно. Docker сам подтянет самый свежий и безопасный патч внутри двенадцатой ветки. `slim` означает, что оттуда вырезали весь мусор и тяжелые утилиты, вроде C++ компилятора, заголовочных файлов и инструменты сборки, чтобы контейнер не весил 2 Гб, а оставался компактным. Нужно заметить, что мы не скачиваем именно Python. Мы скачиваем образ мини-дистрибутив Linux, в котором уже установлен python.    

Далее переходим к созданию папки внутри контейнера:
```
WORKDIR /app
```
Это аналог консольной команды "mkdir /app" или "cd /app". Данная команда создает рабочую директорию внутри контейнера и переносит нас в нее. Все последующие действия (копирование файлов (COPY), запуск команд (RUN)) будут происходить именно в этой виртуальной папке /app внутри контейнера. Это изолирует наше приложение от системных файлов Linux.  

После создания отдельно директории для работы, мы можем перейти к копированию файла со списком библиотек с реального компьютера внутрь контейнера:
```
COPY requirements.txt .
```
Ключевое слово `COPY` копирует файл, идущий после. И копирует он его в папку /app, которую мы создали в предыдущем шаге. Данная строка просто берет файл из корня проекта и копирует в именно в текущую директорию. То, что мы копируем файл в текущую директорию, говорит точка в конце команды. То есть в  прошлом шаге мы перешли в папку /app. С следующем шаге (этом) копируем сюда файл.    

То есть у `COPY` есть универсальный базовый синтаксис. Сначала прописывает ключевое слово `COPY`, после него идет источник. Это первая позиция. На первой позиции можно писать как наименование какого-то файла, так и путь, откуда нужно копировать. Вторая позиция всегда является адресом, куда нужно скопировать. Также нужно знать несколько нюансов. Во-первых, мы не можем указывать на первой позиции абсолютно любой путь в совем компьютере. Это правило контекста, Docker видит файлы только внутри той папки, в которой расположен Dockerfile. Во-вторых, синтаксис позволяет скопировать несколько файлов за раз, нам нужно лишь указать их все через пробел, а последний аргумент в этом общем списке всегда должен быть папкой, ведь он становится второй позицией (куда нужно копировать), а все, что между COPY и последним аргументом - то, что мы копируем. В-третьих, если мы копируем папку, например, `COPY ./src ./app`, то Docker скопирует внутрь контейнера только все содержимое папки src. Саму же папку src он создавать не будет.    

Далее установим в контейнер все бибилиотеки из файла requirements.txt без сохранения кеша:
```
RUN pip install --no-cache-dir -r requirements.txt
```
Команда `RUN` запускает любую консольную команду операционной системы (а нашем случае - Linux внутри контейнера). Все, что установит эта команда, навсегда сохранится в готовом образе. `pip install` - стандартный менеджер пакетов Python, который скачивает бибилиотеки из репозитория PyPI. `-r requirements.txt` - указывает pip, что нужно устанавливать пакеты списком из файла requirements.txt, который мы скопировали в наш контейнер в предыдущем шаге. `--no-cache-dir` - важнейший флаг Docker. Обычно pip при скачивании бибилиотек сохраняет их копии на кэш на диске, чтобы при повторной установке не качать их из интернета. Внутри контейнера этот кеш нам никогда не понадобится, потому что контейнеры Docker создаются один раз и больше никогда не обновляются "изнутри". И флаг `--no-cache-dir` запрещает сохранять эти временные файлы. Данный флаг критически важен для уменьшения размера Docker-образа.   

После этого переходим к копированию нашего проекта в контейнер. Делается это с помощью, уже знакомого, ключевого слова `COPY`:
```
COPY . .
```
После ключевого слова `COPY` у нас стоит две точки. Первая точка указывает то, откуда мы копируем. Точка - это корень проекта на реальном компьютере. Docker смотрит в папку, в котрой лежит сам файл Dockerfile и берет оттуда все файлы. Вторая точка - это то, куда мы копируем. И поскольку мы находимся в директории /app внутри контейнера, то все файлы мы копируем именно в /app.   

Может возникнуть вопрос: "Почему мы сначала копируем requirements.txt, устанавливаем бибилиотеки, а потом копируем все остальное. Не проще ли сначала все скопировать разом, а потом устанавливать?". Все дело в **кешировании слоев**. Docker собирает образы по слоям. Каждая строчка в Dockerfile создает новый слой. Прежде, чем выполнить команду, Docker смотрит, не делал ли он ее ранее. Если делал и ничего не изменилось, он не тратит время на выполнение, а берет готовый слой из кеша за доли секунды. Но у кеша есть правило: если какой-то слой изменился, то все последующие слои за ним "сгорают", и Docker обязан пересобрать их с нуля.    

Если бы мы сначала скопировали весь проект в /app, а потом устанавливали бы все бибилиотеки из requirements.txt, то при первой сборке все было бы нормально. В /app скопировался бы весь проект, а потом 5 минут качался би весь инструментарий. Все слои записались бы в кеш. Но если бы мы после этого решили что-то изменить в main.py и начали бы пересобирать контейнер, то при второй сборке Docker бы дошел до строчки `COPY . .` и увидел, что файл main.py изменился. И все, что стоит после этого копирование - сгорает. Docker должен переписать все последующие слои. И весь кеш с библиотеками удалится. И придется копировать сначала весь проект в /app, а потом опять скачаивать весь инструментарий.   

Именно поэтому му сначала копирем requirements.txt в /app, потом устанавливаем его, и только потом копируем остальные файлы проекта. Это позволяет один раз скачать в контейнер весь инструментарий, и при изменении кода, нам не приходится заного качать бибилиотеки, ведь скачивание располагается до полного копирования кода проекта.     

Переходим к объявлению порта. Контейнер - это глухая коробка. Декларируя порт, мы открываем веб-трафику доступ наружу:
```
EXPOSE 8000
```
Обратите внимание, что эта строка именно декларирует открытие порта, а не открывает его. Открытием порта мы занимаемся при запуске.    

Теперь можем реализовать команду, запукающую контейнер:
```
CMD ["uvicorn", "service.main:app", "--host", "0.0.0.0", "--port", "8000"]
```
Все предыдущие шаги выполнялись один раз - в момент сборки образа. А команда `CMD` - это команда, которая этот образ запускает, она говорит Docker-у, что как только контейнер включится, нужно запустить внутри него сервер Uvicorn, найти в нем приложение и начать слушать запросы.   
Теперь поговорим более подробно про каждый аргумент в скобках. `"uvicorn"` - это главная запускаемая программа. ASGI-веб-сервер, который мы устанавливали через requirements.txt. Сам по себе FastAPI - это просто набор правил и логики (кода), он не умеет самостоятельно слушвать сеть и принимать HTTP-запросы. `uvicorn` выступает в роли "двигателя", который заводит FastAPI-приложение и заставляет его работать как полноценный веб-сайт. `"service.main:app"` - это путь к нашему приложению. service - это папка, main - файл, :app - имя переменной внутри файла main.py, где инициализирован FastAPI. `"--host"` и `"0.0.0.0"` являются флагами настройки сетевого адреса и его значения. Если мы запускаем проект локально на компьютере без Docker, сервер обычно стартует на адресе `127.0.0.1` (localhost). Это значит, что сервер будет принимать запросы только от самого себя. Но Docker-контейнер - это изолированная "коробка" со своей внутренней сетью. Если внутри этого контейнера запустить сервер на `127.0.0.1`, он будет принимать запросы только изнутри себя самогою А мы со своего компьютера (снаружи "коробки") достучаться до него не сможем. Адрес `0.0.0.0` - это специальная сетевая команда, которая приказывает слушать все доступные сетевые интерфейсы. Она открывает сервер контейнера "наружу", позволяя нашему компьютеру пересылать запросы внутрь Docker. `"--port"` и `"8000"` - это флаги настройки порта и номера самого порта. По умолчанию uvicorn тоже пытается занять порт 8000, но в Docker принято указывать это явно. Мы данными флагами жестко привязываем наш веб-сервер к порту 8000 внутри контейнера. Именно на этот порт мы позже будем направлять трафик с компьютера.    

Теперь наш Dockerfile готов к работе. Но нам еще нужно добавить одну деталь - `.dockerignore`. Этот файл очень похож на `.gitignore`, только если `.gitignore` мы использум, чтобы не тащить мусор в репозиторий git, то `.dockerignore` используется, чтобы не тащить мусор в контейнер.   

Сам файл `.dockerignore` создается в корне проекта. Разберем, что мы должны "затенить" из проекта, чтобы контейнер не был захламлен мусором:
1) `.vscode/`, `.idea/`, `.env`, `.DS_Store` - настройки редакторов, файл с секретами проекта и мусор от операционной системы MacOS. Первые 2 строки - это папки с настрйоками редакторов кода. `.env` - это текстовый файл, в который записываются пароли от баз данных, ключи доступа к API, секретные токены. `DS_Store` - скрытый системный файл, который автоматически создает ОС macOS.    
2) `.git/`, `.gitignore` и `.dockerignore` - история репозитория и сборники игнорируемых файлов. В `.git/` git хранит всю историю коммитов, веток, изменений и настроек. И для контейнера это абсолютно бесполезная информация, так как контейнеру нужен только текущий рабочий код, чтобы запустить сервер. А файлы `.gitignore` и `.dockerignore` просто не нужны в контейнере.   
3) `venv/`, `__pycache__/`, `*.py[cod]` - папка локального виртуального окружения, а также кеш и скомпилированные файлы. Исключить папку `venv/` - критически важно. В этой папке установлены все локальные библиотеки на реальном компьютере. Командой `RUN pip install...` мы уже скачали все необходимые библиотеки в контейнер. И если мы не внесем в `.dockerignore` папку с локальным инструментарием, то она затрет или перемешается с тем, что Docker установил себе сам. Поэтому крайне важно вносить в `.dockerignore` папку с локальным инструментарием. Также сразу лучше добавить и такие строки как `env/` и `.venv/`. Это тоже папки виртуального окружения, только `/env` - это старое обозначение, которое уже редко используется, а `.venv/` - это та же папка venv, только с точкой в начале, которая делает папку скрытой на Linux. `__pycache__/` - кеш компиляции Python. Он создается, когда мы запускаем Python-скрипты локально. В кеше хранятся файлы с расширением `.pyc`. Это код, скомпилированный в байт-код для более быстрого повторного запуска. И он нам не нужен, так как этот кеш Python создал специально под наш компьютер и ОС. Если скопировать эти файлы внутрь Linux-контейнера Docker, они там будут бесполезны. Docker должен скомпилировать код сам, внутри своей изолированной среды. `*.py[cod]` - это маска для скомпилированных файлов Python. Она обозначает любой файл, расширение которого начинается на `.py` и заканчивается на `c` `o` или `d`.   
4) `*.ipynb`, `.ipynb_checkpoints/`, `*.ipynb.*`, `*.ipynb~` - ноутбуки, в которых мы проводили исследования. Они нам тоже уже не нужны, так как задачей ноутбуков является исследование данных и проектирование модели. Ноутбуки свою функцию выполнили, в контейнере они не нужны. `*` - это маска, она означает, что наименование у файла может быть каким-угодно, главное, чтобы у него было расширение `.ipynb`. `*.ipynb~` - знак тильды в конце означает **временный файл резервной копии**, их часто создают текстовые редакторы или Linux-системы в момент, когда файл открыт и мы его правим. `.ipynb_checkpoints/` - это папка, в которой хранятся промежуточные версии ноутбуков на случай сбоя. Эта папка также не нужна в контейнере. В `*.ipynb.orig`расширение `.orig` появляется тогда, когда мы делаем `git merge` (слияние веток) и возникает конфликт слияния. Git создает файл `file_name.ipynb.orig`, чтобы сохранить оригинальную версию файла до того, как мы начнем исправлять вручную ошибку. В `.gitignore` эта строка нужна, чтобы случайно не запушить этот конфликтный файл в репозиторий, в `.dockerignore` мы также ее пропишем, а также включим в файл строку `*.ipynb.*`, чтобы в контейнер точно не просочились файлы типа `research.ipynb.orig` или `research.ipynb.bak`.    
5) `data/` и `*csv.` - папка с нашими датасетами и файлы с расширением csv. Тут может быть 2 сценария. Первый: если у нас в проекте уже есть сохраненный файл `model.joblib`, то API при старте просто загружает этот файл в делает предсказание. Ему не нужны исходные сырые данные `train.csv` или `test.csv` для работы. В этом случае папка data/ - балласт. Второй сценарий: если Docker-контейнер при старте должен был бы сначала запустить `python service/train.py`, заного обучить модель на сырых данных и только потом включить API - тогда папка data/ была бы нужна внутри контейнера. Но поскольку у нас первый сценарий, мы вносим папку data/ и файлы csv в `.dockerignore`.   

Теперь, когда у нас готовы Dockerfile и .dockerignore, мы можем приступать к контейнеризацит. Запустим Docker и введем в терминале (в корне проекта) команду `docker build -t house-prices-api .`. Данная команда состоит из 3 ключевых элементов: действие, метка(имя) и контекст. Действие `docker build` запускает процесс сборки образа. Docker запускаяет Builder, который "идет" в указанную папку, находит там файл Dockerfile и начинает выполнять его строчку за строчкой.  Метка (имя) `-t house-prices-api` необходимо для присваивания имени образу. Флаг `-t` расшифровывается как tag (название), а `house-prices-api` - имя, которое мы присваиваем образу. Если запустить сборку без этого флага, Docker соберет образ, но вместо имени присвоит ему случайный длинный хэш из букв и цифр. Искать и запускать такой образ в будущем, мягко говоря, затруднительно. Флаг `-t` дает образу понятное название. Контекст сборки `.` указывает путь к контексту сборки (в данном случае - текущая папка, в которой открыт терминал). Точка приказывает Docker-у искать файл Dockerfile прямо в этой папке (в которой находимся), а когда в Dockerfile сработает команда `COPY . .`, то Docker возьмет файлы с этой папки.   

После запуска данной команды начнется процесс контейнеризации. Где-то в терминале должен быть временной счетчик, который помимо времени сборки должен отражать еще и этап. Когда все закончится, что на этом счетчике будет указано время контейнеризации и строка "FINISHED".   

Теперь запустим сформированный образ. Сделаем это с помощью команды `docker run -d -p 8080:8000 --name my-ml-service house-prices-api`. Разберем, что делает эта команда. `docker run` говорит Docker-у, чтобы он взял готовый образ и запустил его в виде изолированного контейнера. `-d` - запускает контейнер в фоновом режиме и возвращает нам управление консолью. `-p 8080:8000` - это проброс портов. Порты связываются по схеме компьютер:контейнер. Внтури Dockerfile мы настроили Uvicorn на порт 8000. Это порт внутри изолированного контейнера. И флаг `-p 8080:8000` связывает порт 8080 на реальном компьютере с портом 8000 внутри контейнера. После этого любой запрос, который мы отправим на `http://localhost:8000`, Docker автоматически перенаправит внутрь контейнера FastAPI. `--name my-ml-service` задает красивое имя контейнеру. Если его не указать, Docker назовет контейнер случаным словосочетанием/буквосочетанием. `house-prices-api` - имя образа, который мы собрали на прошлом шаге.   

Теперь, когда контейнер поднят, можно протестировать работаспособность сервиса. Переходим по адресу `http://localhost:8080/docs` в браузере и тестируем. Обратите внимание на то, что поменялся порт. Если при запуске сервиса локально мы переходили по адресу `http://localhost:8000/docs` (или `http://127.0.0.1:8000/docs`, это одно и то же), то сейчас мы используем тот же самый адрес, но с портом 8080. Все из-за того, что мы мы приказали Docker-у слушать реальный компьютер на порту 8080, и если на этот порт приходит запрос, то его нужно перекинуть внутрь контейнера на порт 8000. Мы это прописали в команде запуска.   

Чтобы завершить работу контейнера необходимо ввести команду `docker stop my-ml-service`. А если мы хотим полностью удалить контейнер, то можно ввести команду `docker rm my-ml-service`.  

## Функции, атрибуты и методы Pandas   
`pandas.read_csv(path)` - функция из pandas, принимающая на вход путь к csv-файлу, и преобразующая файл в объект DataFrame.   
`info()` - метод объектов NDFrame. Возвращает "None". Показывает количество столбцов и строк в датасете, а также показывает количество непустых значений в каждом столбце. Чаще всего применяется к объектам DataFrame. Работает как функция "print()", просто выводит данные в консоль. Сохранить их в переменную не получится, так как возвращается None.       
`describe()` - метод объектов NDFrame, возвращает тот же тип объекта, для которого был вызван. Показывают различную статистику по данным: минимум, максимум, среднее и т.д. Активно применяется как для объектов DataFrame, так и для объектов Series. При применении метода к Series получаем полную статистику по колонке. При применении к DataFrame, получаем таблицу, где строки - это параметры (mean, std, max,...), а столбцы это изначальные столбцы из DataFrame. По умолчанию считает статистику только для числовых данных, но если нужна статистика по строчным, то используется параметр "include" со значениями "all/str(string)". В случае применения include='all' - отображает данные по всем колонкам (и числовым и строковым), только для строковых одни данные отображаются (unique, top, freq), а для числовых колонок другие (mean, std, ...). Исходный объект не меняет, но результат можно присвоить переменной. Аргумента "inplace" у метода нет.    
`isnull()` - метод объектов NDFrame. Возвращает объект того же типа, к которому применяется, только заполненный "True/False" (True - есть пропуск, False - нет пропуска). Нет параметров. Исходный объект не меняется, метода "inplace" нет.   
`sum()` - метод объектов NDFrame. Суммирует значения. Если применяется к DataFrame - возвращает Series. Если применяется к Series - возвращает scalar (одно число). Работа метода с Series понятна. В случае работы с DataFrame играет значение параметр "axis". Если он равен "0", то сложение идет по вертикали и возвращается Series, где индексы - это наименования колонок DataFrame, а значения - сумма значений каждой колонки DataFrame. Если же "axis" равен "1", то сложение идет по горизонтали. И мы получаем Series, где индексы равны индексам DataFrame, а значение - это сумма всех значений строки. Также важно понимать, что в современных версиях pandas есть еще параметр "numeric_only", который по умолчанию False. То есть в сложении будут участвовать все типы, и числовые и строковые (конкатенация). Но если в одной строке или столбце встретятся и строки и числа, то Pandas выдаст ошибку TypeError. При значении True, все нечисловые колонки будут проигнорированы. Исходный объект не меняется, параметра "inplace" нет. Имеет аналогичную функцию в Python, однако принцип работы разный.   
`sort_values()` - метод, определенный отдельно для Series и DataFrame. Возвращает тот тип, к которому был применен. Причина, по которой он определен не в NDFrame, а по отдельности заключается в разных подходах к сортировке. При применении к Series все понятно, тут можно использовать параметр "ascending". Значение False означает сортировку по убыванию. Значение True означает сортировку по возрастанию. При сортировке DataFrame обязательно использовать параметр "by" (by='SalePrice' или by=['SalePrice']). Этому параметру нужно предоставить имя колонки, по которой будет осуществляться сортировка. Также можно сортиовать по нескольким столбцам. Для этого параметру "by" нужно передать список из названий столбцов (by=['Neighborhood', 'LotFrontage']). Он отсортирует их сначала по первому столбцу из списка. А потом, если в первом столбце были одинаковые значения, метод начнет сортировать по второму столбцу. Если все значения из первого столбца были разные, то второй сортировки не будет. Также важно помнить про аргумент "inplace", так как данный метод не вносит изменения в оригинал по умолчанию, а создает копию. Но на практике принято использовать присваивание новой переменной, нежели вносить изменения в исходный объект. Такой подход экономит память.      
`value_counts()` - метод объектов Series. Он применяется к Series, подсчитывает количество каждого уникального значения и возвращает Series, где индекс - это уникальное значение, а значение - это число, сколько раз встретилось какое-то знчение в изначальном Series. Есть аргумент dropna. Если оно False, то к уникальным значениям добавляется NaN (пропуски), если значение True, то пропуски игнорирются и не попадают в статистику. Метода inplace нет.    
`fillna()` - метод объектов NDFrame. Возвращает тот же объект, к которому применяется. Используется для заполнения пропусков в датасете. Аргументом является значение, которое мы хотим вставить на место пропуска. В случае применения к Series, все работает легко, просто заполняются пропуски в колонке. При применении к DataFrame все немного сложнее. Можно заменить все пропуски в DataFrame одним значением. Можно применить словарь для того, чтобы заменить пропуски разными значениями в зависимости от колонки (df.fillna({'A': 0, 'B': 99})). Тут мы заполняем пропуски в колонке 'A' нулями, а пропуски в колонке 'B' значениями 99. Данный метод относится к методам-трансформерам, поэтому в нем есть параметр inplace, но его лучше не использовать, так как данный параметр не экономит память.    
`groupby()` - это метод объектов NDFrame. Он применяется для того, чтобы сгруппировать данные. В первых скобках () указывается колонка, по которой нужно группировать. После этого возвращается объект DataFrameGroupBy, это тот же датасет, только разделенный на группы по уникальным значениям выбранной колонки. Далее во вторых скобках [] указывается вторая колонка, с чьими значениями мы собираемся взаимодействовать. После этого возвращается уже объект SeriesGroupBy. Это тот же самый Series, только поделенный на несколько групп. Далее мы можем применять какие-то функции, которые будут взаимодействовать с каждой группой. И важно понимать, что мы можем использовать на этих группах методы NDFrame, потому что каждая такая группа в SeriesGroupBy - это объект Series. Это было описание того, как он применяется к DataFrame. В случае с Series все тяжелее. Там сортировка может быть осуществленна по внешнему списку, либо по индексу, либо через словарь или функцию. На данный момент углубляться не будем.    
`transform()` - это метод объектов GroupBy. GroupBy - родительский класс для классов DataFrameGroupBy и SeriesGroupBy. Он принимает какой-то код, который применяется к каждой группе в объекте SeriesGroupBy или DataFrameGroupBy.    
`median()` - это метод объектов класс NDFrame. Он высчитывает 50% процентиль (медиану). В случае применения к Series возвращает scalar - медиану всех значений в колонке. В случае применения к DataFrame возвращает Series, где индекс - это названия столбцов (при axis=0) или индексы строк (при axis=1), а значения - вычисленные медианы.   
`any()` - метод объектов класса NDFrame. Проверяет, есть ли у объекта хотя бы один элемент, который оценивается как True. Если применить данный метод после "isnull()", то при применении к Series возвращает scalar (True (если есть пропуски) или False (если пропусков нет)). При применении к DataFrame возвращает Series (по одному True/False на каждый столбец).    
`dropna()` - метод объектов класса NDFrame. Имеет множество параметров. Если используется для DataFrame, то важно применять аргумент "subset='...'", в котором указывается наименование колонки, в которой проверяются пропуски. И удаляются все строки, которые имют пропуски в этой колонке. Также важно значть, что данный метод не совершает изменения в самом объекте, а возвращает новый по умолчанию. Поэтому важно использовать либо присваивание, либо аргумент "inplace=True".   
`astype()` - метод объектов класса NDFrame. Возвращает тот же тип объекта, к которому и был применен. Применяется к Series или DataFrame и преобразовывает данные в ячейках в тот тип, который был передан в метод.    
`select_dtypes()` - метод объектов класса NDFrame. Возвращает объект того же типа, к которому применяется. В качестве параметра используется "include", который принимает типы данных. Таким образом, данный метод вычленяет из данных только те колонки, которые соответствуют указанному типу в "include" (может принимаеть "number", который включает все числовые типы, а также bool; также принимает string/object, object устарел). На Series работает несколько иначе, чем на DataFrame. Поскольку Series - это однородный контейнер, где все значения одного типа, то при использовании "select_dtypes()" на Series, метод просто проверяет тип данных, и если они совпадают с тем, что указано в include, то возвращается исходный Series, а если нет, то пустой Series.    
`columns` - атрибует свойство объектов класса DataFrame. Возвращает "pandas.Index" - это неизменяемая система координат с хеш-поиском и выравниваением. При примененении к DataFrame будет содержать в себе перечень всех наименований колонок.   
`to_list()` - метод одномерных объектов (pandas.Index, pandas.Series). Он не наследуется ни от какого класса, это независимая реализация в Index и Series. Возвращает обычный Python list. Но лучше использовать написание именно "tolist()", так как это алиас для совместимости с NumPy, сделанный для разработчиков, привыкших к синтаксису NumPy.      
`drop()` - метод объектов класса NDFrame, возвращает тот же тип объекта, к которому применен. Удаляет из объектов конкретные колонки/строки. Имеет несколько параметров: список удаляемых колонок/строк, а также axis. axis принимает 2 значения: 0 или 1. Если 0, то удаление идет по строкам, если axis=1, то удаление идет по колонкам. Также, когда мы удаляем колонки, нам нужно просто передать массив с наименованиями, если удаляем строки, то нужно передать массив индексов строк. Но в новых стандартах axis считается уже устаревшим (менее читаемым), и лучшей практикой является применение "columns/index". Если хотим удалить колонку, пишем columns=['...', '...', '...'], если удаляем строку, то пишем index=['...', '...', '...']. В Series данный метод работает так же, как и удаление строк в DataFrame. Он удаляет строки по меткам индекса. Если индекс стандартный (0, 1, 2, 3, ...), то s.drop([2]) удалит 3-й элемент, если индекс кастомный, то удаление идет строго по меткам. Имеет параметр inplace, то это плохой тон в ML, не используется.    
`pandas.DataFrame` - это класс pandas. Когда мы употребляем его со скобками (), мы вызываем конструктор этого класса, чтобы создать новый объект (таблицу). Данному конструктору можно дать сырые данные в виде списка словарей, массива NumPy или просто список списков, и он преобразует их в объект DataFrame с индексами, названиями колонок и методами для анализа. 


## Функции и методы NumPy  
`numpy.log1p()` - глобальная функция из NumPy. Вычисляет натуральный лагорифм от аргумента + 1.   
`numpy.nan` - специальное значение с плавающей точкой для обозначения пропущенных, неопределенных или некорректных данных в числовых массивах.    
`tolist()` - этот метод определен в классе numpy.ndarray. Преобразует массивы ndarray в обычные Python list.   
`mean()` - этот метод определен в классе numpy.ndarray. Вычисляет среднее арифметическое из массива ndarray. Возвращает одно усредненное число.    
`numpy.sqrt()` - глобальная функция из NumPy. Вычисляет корень из чисел и массивоподобных структур. Причем массивоподобные структуры могут быть не только типа ndarray, а также и list, tuple, pandas.Series и даже pandas.DataFrame. Данная функция извлегает квадратный корень из каждого элемента массива по отдельности и возвращает массив корней.    
`numpy.argmin()` - глобальная функция из NumPy. Ищет индекс самого маленького элемента в массиве. Принимает любые массивоподобные струкутры.   


## Функции и методы Python  
`keys()` - метод класса dict из Python. То есть применяется к обычным словарям и возвращает объект "dict_keys", который не является списком, а скорее "окном" в ключи словаря. "dict_keys" - итерируемый объект.   
`list()` - это функция из Python. Принимает любой итерабельный объект и извлекает их него элементы, записывая в новый изменяемый массив.    
`append()` - метод класса list из Python. Применяется к обычным массивам (lists) и добавляет в конец массива элемент, указанный в качестве аргумента.   


## Функции, классы и методы sklearn  
#### Подмодуль base
`TransformerMixin` - это базовый миксин (mixin), который предоставляет метод fit_transform() любому классу, реализующему fit() и transform(). Данный класс объединяет все трансформеры в sklearn, независимо от подмодуля. Класс является трансформером, если реализует методы fit() и transform(). В sklearn есть множество различных трансформеров, например, OrdinalEncoder, OneHotEncode, ColumnTransformer и многие другие. Во всех них определены методы fit() и transform() по отдельности, а fit_transform() уже идет "в наследство" от их общего родительского класса TransformerMixin.     
`BaseEstimator` - это **базовый** класс, от которого наследуются практически все объекты в библиотеке: модели (Lasso, Random Forest), препроцессоры (StandardScaler) и конвейеры (Pipeline). Цель данного класса - обеспечить единый интерфейс. Данный класс дает объектам 2 критически важные способности: (1) управление параметрами (get_params() и set_params()). get_params() позволяет спрсить у модели, какие настройки у нее сейчас есть (например, alpha в Lasso). set_params() позволяет изменить эти настройки, создавая новый объект. (2) представление объекта (логирование). Когда мы пишем print(model), мы видим текстовый вывод со всеми настройками, а не адрес в памяти типа. Также в данном классе заложена логика того, как должна реагировать модель на входные данные. Когда мы вызываем fit() у модели, внутри модели происходит обращение к методам проверки, которые прописаны в BaseEstimator. Эти методы проверяют: нет ли в данных бесконечных значений или NaN, являются ли данные числами; и если пришел DataFrame, эти методы создают атрибут "feature_names_in_".    
`feature_names_in_` - атрибут класса BaseEstimator. Он доступен объектам классов, которые наследуются от BaseEstomator. Появляется в объекте только после вызова метода fit(). Возвращает массив numpy.ndarrey, в котором каждый элемент - это текстовое название колонки оригинального DataFrame.   

#### Подмодуль preprocessing       
`OrdinalEncoder` - класс из подмодуля preprocessing модуля sklearn. Наследуется от базового класса "TransformerMixin". Является трансформером, реализует методы "fit()" и "transform()". Имеет несколько парметров:  
- *categories* - данный параметр принимает список значений в определенном порядке для порядковых признаков. Имеет 2 возможных значения: 'auto' или list[list]. Значение по умолчанию - 'auto'. Всегда нужно передавать свой собственный список, так как при 'auto' сортировка значений происходит в алфавитном порядке.   
- *drop* - удаляет лишнюю колонку при преобразовании строк в числа, решая тем самым проблему мультиколлинеарности. Мультиколлинеарность - это проблема, при которой наличие всех колонок в категориальном признаке мешает модели правильно определить вес параметра. Например, у нас есть категориальный признак, он имеет 3 возможных значения. Когда на одном значении 1, то на других 0. Нужно ли нам в этом случае 3 колонки, чтобы определить, какое значение имеет признак? Нет, достаточно двух, так как если в оставшихся двух колонках 0, значит 1 в третьем столбце. Почему мультиколлинеарность является проблемой? Потому что линейная модель имеет формулу "$Цена = w_1*x_1 + w_2*x_2 + w_3*x_3 + b$", и когда модели нужно найти одни единственные веса для $w_1, w_2, w_3$, возникает проблема с бесконечным количеством решений системы уравнений из-за связи $x_1 = 1 - x_2 - x_3$. Такая система может иметь следующие значени: "$w_1=100, w_2=200, w_3=300$", "$w_1=1000, w_2=700, w_3=300$", "$w_1=-500, w_2=800, w_3=300$". Все 3 варианта дают одинаковую ошибку на обучающих данных, компьютер не понимает какое значение выбрать, веса начинают "скакать", становятся огромными или отрицательными, модель становится нестабильной. Проблемой мультиколлинеарности страдают только линейные модели и только категориальные признаки (так как только они заменяются несколькими бинарными колонками, порядковые признаки такими проблемами не страдают). Данный параметр имеет несколько возможных значений: *None* - значение по умолчанию (никакого эффекта не оказывает, так как ни одна колонка не удаляется); *'first'* - удаляет первую категорию (превращает "0, 1, 2" в "0, 1"); *'if_binary'* - удаляет колонку только если у признака ровно 2 уникальных значения. Если >2, то оставляет все. В контексте OrdinalEncoder данный признак избыточен, его не нужно использовать вовсе.   
- *handle_unknown* - регулирует рекцию энкодера на незнакомое значение. Мы передаем в OrdinalEncoder массив из массивов возможных значений. Потом обучаем этот энкодер на этих значениях. И если в обучающей выборке какое-то значение не попадалось, но попалось в тестовой выборке, тогда энкодер ведет себя в соответствии с переданным значением в handle_unknown. Может принимать 2 значения (только для OrdinalEncoder, у OneHotEncoder всего 3 значения): 'error' и 'use_encoded_value'. Значение 'error' является значением по умолчанию. Означает, что если энкодер встретил незнакомое значение в тестовой выборке, значит нужно вернуть ошибку. Пайплайн упадет. Значение 'use_encoded_value' говорит, что при встрече незнакомого значения нужно присвоить ему вес, указанный в "unknown_value" - это еще один параметр класса OrdinalEncoder.   
- *unknown_value* - данный параметр отвечает за значение, которое будет присвоено значению, которое энкодер увидит впервые; работает только если *handle_unknown='use_encoded_value'*. К примеру, у нас есть параметр "качество камина". Есть перечень возможных значений, расположенных от худшего к лучшему = ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], а в тестовой выборке может попасться значение 'Super'. Энкодер не знает, насколько хорошее или плохое это значение относительно остальных, поэтому ему будет причислено число, которое не было занято другими значениями, и как правило, это -1. Параметр может принимать несколько значений: *int* - любое целое число (например -1); *float* - любое десятичное число; *np.nan* - NaN из NumPy. По умолчанию параметр имеет значение -2, но как говорилось ранее, обычно причисляют значение, предшествующее худшему, то есть -1.   
- *encoded_missing_value* - параметр, отвечающий за то, какое значение будет иметь пропуск в признаке. Снова рассмотрим пример с тем же признаком "качество камина". Изначально нам в руки попадает необработанный датасет, где есть много пропусков. Пропуски в датасете означают отсутствие камина вовсе. В "data_description.txt" четко написано, что NA - это отсутствие камина. Но из-за особенностей работы pandas, это "NA" превращается в NaN, что классифицируется как пропуск. Параметр encoded_missing_value позволяет обрабатывать все пропуски. К примеру, мы можем не обрабатывать пропуски в датасете, а оставить все, как есть. В этом случае даже не потребуется обрабатывать пропуски в учебном датасете, они просто будут заменены значением, которое мы присвоим encoded_missing_value. Данный пораметр может преобретать следующие значения: *int* - любое целое число; *foat* - любое десятичное число; *np.nan* - NaN из NumPy. По умолчанию этот параметр имеет значение NaN. Чтобы данный параметр работал, нужно указывать *handle_unknown='use_encoded_value'*.  

Есть важный нюанс, который следует учитывать. *handle_unknown + encoded_missing_value* - это параметр, отвечающий за пропуски в тренировочных данных. Если мы их не заполнили, то за нас это сделает "encoded_missing_value". И нам не нужно будет указывать NaN в массиве categories_list на первой позиции. Это за нас сделает энкодер. *handle_unknown + unknown_value* - рабатает только с НЕИЗВЕСТНЫМИ значениями в ТЕСТОВЫХ датасетах.   

Это значит, что при незаполненных пропусках в тренировочном датасете есть несколько вариантов развития событий: 
- Указываем handle_unknown='use_encoded_value'; unknown_value=-1; encoded_missing_value=-2. В этом случае, все пропуски принимают значения -2, а реальные значения начинаются с 0. Неизвестные значения в тестовых датасетах получают значение -1.  
- Указываем только handle_unknown='use_encoded_value'; unknown_value=-1. В этом случае получаем ошибку, так как модель не понимает, как интерпритировать пропуски. Пропуски не подходят под unknown_value, так как этот параметр предназначен только для неизвестных значений в тестовом датасете. Но это только для линейных моделей. Бустинг-модели нативно поддерживают np.nan и автоматически строят по ним оптимальные ветки сплита.  
- Указываем только handle_unknown='use_encoded_value' и encoded_missing_value=-2. В этом случае ошибки не будет, пайплан не упадет, но при встрече неизвестных значений в тестовом датасете, они будут получать значение -2, так как чтобы работал encoded_missing_value, нам нужно указать handle_unknown='use_encoded_value', и соответственно unknown_value тоже будет работать, но будет преобретать значение по умолчанию -2. И NaN будет равен тому же значению, что и любое новое значение в тестовом датасете.  
- Ничего не указываем = handle_unknown='error'. Пайплайн упадет.   

`OneHotEncoder` - класс из подмодуля preprocessing модуля sklearn. Наследуется от базового класса "TransformerMixin". Является трансформером, реализует методы "fit()" и "transform()". Имеет несколько парметров:  
- *categories* - данный параметр принимает перечень всех значений категориальных признаков. Может принимать 2 значения: list[list] и 'auto'. В случае выбора list[list] нужно будет передать массив, где каждый элемент является массивом значений одного из категориальных признаков. В случае же выбора 'auto', энкодер сам совершает следующие действия: (1) Фильтрует (берет только валидные значения), то есть находит все уникальные значения для каждой категориальной колонки, уникальные значению ищутся раздельно, энкодер обрабатывает каждую колонку из categirical_features независимо друг от друга. Данный процесс начинает только тогда, когда энкодер попадает в preprocessor, куда мы передаем этот энкодер и массив категориальных признаков. (2) Сортирует (все уникальные значения, которые он признал валидными); Порядок самих категориальных колонок энкодер не трогает, он следует строго по списку categorical_features, а вот все значения для кадой колонки он сортирует в алфавитном порядке (если строки) или по возрастанию (если числа). (3) Запониминает отсортированный список в атрибут "categories_[i]". categories_[i] - это список массивов (list of ndarrays). i - это индекс элемента массива categorical_features. То есть мы можем использывать запись categorical_encoder.categories_[0] и увидеть отсортированный список всех уникальных значений для первой категориальной колонки. categories_[i] появляется только после применения метода fit() у preprocessor-а. (4) Создание матрицы; во время применения метода transform() у preprocessor-а, энкодер создает матрицу из множества бинарных колонок. Количество колонко этой матрицы = суммарное количество элементов в атрибует "categories_" (но только, если параметр drop=None, и соответственно, он не удаляет никакие колонки).  
- *drop* - данный параметр удаляет лишнюю колонку при преобразовании строк в числа, решая тем самым проблему мультиколлинеарности. Понятие мультиколлинеарности более подробно разобрано в этом же параметре в OrdinalEncoder. Только стоит напомнить, что проблемой мультиколлинеарности страдают только линенйые модели в категориальных признаках. В OrdinalEncoder данный параметр был избыточен и мог принимать только 3 значения (None, 'first', 'if_binary'). В OneHotEncoder данный парметр может принимать 4 разных значения: *None* - отключает данный параметр, *'first'* - удаляет одну колонку в преобразованной матрице у каждого категориального признака. *'if_binary'* - удаляет колонку в преобразованной матрице у каждого категориального признака, но только в том случае, если в колонке всего 2 значения. Если признак имеет >2 значений, то оставляет все как есть. *array-like* - это список/массив длины N, где N - количество категориальных признаков. Каждый элемент списка указывает, какую именно категорию удалить для соответствующего признака. Если удалять не нужно - ставится None. Синтаксис выглядит так: drop=['Control', None, 'Red']. Это значит, что мы берем первый признак из categorical_feature и удаляем из преобразованной матрицы колонку 'Control', далее доходим до второго признака и ничего не трогаем, переходим к третьему признаку и удаляем из преобразованной матрицы колонку 'Red'. Зачем это нужно, если есть 'first'? Дело в том, что 'first' удаляет категорию, которая стоит первой после сортировки. Но в статистике и бизнес-логике часто нужно удалить конкретную смысловую категорию. array-like даёт полный контроль над тем, что станет референсной (базовой) группой в линейной модели.  
- *sparse_output* - параметр, отвечающий за формат данных, которые возвращает энкодер после преобразования. Формат данных может быть представлен в виде "сжатой" (разряженной) матрицы или обычного "плотного" массива. Может иметь значения: True или False (по умолчанию). При True энкодер возвращает разреженную матрицу (scipy.sparse). Она хранит только индексы и значения всех ненулевых элементов. Если у нас много категорий (сотни или тысячи), то это сэкономит много огромный объем оперативной памяти. При False возвращается обычный плотный массив (numpy.ndarrya). В памяти хранятся все нули и единицы. Нужно использовать, когда категорий мало или нужно сразу передать результат в библиотеку, которая не поддерживает разреженные форматы.  
- *dtype* - данный параметр отвечает за тип данных, в котором будут представлены числа в результирующей матрице. Этот параметр принимает любой стандартный числовой тип данных из библиотеки NumPy. По умолчанию имеет значение "numpy.float64". Но можно использовать "numpy.float32", если данных очень много. Это сокращает потребление памяти в 2 раза по сравнению с float64. Также можно использовать и целые типы, например, "numpy.int32" или "numpy.int8", если мы хотим видеть в матрице простые 0 и 1 без точек. Но проблема заключается в том, что в этом случае sklearn выдаст DataConversionWarning или сам скастит обратно во float, так как после энкодеров часто идут матричные операции, линейные модели или StandardScaler/MinMaxScaler, которые требуют вещественные числа. Поэтому для OneHotEncoder лучше оставлять float64/float32.   
- *handle_unknown* - данный параметр отвечает за то, как себя будет вести энкодер при встрече с незнакомым значением на тестовой выборке. В отличии от OrdinalEncoder, где данный параметр может приобрести только 2 значения, в OneHotEncoder данный параметр обладает 3 возможными значениями: *'error'* (значение по умолчанию, но только если не заданы min_frequency или max_categories) - выдает ошибку при встрече с незнакомым значением; *'ignore'* - игнорирует новое значение из тестовой выборки, выставяляя во все бинарные колонки признака нули; *'infrequent_if_exist'* - это значение, созданное для борьбы с high-cardinality (признаками с десятками/сотнями редких категорий). Это работает следующим образом: мы используем еще 2 параметра для обозначения порога редкости - *min_frequency=10* или *max_categories=20* (они взаимоисключающие, можно использовать только один из них). Во время вызова метода fit() у preprocessor-а энкодер считает частоту каждой категории (сколько раз встречается каждое уникальное значение в каждом признаке). Категории, встречающиеся реже порога, не получают отдельных колонок. Вместо этого они объединяются в одну общую колонку с именем {FeatureName}_infrequent_sklearn. При вызове метода transform у preprocessor-а, любая неизвестная категория из тестовой выборки попадает в эту же колонку. К примеру, если у нас есть признак Neighborhood, который содержит в себе 25 уникальных значений, и 20 из них встречаются реже 5 раз (то есть min_frequency=5), то после вызова метода fit() у preprocessor-а энкодер создаст 5 колонок для частых районов (те, которые встречаются чаще 5 раз) и одну колонку 'Neighborhood_infrequent_sklearn', куда будут определены все редкие районы.   
- *min_frecuency* - дополнительный параметр, который идет в связке с handle_unknown=infrequent_if_exist. Принимает: *int* (абсолютное количество значений), *float* (доля от общего числа значений, например: 0.01 означает, что признаки, встречающиеся реже, чем в 1% строк уйдет в общую колонку), *None*. Если какое-то значение признака встречается реже, чем указано в min_frequency, то оно лишается отдельной колонки и помещается в колонку {FeatureName}_infrequent_sklearn, где располагаются другие такие же редкие значенния, а также незнакомые значение из тестовой выборки.  
- *max_categories* - дополнительный параметр, который идет в связке с handle_unknown=infrequent_if_exist. Принимает: *int*, *None*. Данный параметр выбирает самые часто встречающиеся категории (топ N, где N - это значение данного парамтера) и оставляет их как отдельные столбцы. Все остальные (менее частые) категории объединяет в один общий столбец, который по умолчанию называется infrequent_category.  То есть если max_categories = 20, то энкодер выделит 19 столбцов под самые частые уникальные значения, а 20-й столбец станет "сборной" для самых редких значений.   

#### Подмодуль compose  
`ColumnTransformer` - класс из подмодуля compose модуля sklearn. Имеет множественное наследование, его основными родительскими классами являются TransformerMixin и BaseEstimator. Является трансформером, реализует методы "fit()" и "transform()". Имеет несколько параметров:   
- *transformers* - данный параметр является обязательным и должен принимать данные типа *list[tuple(name, transformer, columns)]*. **name** означает наименование трансформера, может быть произвольной строкой; **transformer** - это энкодер, который должен быть создан до момента объявления объекта ColumnTransformer; **columns** - это список наименований колонок, массив с порядковыми, категориальными или числовыми колонками колонками. Как правило в параметр transformers попадает 3 кортежа, по одному на каждый тип признаков: числовой, категориальный и порядковый.   
- *remainder* - управляет судьбой колонок, которые физически присутствуют в исследуемом DataFrame, но не указаны ни в одном из массивов признаков (numeric_features, ordinal_features, categorical_features) внутри параметра *transformers*. Preprocessor анализирует те данные, что мы ему передали (X_train) и сравнивает их с объединененным списком numeric + ordinal + categorical. Если находит лишние колонки в X_train (те, которые не указаны в numeric + ordinal + categorical, например 'Id', 'SalePrice' или 'SalePrice_log', которые мы не помещали в numeric_features), то применяет правило remainder. Может принимать 3 значения: *'drop'* (значение по умолчанию) - удаляет из выходной матрицы те колонки, которые не обнаружены в numeric + ordinal + categorical; *passthrough* - оставляет как есть, добавляет в конец выходной матрицы в исходном порядке; *Transformer (объект)* - применяет этот трансформер ко всем "хвостовым" колонкам. В sklearn transformer - это архитектурный интерфейс. Любой объект, у которого есть методы .fit() и .transform() считается трансформером. И когда параметру remainder приравнивается *Transformer* - это означает, что все колонки, которые не перечислены в numeric + ordinal + categorical нужно обработать вот этим самым трансформером, вместо того, чтобы удалить или оставить сырыми. Например, трансформером может быть StandardScaler() - он автоматически масштабирует все неуказанные числовые колонки. Вообще, различных трансформеров в sklearn десятки.   
- *sparse_threshold* - это автоматический переключатель на выходе ColumnTransformer. Данный параметр отвечает за то, в каком виде будет выходная матрица - numpy.ndarray или scipy.sparse. Принимает в качестве значения исключительно float в диапозоне [0.0, 1.0]. По умолчанию имеет значение "0.3". При "0.0" всегда возвращает плотную numpy.ndarray. При "0.3" если доля ненулевых элементов (density) в итоговой матрице < 0.3 (30%) - возвращает scipy.sparse, иначе - ndarray. При "1.0" всегда возвращает разреженную матрицу (если хоть один трансформер выдал sparse). sparse_threshold не конфликтует с sparse_output из OneHotEncoder. Это пост-обработчик склейки, который автоматически переключает формат финальной матрицы при высокой разреженности. Данный парметр нужен в большенстве своем для экономии памяти. В NLP или рекомендательных системах матрицы часто на 99% состоят из нулей. Без sparse_threshold мы бы получил OOM (Out Of Memory) на склейке. Значение 0.3 является золотым стандартом и его редко когда нужно менять.  
- *verbose_feature_names_out* - данный параметр отвечате за добавление префикса "name__" к именам колонок. Принимает в качестве значений: True (по умолчанию) или False. При True это выглидит так: num__TotalSF, ord__ExterQual, cat__Neighborhood_CollgCr. "name" - это наименование трансформера из параметра transformers: num - numeric, cat - categorical, ord - ordinal.    

#### Подмодуль model_selection
`train_test_split()` - функция из подмодуля model_selection модуля sklearn. Реализует разделение датасета на тренировочную и валидационную выборку. Возвращает 4 переменных: X_train (тренировочный датасет), X_val (валидационный датасет), Y_train (тренировочная целевая переменная), Y_val (валидационная целевая переменная). Строго в этом порядке. Принимает несколько аргументов:  
- **arrays* - означает "принимай любое количество позиционных аргументов", то есть можно передать столько массивов/DataFrame/Series, сколько нужно, они все будут разделены. Можно передать 1, 2, 3 и более датасетов, но главное, чтобы они все имели одинаковое количество строк. Порядок возврата строго соответствует порядку передачи. Это скорее правило, нежеле параметр функции; на практике выглядит так: train_test_split(A, B, y, test_size=0.2). То есть мы не пишем нигде **arrays*, а просто передаем необходимое число датасетов.    
- *test_size* - параметр, отвечающий за долю валидационной выборки. Принимает float от 0.0 до 1.0 или int (абсолютное количество строк); также принимает None, при этом значении берется 25%. Обычно имеет значение 0.2, то есть берутся первые 20% строк, а остальные идут в train. Но это только если shuffle=False. При shuffle=True берутся случайные 20% после перетосовки.  
- *train_size* - параметр, отвечающий за долю тренировочной выборки. Принцип работы аналогичен test_size. В качестве аргумента можно указать только train_size или test_size, так как второй параметр автоматически высчитывается исходя из первого.   
- *random_state* - seed для генератора случайны чисел. Позволяет на разных запусках получать один и тот же результат, одинаковые строки от запуска к запуску для валидационного и тренировочного датасета. Принимает любое целое число, но как правило приравнивается к 42. По умолчанию имеет значение None. Если shuffle=False, то данный параметр игнорируется, даже если задан, но по умолчанию shuffle=True.   
- *shuffle* - перемешивает строки случайным образом. Принимае либо True (значение по умолчанию) либо False. Если random_state не указан (None), то каждый раз перемешивание происходит по разному, если random_state указан, то алгоритм перемешивания идентичный от запуска к запуску.   
- *stratify* - данный параметр отвечает за массив меток для сохранения пропорций классов. Он нужен только для задач классификации, когда классы в данных не сбалансированы. К примеру, у нас есть 1000 медицинских записей: 950 из них - это "здоровые" (класс 1), а 50 - "больные" (класс 2). При обычном разделении 80/20 может случиться так, что в train попадет 49 больных, а в test 1 больной (или вообще 0). При таком раскладе модель никогда не научится распознавать редкий класс, а валидация даст бессмысленную метрику. Для того, чтобы такого не было, используют stratify, которому приравнивается Y (целевая переменная). В этот момент функция группирует строки по классам, внутри каждого класса делает случайную выборку 80/20 и склеивает обратно. Таким образом, в train и test соотношение классов будет строго одинаковым. Еще один важный нюнас заключается в том, что в линейной регрессии это параметр не используется, так как в линейной регрессии как правило непрерывная целевая переменная (то есть число, которое может принимать любое значение в диапозоне, а не в дискретной категории, например, данный прект - целевая переменная это цена дома, она может принимать любое значение в диапозоне). А параметр данный применяется в тех случаях, где целевая переменная - дискретная, то есть принимает строго определенные значения: 0/1, halthy/sick, A/B/C.  

`cross_val_score()` - функция из модуля sklearn подмодуля model_selection. Она предназначена для того, чтобы оценить качество модели на кросс-валидации и вернуть массив оценок для каждого фолда. Имеет 5 параметров:
- *estimator* - объект, который мы тестируем. Это может быть отдельная модель (например, Ridge()) или целый Pipeline. Главное, чтобы у этого объекта были методы "fit()" и "predict()".  
- *X* - матрица признаков. То, какой она будет зависит от *estimator*. Если в качестве *estimator* пердается пайплайн, то на месте *X* должен быть сырой датасет (так как преобразование в матрицу происходить в папйплайне). А если же мы в качестве *estimator* передается модель (например, Ridge()), то передавать в качестве *X* нужно уже числовую матрицу, полученную от препроцессора.   
- *y* - вектор целевой переменной.   
- *cv* - количество фолдов, на которые мы разбиваем датасет (или матрицу). Если передать цифру 5, то датасет будет делиться на 5 частей.  
- *scoring* - метрика, по которой мы оцениваем модель. Данный параметр может принимать три типа значений: *None* (по умолчанию), *string* (текстовое имя одной из десятков стандартных метрик scikit-learn), *Callable* (собственная кастомная функция метрики). Значения типа *string* самые часто используемые. Например, для задач классификации используются следующие значения: *arruracy* (общая доля правильных ответов), *precision* (точность, способность не присваивать класс "1" объектам класса "0"), *recall* (полнота, способность находить все объекта класса "1"). И многие другие. Для задач регрессии используются такие значения как *neg_mean_squared_error* (среднеквадратичная ошибка (MSE) со знаком минус), *neg_root_mean_squared_error* (корень из среднеквадратичной ошибки (RMSE) со знаком минус). И многие другие.   

`GridSearchCV` - класс из подмодуля "model_selection" модуля "sklearn". Является одним из инструментов подбора гиперпараметра. Имеет метод "fit()", который запускает процесс кросс-валидации. Аргументами данного метода, по классике, являются "X" и "y". Имеет следующие параметры:
- *estimator* - объект, который мы тестируем. Это может быть отдельная модель (например, Ridge()) или целый Pipeline. Главное, чтобы у этого объекта были метод "fit()".   
- *param_grid* - словарь или список словарей, задающий "сетку" параметров для перебора. Синтаксис данного параметра зависит от того, чем у нас является *estimator*. Если *estimator* - это пайплайн, то ключ должен строго состоять из трех параметров: "имя шага в пайплайне" + "__" + "название параметра модели". А значение такого ключа - обязательно массив. Чаще всего используется обычный Python list, но можно использовать и массив NumPy. Например, мы создали такой пайплайн:
```
pipe = Pipeline(steps=[
    ('prep', preprocessor),
    ('ridge_model', Ridge())
])
```  
Наш *estimator* является пайплайном, значит параметр *param_grid* будет иметь вид: {'ridge_model__alpha': [0.1, 0.5, 1, 5, 10, 50]}. "ridge_model" в данном случае берется из названия, которое сами дали этому шагу внутри пайплайна. А "alpha" берется из названия аргумента в конструкторе самого класса Ridge().    
Если же наш *estimator* - это модель, то применяется другой синтаксис. GreedSearchCV будет искать параметры напрямую в самой модели. Ключ будет просто 'alpha', без всяких имен и двойных подчеркиваний. Значение также является массивом.    
- *scoring* - метрика, по которой GridSearchCV будет определять, какая выиграла. Может принимать значение *None*, *string* или кастомный объект *make_scorer*. Возможных значений типа *string* огромное множество для самых разных задач классификации, регрессии и других.  
- *cv* - определяет количество фолдов в кросс-валидации. 
- *refit* - принимает булево значение. Если True (значение по умолчанию), то после того, как GridSearchCV найдет наилучшую комбинацию параметров на кросс-валидации, он автоматически переобучит модель с этими параметрами на всех предоставленных данных. Если значение параметра False, то соответственно, не будет переобучать.  
- *n_jobs* - параметр для параллельных вычислений, который задействет многоядерность процессора. Поскольку оценка каждого фолда и каждого параметра происходит независимо друг от друга, этот процесс можно легко распараллелить. Если принимает значение -1, то компьютер задействет все доступные ядра процессора. Если принимает значение 2, то задействет только 2 ядра.  
- *verbose* - отвечает за детализацию логов в процессе вычислений. По умолчанию имеет значение 0. Если 0, то в консоли ничего не отображается. Если принимает значение 1, то показывает общее количество комбинаций и фолдов. Если значение 2 или 3, то будет детально выводить время старта и окончания расчетов для каждой отдельной итерации.  
- *return_train_score* - Принимает булево значение. Если выставить в True, то в итоговую таблицу результатов (grid.cv_results_) добавятся метрики качества не только на валидационных фолдах, но и на обучающих. Это очень полезно, если есть необходимость проверить модель на переобучение (overfitting): если на трейне метрика идеальная, а на валидации проваливается - модель переобучилась.

#### Подмодуль pipeline
`Pipline` - класс модуля sklearn подмодуля pipeline. Позволяет объединить последовательность трансформации данных и финальную модель в единый объект. Основной родительский класс - "_BaseComposition ". Главным преимуществом является то, что данный класс предотвращает утечку данных. Имеет несколько параметров:
- *steps* (list of tuples) - является обязательным параметром, принимает список кортежей. Все шаги, кроме последнего обязаны быть трансформерами (то есть иметь методы "fit()" "transform()" и, соответственно, "fit_transform()"). Последний шаг может быть как трансформером, так и оценщиком/моделью (иметь метод "fit()", и например, "predict()").   
- *memory* - путь к директории или объект для кеширования промежуточных результатов. Может быть строкой, если путь, а может быть объектом joblib.Memory. По умолчанию имеет значение None. Если гиперпараметры финальной модели подбираются через GridSearchCV, то препроцесинг данных будет происходить заново на каждой итерации. "memeory" позволяет закешировать результаты первых шагов на жесткий диск, что ускоряет работу. 
- *verbose* - параметр, который при значении True выводит в консоль время, затраченное на "fit()". Но по умолчанию имеет значение False.   

## Функции, классы и методы модуля joblib
`joblib.dump(model, 'name')` - dump() является функцией модуля joblib, возвращает объект класса Pipeline. При вызове данной функции происходит процесс сериализации. Берется объект Python и превращается в последовательность байтов (единиц и нулей). И сохраняется после этого. В контексте ML, мы помещаем в качестве аргументов обученную модель и прописываем имя, которым она будет названа в формате 'имя.расширение'. В данном случае расширение '.joblib', а имя 'model'. После запуска кода, где есть такая функция, в корне появляется файл 'model.joblib'.    
`joblib.load('name')` - load() является функцией модуля joblib, возвращает объект того же типа, который был сохранен через dump(). При вызове данной функции происходит обратный процесс процессу сериализации - десериализация. Программа ищет в папке файл, считывает поток байтов и воссоздает в оперативной памяти точную копию того объекта, который был сохранен.   

## Функции, классы и методы модуля Pydantic
`create_model()` - функция-конструктор, которая динамически создает класс. Принимает множество параметров, но главные из них только 2: наименование класса (строка) и распакованный словарь (**dict). Возвращает класс, который содержит столько атрибутов, сколько было пар в переданном словаре. Созданный функцией класс наследуется от класса "BaseModel".    
`BaseModel` - класс из бибилиотеки Pydantic, который является её фундаментом. Любой объект, который относится к классу BaseModel или к классу, который наследуется от "BaseModel", получает 3 особенности: (1) Валидация - как только данные попадают к такому объекту, он проверяет, соответствует ли входной JSON классу BaseModel. Если нет, то BaseModel выдаст ошибку, указав в чем конкретно проблема. (2) Парсинг - BaseModel умеет преобразовывать данные. Если в JSON пришло число в кавычках, а в схеме указано int, то BaseModel сам сконвертирует строку в число. (3) Сериализация - у каждого объекта, созданного на основе "BaseModel", появляются разные методы. Самый важный такой метод - ".model_dump()", который берет сложный объект (класса "BaseModel" или наследуемого от него) и превращает его обратно в обычный Python-словарь. Это необходимо, потому что pandas.DataFrame не умеет работать с объектами Pydantic, но отлично умеет создавать таблицы из обычных словарей.    
`model_dump()` - метод объектов класса BaseModel. Превращает объект Pydantic в обычный Python-словарь.  